In [2]:
from google.colab import drive

drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
%pip install -q lightgbm optuna catboost
from collections import Counter, defaultdict

import pandas as pd
from sklearn.linear_model import LinearRegression
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import HuberRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.svm import SVR
from threadpoolctl import threadpool_limits
from catboost import CatBoostRegressor
import lightgbm as lgb
import numpy as np
import optuna
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/kamp_2026/data/okm_augumented_2021.csv", encoding="utf-8-sig")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 2.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 23.3 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 16.5 MB/s eta 0:00:00


### 데이터 분할

- 시간순 홀드아웃 분할(1~8월 학습, 9월 테스트)

In [4]:
# 9월 테스트 분리, 모델 비교는 1~8월만
test_df = df.loc[df["날짜"].floordiv(100).eq(202109)].copy()
train_df = df.loc[df["날짜"].between(20210101, 20210831)].copy()

assert not train_df.empty and not test_df.empty
assert len(train_df) + len(test_df) == len(df)
assert train_df["날짜"].max() < test_df["날짜"].min()
print(f"학습(1~8월) {len(train_df):,}행")
print(f"테스트(9월) {len(test_df):,}행")

# 공통 설정: 시드, 정답 열, 월별 확장 윈도우 검증월, 평가 지표, CV 점수 모음
SEED = 42  # 전 모델·탐색 공통 시드
POWER = ["15분", "30분", "45분", "60분"]
VALIDATION_MONTHS = [202104, 202105, 202106, 202107, 202108]  # 1~3월→4월, …, 1~7월→8월
cv_results = {}  # cv_results["모델명"] = {"scores": 월별 MAE·RMSE·R2 DataFrame}


def metrics(y, pred):
    y, pred = np.asarray(y).ravel(), np.asarray(pred).ravel()
    return dict(MAE=mean_absolute_error(y, pred), RMSE=np.sqrt(mean_squared_error(y, pred)),
                R2=r2_score(y, pred))

학습: 1~8월, 5,832행
별도 평가: 9월, 336행


## 기준 모델: OLS

- 선형회귀(OLS), 월별 확장 윈도우 CV, MAE·RMSE·R²

In [5]:
# OLS: 시간·요일·생산량·기온 원값 선형회귀 (피처 가공·규제·가중치 없음)



def ols_features(frame):
    """시간·생산량·기온 원값 + 요일(0=월~6=일)"""
    x = frame[["시간", "생산량", "기온"]].copy()
    x["요일"] = pd.to_datetime(frame["날짜"].astype(str), format="%Y%m%d").dt.dayofweek
    return x


ols_pipeline = Pipeline([
    ("features", FunctionTransformer(ols_features)),
    ("ols", LinearRegression()),
])

cv_rows_ols = []
for month in VALIDATION_MONTHS:
    tr = train_df.loc[train_df["날짜"].between(20210101, month * 100 - 1)]
    va = train_df.loc[train_df["날짜"].floordiv(100).eq(month)]
    fold_model = clone(ols_pipeline).fit(tr, tr[POWER])
    prediction = fold_model.predict(va.drop(columns=POWER))
    cv_rows_ols.append({"month": month, **metrics(va[POWER], prediction)})

cv_scores_ols = pd.DataFrame(cv_rows_ols).set_index("month")
print("OLS 월별 확장 교차검증")
print(cv_scores_ols.to_string())
print("월평균 MAE:", cv_scores_ols["MAE"].mean())
cv_results["OLS"] = {"scores": cv_scores_ols}

# 최종 모델: 1~8월 전체(train_df) 학습
model_0 = clone(ols_pipeline).fit(train_df, train_df[POWER])


OLS 월별 확장 교차검증
              MAE       RMSE        R2
month                                 
202104  35.223609  41.697684  0.402412
202105  36.795449  43.739571  0.364793
202106  35.176090  40.919197  0.458009
202107  37.163334  44.002327  0.468148
202108  41.336262  48.941970  0.372494
월평균 MAE: 37.13894917716369


## 비교 모델

#### M242 (Huber 회귀)

- Huber 회귀, 원핫 인코딩·표준화, 시간 감쇠·생산량 0 표본가중, 월별 확장 윈도우 CV, MAE·RMSE·R²

In [6]:
# M242 전처리: 시간 열 손상 행(7/13, 7/15의 48행) 제외, 유효 시간 0~23
# 공유 train_df·test_df 유지, M242 전용 변수 사용
m242_train_df = train_df.loc[train_df["시간"].between(0, 23)]
m242_test_df = test_df.loc[test_df["시간"].between(0, 23)]
assert len(m242_test_df) == len(test_df)  # 9월 손상 행 없음 → 공유 test_df 사용

assert not m242_train_df.duplicated(["날짜", "시간"]).any()
assert not m242_test_df.duplicated(["날짜", "시간"]).any()
print(f"학습 {len(m242_train_df):,}시간 / 테스트 {len(m242_test_df):,}시간")

학습: 5,784시간, 별도 평가: 336시간


In [7]:
# M242: 생산량·기온 기반 Huber 회귀 (직전 전력·평균·공장인원 미사용)


def m242_features(frame):
    """날짜·시간·생산량·기온 4개 열 기반 피처"""
    assert frame["시간"].between(0, 23).all(), "hour must be 0..23"
    hour = frame["시간"]
    weekday = pd.to_datetime(frame["날짜"].astype(str), format="%Y%m%d").dt.dayofweek
    log_prod = np.log1p(frame["생산량"])
    on = log_prod.gt(0).where(log_prod.notna()).astype(float)  # 가동 여부
    hinge = (log_prod - np.log1p(100)).clip(lower=0)  # 생산량 100 초과분
    x = pd.DataFrame({
        "hour_weekday": hour.astype(str) + "_" + weekday.astype(str),  # 시간×요일 범주
        "기온": frame["기온"],
        "log_prod": log_prod,
        "on": on,
        "temp_on": frame["기온"] * on,
        "temp_above20": (frame["기온"] - 20).clip(lower=0),
    }, index=frame.index)
    for day in range(7):
        x[f"prod_day{day}"] = log_prod * weekday.eq(day)
    for h in range(24):
        x[f"prod_hour{h}"] = log_prod * hour.eq(h)
    for h in range(24):
        x[f"prod_hinge_hour{h}"] = hinge * hour.eq(h)
    x["prod_sin"] = log_prod * np.sin(2 * np.pi * hour / 24)
    x["prod_cos"] = log_prod * np.cos(2 * np.pi * hour / 24)
    return x


# 피처 생성 → 시간×요일 원핫 + 수치 열 표준화 → 출력 4개(15·30·45·60분)마다 Huber 회귀
m242_pipeline = Pipeline([
    ("features", FunctionTransformer(m242_features)),
    ("preprocess", ColumnTransformer(
        [("category", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False),
          ["hour_weekday"])],
        remainder=make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()),
    )),
    ("huber", MultiOutputRegressor(HuberRegressor(alpha=1.5, epsilon=1.05, max_iter=2000, tol=1e-7))),
])


def m242_sample_weight(train):
    """최근 가중(반감기 90일), 생산량 0 행 2배 가중"""
    date = pd.to_datetime(train["날짜"].astype(str), format="%Y%m%d")
    weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
    weight = np.where(train["생산량"].eq(0), 2.0, 1.0) * weight / weight.mean()
    return weight / weight.mean()


def fit_m242(train):
    model = clone(m242_pipeline)
    with threadpool_limits(limits=2):  # 연구 시 스레드 수 고정(재현용)
        return model.fit(train, train[POWER], huber__sample_weight=m242_sample_weight(train))


# 1~3월→4월, …, 1~7월→8월. 날짜 기준 분할, test_df 미사용
assert m242_train_df["날짜"].between(20210101, 20210831).all()
assert m242_train_df[POWER].notna().all().all()

cv_rows_m242, cv_predictions_m242 = [], []
for month in VALIDATION_MONTHS:
    tr = m242_train_df.loc[m242_train_df["날짜"].between(20210101, month * 100 - 1)]
    va = m242_train_df.loc[m242_train_df["날짜"].floordiv(100).eq(month)]
    assert not tr.empty and not va.empty
    assert tr["날짜"].max() < va["날짜"].min()
    fold_model = fit_m242(tr)
    prediction = pd.DataFrame(fold_model.predict(va.drop(columns=POWER)), index=va.index, columns=POWER)
    cv_rows_m242.append({"month": month, **metrics(va[POWER], prediction)})
    cv_predictions_m242.append(prediction.assign(month=month))

cv_scores_m242 = pd.DataFrame(cv_rows_m242).set_index("month")
cv_predictions_m242 = pd.concat(cv_predictions_m242).sort_index()
print("M242 월별 확장 교차검증")
print(cv_scores_m242.to_string())
print("월평균 MAE:", cv_scores_m242["MAE"].mean())

# 비교 모델도 동일 5개 검증월 점수 등록
# 예: cv_results["모델명"] = {"scores": 월별점수DataFrame}
cv_results["M242"] = {"scores": cv_scores_m242}

# 최종 모델: 동일 파이프라인, 1~8월 전체(m242_train_df) 재학습. model_1.predict(frame)
model_1 = fit_m242(m242_train_df)

M242 월별 확장 교차검증
              MAE       RMSE        R2
month                                 
202104  18.838011  33.147559  0.622357
202105  11.390720  24.806295  0.795690
202106  12.163612  20.233590  0.867479
202107  13.236095  24.133745  0.843767
202108   8.862750  15.363157  0.938168
월평균 MAE: 12.898237519577219


#### LightGBM

- LightGBM(L1 손실), Optuna TPE 튜닝·MedianPruner(RUN_TUNING 시), 월별 확장 윈도우 CV, MAE·RMSE·R²

In [8]:
# 출처: 013model_blend.ipynb LightGBM (01 공통 변수·model_2 인터페이스)
# 9월 예측·평가: '9월 테스트' 셀
RUN_TUNING = False  # True: Optuna 재튜닝(약 20분) / False: 012 최적값
N_TRIALS, TIMEOUT = 50, 20 * 60  # 튜닝 종료: 50회 또는 20분 중 먼저 도달(진행 중 시도는 완료)
LGBM_PARAMS_012 = dict(n_estimators=900, learning_rate=0.029476, num_leaves=7, min_child_samples=51,
                       subsample=0.934594, colsample_bytree=0.558586, reg_alpha=0.027533, reg_lambda=0.074693)
LGBM_DROP_COPIED = False  # True: 1~4월(전력이 5·6월 복사본) 학습 제외
COPY_END = 20210430
# 저장 파라미터: 012 출력 소수 6자리 반올림. 013 기록 CV 14.386
# LGBM_DROP_COPIED 기본값 False(013과 동일)

# 유효 시간만 유지, 누수 열 제거. 공유 train_df 유지
lgbm_train_df = (train_df.loc[train_df["시간"].between(0, 23)]
                 .drop(columns=["평균", "공장인원"])
                 .sort_values(["날짜", "시간"]).copy())
assert not lgbm_train_df.duplicated(["날짜", "시간"]).any()
assert lgbm_train_df[POWER].notna().all().all()


# 4-1. LightGBM 정의: 입력 열(날짜·시간·생산량·기온)만, 출력 4개(15·30·45·60분)별 모델


def lgbm_features(frame):
    """생산량 이웃 값은 같은 날짜 내에서만(첫·끝 시간 NaN → LightGBM 처리)"""
    assert frame["시간"].between(0, 23).all(), "hour must be 0..23"
    prod = frame.sort_values(["날짜", "시간"]).groupby("날짜")["생산량"]
    x = pd.DataFrame({
        "시간": frame["시간"],
        "weekday": pd.to_datetime(frame["날짜"].astype(str), format="%Y%m%d").dt.dayofweek,
        "생산량": frame["생산량"],
        "기온": frame["기온"],
        "day_prod": prod.transform("sum"),  # 하루 총생산량: 휴무일 구분
    }, index=frame.index)
    for k in (1, 2):
        x[f"prev{k}_prod"] = prod.shift(k)   # k시간 전 생산량
        x[f"next{k}_prod"] = prod.shift(-k)  # k시간 뒤 생산량
    return x


LGBM_BASE = dict(n_estimators=500, learning_rate=0.03, num_leaves=15, min_child_samples=20,
                 subsample=0.8, colsample_bytree=0.8, reg_alpha=1e-3, reg_lambda=1.0)


def make_lgbm(params):
    """params만 바꾼 LightGBM 파이프라인. subsample_freq=1 → subsample 적용"""
    return Pipeline([
        ("features", FunctionTransformer(lgbm_features)),
        ("lgbm", MultiOutputRegressor(lgb.LGBMRegressor(
            objective="regression_l1", subsample_freq=1, random_state=SEED, n_jobs=2, verbose=-1, **params))),
    ])


def lgbm_train_rows(train):
    """LGBM_DROP_COPIED=True: 1~4월(복사 구간) 제외. 남는 행 없으면(4·5월 검증) 원본 유지"""
    if not LGBM_DROP_COPIED:
        return train
    clean = train.loc[train["날짜"].gt(COPY_END)]
    return clean if len(clean) else train


def fit_lgbm(train, params):
    rows = lgbm_train_rows(train)
    return make_lgbm(params).fit(rows, rows[POWER])


def lgbm_folds(frame):
    """1~3월→4월, …, 1~7월→8월 순 (검증월, 학습, 검증). 날짜 기준 분할"""
    for month in VALIDATION_MONTHS:
        tr = frame.loc[frame["날짜"].between(20210101, month * 100 - 1)]
        va = frame.loc[frame["날짜"].floordiv(100).eq(month)]
        assert not tr.empty and not va.empty
        assert tr["날짜"].max() < va["날짜"].min()
        yield month, tr, va


def lgbm_cv(params, trial=None, predictions=None):
    """013과 동일 4~8월 확장 CV. 전처리·학습은 fold 학습 행에만 적합"""
    rows = []
    for step, (month, tr, va) in enumerate(lgbm_folds(lgbm_train_df)):
        fold_model = fit_lgbm(tr, params)
        prediction = fold_model.predict(va.drop(columns=POWER))
        if predictions is not None:  # 블렌딩용 OOF: 검증월 미학습 모델 예측만 저장
            predictions.append(pd.DataFrame(prediction, index=va.index, columns=POWER).assign(month=month))
        rows.append({"month": month, **metrics(va[POWER], prediction)})
        if trial is not None:
            trial.report(float(np.mean([row["MAE"] for row in rows])), step)
            if trial.should_prune():
                raise optuna.TrialPruned()
    return pd.DataFrame(rows).set_index("month")


# 4-2. 하이퍼파라미터: RUN_TUNING=True 시 Optuna(TPE), 4~8월 확장 CV 월평균 MAE 최소화. 9월 미사용


def lgbm_objective(trial):
    params = dict(
        n_estimators=trial.suggest_int("n_estimators", 200, 1500, step=100),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        num_leaves=trial.suggest_int("num_leaves", 7, 63, log=True),
        min_child_samples=trial.suggest_int("min_child_samples", 5, 100, log=True),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
    )
    return float(lgbm_cv(params, trial)["MAE"].mean())


if RUN_TUNING:
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study_lgbm = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1),
    )
    study_lgbm.enqueue_trial(LGBM_BASE)  # 첫 시도는 기본 설정 → 튜닝 전후 비교 기준
    study_lgbm.optimize(lgbm_objective, n_trials=N_TRIALS, timeout=TIMEOUT, show_progress_bar=True)
    done = study_lgbm.get_trials(states=[optuna.trial.TrialState.COMPLETE])
    assert done, "완료된 시도가 없습니다. 위 오류 메시지를 확인하세요."
    lgbm_params, tuning_trials = study_lgbm.best_params, len(done)
    print(f"완료 {len(done)}회 / 가지치기 {len(study_lgbm.trials) - len(done)}회 / 최적 월평균 MAE {study_lgbm.best_value:.3f}")
else:
    lgbm_params, tuning_trials = LGBM_PARAMS_012, None
print(pd.DataFrame({"기본": LGBM_BASE, "사용": lgbm_params}).to_string())
# 반올림 없는 딕셔너리 출력(재사용용)
print("\n사용 파라미터")
print(f"LGBM_PARAMS_012 = {lgbm_params!r}")


# 4-3. 기본·사용 파라미터 월별 CV, 사용 파라미터 결과 등록
cv_scores_lgbm_base = lgbm_cv(LGBM_BASE)
cv_predictions_lgbm = []
cv_scores_lgbm = lgbm_cv(lgbm_params, predictions=cv_predictions_lgbm)
cv_predictions_lgbm = pd.concat(cv_predictions_lgbm).sort_index()
print("LightGBM(기본) 월별 확장 교차검증")
print(cv_scores_lgbm_base.to_string())
print("월평균 MAE:", cv_scores_lgbm_base["MAE"].mean())
print("\nLightGBM 월별 확장 교차검증 (사용 파라미터)")
print(cv_scores_lgbm.to_string())
print("월평균 MAE:", cv_scores_lgbm["MAE"].mean())
cv_results["LightGBM"] = {"scores": cv_scores_lgbm}

# 최종 모델: 동일 설정, 1~8월 학습. predict(frame) → (행 수, 4)
model_2 = fit_lgbm(lgbm_train_df, lgbm_params)


                        기본          사용
n_estimators       500.000  900.000000
learning_rate        0.030    0.029476
num_leaves          15.000    7.000000
min_child_samples   20.000   51.000000
subsample            0.800    0.934594
colsample_bytree     0.800    0.558586
reg_alpha            0.001    0.027533
reg_lambda           1.000    0.074693

최종 사용 파라미터 (RUN_TUNING=True: 이번 최적값, False: 저장값):
LGBM_PARAMS_012 = {'n_estimators': 900, 'learning_rate': 0.029476, 'num_leaves': 7, 'min_child_samples': 51, 'subsample': 0.934594, 'colsample_bytree': 0.558586, 'reg_alpha': 0.027533, 'reg_lambda': 0.074693}
LightGBM(기본) 월별 확장 교차검증
              MAE       RMSE        R2
month                                 
202104  23.949701  38.817114  0.482125
202105  17.213884  30.664519  0.687796
202106  14.071321  21.317713  0.852898
202107  11.453167  16.945181  0.922978
202108  10.964083  17.868487  0.916357
월평균 MAE: 15.530431161638031

LightGBM 월별 확장 교차검증 (사용 파라미터)
              MAE       RMSE     

#### N207 (M242 + 주말 잔차 SVR)

- Huber 회귀 + 주말 잔차 RBF SVR, 월별 확장 윈도우 CV, MAE·RMSE·R²

In [9]:
# N207: 평일 M242, 주말 M242 + 주말 잔차 RBF SVR 보정. 기본 모델 fit_m242(M242 셀)
# 전처리: 시간 열 손상 행(7/13, 7/15의 48행) 제외. 공유 train_df 유지
n207_train_df = train_df.loc[train_df["시간"].between(0, 23)]


def is_weekend(frame):
    return pd.to_datetime(frame["날짜"].astype(str), format="%Y%m%d").dt.dayofweek.ge(5)


def n207_residual_features(frame):
    """주말 잔차 보정용: 시간·요일(범주), 기온·로그 생산량"""
    assert frame["시간"].between(0, 23).all(), "hour must be 0..23"
    return pd.DataFrame({
        "시간": frame["시간"],
        "기온": frame["기온"],
        "weekday": pd.to_datetime(frame["날짜"].astype(str), format="%Y%m%d").dt.dayofweek,
        "log_prod": np.log1p(frame["생산량"]),
    }, index=frame.index)


# 주말 잔차(정답 − M242 예측) 학습: 시간·요일 원핫 + 기온·로그 생산량 표준화 → 출력 4개마다 RBF SVR
n207_residual_pipeline = Pipeline([
    ("features", FunctionTransformer(n207_residual_features)),
    ("preprocess", ColumnTransformer(
        [("category", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["시간", "weekday"])],
        remainder=make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()),
    )),
    ("svr", MultiOutputRegressor(SVR(kernel="rbf", C=10, epsilon=1, gamma="scale", tol=1e-4))),
])


class N207Model:
    """M242(전체 행 학습) + 주말 행 잔차 보정기. predict(frame) → (행 수, 4)"""

    def fit(self, train):
        self.base = fit_m242(train)
        residual = train[POWER] - self.base.predict(train)
        weekend = is_weekend(train)
        with threadpool_limits(limits=2):
            self.residual = clone(n207_residual_pipeline).fit(train.loc[weekend], residual.loc[weekend])
        return self

    def predict(self, frame):
        pred = self.base.predict(frame)
        weekend = is_weekend(frame).to_numpy()
        if weekend.any():
            pred[weekend] += self.residual.predict(frame.loc[weekend])
        return pred


cv_rows_n207 = []
for month in VALIDATION_MONTHS:
    tr = n207_train_df.loc[n207_train_df["날짜"].between(20210101, month * 100 - 1)]
    va = n207_train_df.loc[n207_train_df["날짜"].floordiv(100).eq(month)]
    assert tr["날짜"].max() < va["날짜"].min()
    fold_model = N207Model().fit(tr)
    cv_rows_n207.append({"month": month, **metrics(va[POWER], fold_model.predict(va.drop(columns=POWER)))})

cv_scores_n207 = pd.DataFrame(cv_rows_n207).set_index("month")
print("N207 월별 확장 교차검증")
print(cv_scores_n207.to_string())
print("월평균 MAE:", cv_scores_n207["MAE"].mean())
cv_results["N207"] = {"scores": cv_scores_n207}

# 최종 모델: 1~8월 전체(n207_train_df) 학습
model_3 = N207Model().fit(n207_train_df)

N207 월별 확장 교차검증
              MAE       RMSE        R2
month                                 
202104  18.491166  32.689998  0.632711
202105  11.149117  24.093301  0.807266
202106  11.967688  19.894084  0.871889
202107  13.151393  23.795125  0.848121
202108   8.761287  15.072396  0.940486
월평균 MAE: 12.704130125378361


#### N045 (CatBoost)

- CatBoost(MAE 손실, Ordered boosting), 생산량 0 표본가중, 월별 확장 윈도우 CV, MAE·RMSE·R²

In [10]:
# N045: M242 생산량 표현 + CatBoost(MAE 손실, depth 4). m242_features(M242 셀) 사용
# 전처리: 시간 열 손상 행(7/13, 7/15의 48행) 제외. 공유 train_df 유지
n045_train_df = train_df.loc[train_df["시간"].between(0, 23)]


def n045_features(frame):
    """M242 피처 + 시간·요일 원값. 시간·요일·시간×요일은 CatBoost 범주형"""
    x = m242_features(frame)
    calendar = pd.DataFrame({
        "시간": frame["시간"],
        "weekday": pd.to_datetime(frame["날짜"].astype(str), format="%Y%m%d").dt.dayofweek,
    }, index=frame.index)
    return pd.concat([calendar[["시간"]], x[["기온"]], calendar[["weekday"]], x.drop(columns="기온")], axis=1)


# 피처 생성 → 출력 4개(15·30·45·60분)별 CatBoost. cat_features 튜플(sklearn clone 호환)
n045_pipeline = Pipeline([
    ("features", FunctionTransformer(n045_features)),
    ("catboost", MultiOutputRegressor(CatBoostRegressor(
        loss_function="MAE", iterations=600, learning_rate=0.04, depth=4, boosting_type="Ordered",
        cat_features=("시간", "weekday", "hour_weekday"), one_hot_max_size=24,
        random_seed=SEED, thread_count=2, allow_writing_files=False, verbose=False))),
])


def fit_n045(train):
    """생산량 0 행 2배 가중(평균 1 정규화)"""
    weight = np.where(train["생산량"].eq(0), 2.0, 1.0)
    model = clone(n045_pipeline)
    with threadpool_limits(limits=2):
        return model.fit(train, train[POWER], catboost__sample_weight=weight / weight.mean())


cv_rows_n045 = []
for month in VALIDATION_MONTHS:
    tr = n045_train_df.loc[n045_train_df["날짜"].between(20210101, month * 100 - 1)]
    va = n045_train_df.loc[n045_train_df["날짜"].floordiv(100).eq(month)]
    assert tr["날짜"].max() < va["날짜"].min()
    fold_model = fit_n045(tr)
    cv_rows_n045.append({"month": month, **metrics(va[POWER], fold_model.predict(va.drop(columns=POWER)))})

cv_scores_n045 = pd.DataFrame(cv_rows_n045).set_index("month")
print("N045 월별 확장 교차검증")
print(cv_scores_n045.to_string())
print("월평균 MAE:", cv_scores_n045["MAE"].mean())
cv_results["N045"] = {"scores": cv_scores_n045}

# 최종 모델: 1~8월 전체(n045_train_df) 학습
model_4 = fit_n045(n045_train_df)

N045 월별 확장 교차검증
              MAE       RMSE        R2
month                                 
202104  19.792803  33.821398  0.606847
202105  15.337564  30.398481  0.693190
202106  13.473643  21.450855  0.851055
202107  13.349868  22.764914  0.860987
202108  12.437614  21.293472  0.881219
월평균 MAE: 14.87829833695518


#### N002 (ExtraTrees)

- ExtraTrees, 중앙값 결측 대체, 월별 확장 윈도우 CV, MAE·RMSE·R²

In [22]:
# N002 원본: scripts/nonlinear_configs.json, scripts/monthly_research.py
# 원본 피처·모델 설정 유지, 시드·분할은 SEED·VALIDATION_MONTHS
from sklearn.ensemble import ExtraTreesRegressor

n002_train_df = train_df.loc[train_df["시간"].between(0, 23)].sort_values(["날짜", "시간"])


def n002_features(frame):
    """원본 시간·생산량·기온 + 요일 (전력 이력 미사용)"""
    x = frame[["시간", "생산량", "기온"]].copy()
    x["weekday"] = pd.to_datetime(frame["날짜"].astype(str), format="%Y%m%d").dt.dayofweek
    return x


# fold 학습 중앙값 결측 대체 → 4개 출력 공동 학습 ExtraTrees
n002_pipeline = Pipeline([
    ("features", FunctionTransformer(n002_features)),
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("extra_trees", ExtraTreesRegressor(
        n_estimators=300, criterion="squared_error", min_samples_leaf=5,
        max_features=1.0, random_state=SEED, n_jobs=2)),
])


def fit_n002(train):
    model = clone(n002_pipeline)
    with threadpool_limits(limits=2):
        model.fit(train, train[POWER])
    model.named_steps["extra_trees"].n_jobs = 1  # 원본과 동일 추론 합산 순서 고정
    return model


cv_rows_n002 = []
for month in VALIDATION_MONTHS:
    tr = n002_train_df.loc[n002_train_df["날짜"].between(20210101, month * 100 - 1)]
    va = n002_train_df.loc[n002_train_df["날짜"].floordiv(100).eq(month)]
    assert tr["날짜"].max() < va["날짜"].min()
    fold_model = fit_n002(tr)
    cv_rows_n002.append({"month": month, **metrics(va[POWER], fold_model.predict(va.drop(columns=POWER)))})

cv_scores_n002 = pd.DataFrame(cv_rows_n002).set_index("month")
print("N002 (ExtraTrees) 월별 확장 교차검증")
print(cv_scores_n002.to_string())
print("월평균 MAE:", cv_scores_n002["MAE"].mean())
cv_results["N002"] = {"scores": cv_scores_n002}

# 최종 모델: 1~8월 전체(n002_train_df) 학습
model_15 = fit_n002(n002_train_df)


N002 (ExtraTrees) 월별 확장 교차검증
              MAE       RMSE        R2
month                                 
202104  27.925340  40.186692  0.444936
202105  22.061791  31.785891  0.664544
202106  14.710268  19.446085  0.877594
202107  11.914614  17.846668  0.914565
202108  20.479236  34.667997  0.685143
월평균 MAE: 19.418249763782462


#### N049 (단독 RBF-SVR)

- RBF SVR, 원핫 인코딩·표준화, 월별 확장 윈도우 CV, MAE·RMSE·R²

In [23]:
# N049 원본: scripts/nonlinear_configs.json, scripts/monthly_research.py
# 주말 잔차 보정 없이 4개 출력 직접 예측. 분할: 공통 VALIDATION_MONTHS
n049_train_df = train_df.loc[train_df["시간"].between(0, 23)].sort_values(["날짜", "시간"])


def n049_features(frame):
    """시간·요일, 기온·로그 생산량 (전력 이력 미사용)"""
    x = frame[["시간", "기온"]].copy()
    x["weekday"] = pd.to_datetime(frame["날짜"].astype(str), format="%Y%m%d").dt.dayofweek
    x["log_prod"] = np.log1p(frame["생산량"])
    return x


# 시간·요일 전체 원핫 + fold 학습 중앙값 결측 대체·표준화 → 출력별 RBF-SVR.
n049_pipeline = Pipeline([
    ("features", FunctionTransformer(n049_features)),
    ("preprocess", ColumnTransformer([
        ("category", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["시간", "weekday"]),
        ("numeric", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()),
         ["기온", "log_prod"]),
    ])),
    ("svr", MultiOutputRegressor(SVR(kernel="rbf", C=10, epsilon=1, gamma="scale", tol=1e-4))),
])


def fit_n049(train):
    model = clone(n049_pipeline)
    with threadpool_limits(limits=2):
        return model.fit(train, train[POWER])


cv_rows_n049 = []
for month in VALIDATION_MONTHS:
    tr = n049_train_df.loc[n049_train_df["날짜"].between(20210101, month * 100 - 1)]
    va = n049_train_df.loc[n049_train_df["날짜"].floordiv(100).eq(month)]
    assert tr["날짜"].max() < va["날짜"].min()
    fold_model = fit_n049(tr)
    cv_rows_n049.append({"month": month, **metrics(va[POWER], fold_model.predict(va.drop(columns=POWER)))})

cv_scores_n049 = pd.DataFrame(cv_rows_n049).set_index("month")
print("N049 (RBF-SVR) 월별 확장 교차검증")
print(cv_scores_n049.to_string())
print("월평균 MAE:", cv_scores_n049["MAE"].mean())
cv_results["N049"] = {"scores": cv_scores_n049}

# 최종 모델: 1~8월 전체(n049_train_df) 학습
model_16 = fit_n049(n049_train_df)


N049 (RBF-SVR) 월별 확장 교차검증
              MAE       RMSE        R2
month                                 
202104  21.067044  34.609398  0.588314
202105  18.955017  30.666921  0.687747
202106  14.855824  21.641566  0.848394
202107  15.349609  23.439728  0.852624
202108  14.271804  21.949877  0.873783
월평균 MAE: 16.899859799430622


#### R192 (Huber + 날짜 추세 + 주말 잔차 RBF·poly SVR)

- Huber 회귀(날짜 추세 피처, 시간 감쇠 가중) + 주말 잔차 RBF·poly SVR, 가중 평균(.75/.25), 월별 확장 윈도우 CV, MAE·RMSE·R²

In [11]:
# R192: Huber(날짜 추세 포함) + 주말 잔차 SVR 두 개(RBF·poly)를 .75/.25로 조합.
# 재사용: m242_features·m242_pipeline(M242 셀), n207_residual_features(N207 셀)
# RHuberFeatures·RWeekendModel·R192Model: R100·R222·R191 셀에서도 사용
from sklearn.base import BaseEstimator, TransformerMixin


class RHuberFeatures(TransformerMixin, BaseEstimator):
    """날짜 추세 기준일 fit 시 고정(평가 시 재계산 없음)"""

    def __init__(self, trend=False):
        self.trend = trend

    def fit(self, frame, y=None):
        if self.trend:
            date = pd.to_datetime(frame["날짜"].astype(str), format="%Y%m%d")
            self.anchor_ = float(date.max().toordinal())
        return self

    def transform(self, frame):
        assert frame["생산량"].dropna().ge(0).all()
        x = m242_features(frame)
        if self.trend:
            date = pd.to_datetime(frame["날짜"].astype(str), format="%Y%m%d")
            ordinal = date.map(lambda value: value.toordinal()).astype(float)
            gate = frame["생산량"].eq(0) & date.dt.dayofweek.lt(5)
            x["bounded_date_trend"] = ((ordinal - self.anchor_) / 60).clip(-1, 0) * gate
        return x


class RWeekendModel:
    """전체 학습 행 Huber + 주말 학습 행 잔차 SVR. 출력 순서 = POWER"""

    def __init__(self, kernel="rbf", numeric_weight=1.0, zero_weight=2.0, trend=False):
        self.kernel = kernel
        self.numeric_weight = numeric_weight
        self.zero_weight = zero_weight
        self.trend = trend

    def fit(self, train):
        inputs = train[["날짜", "시간", "생산량", "기온"]]
        date = pd.to_datetime(inputs["날짜"].astype(str), format="%Y%m%d")
        weight = np.exp2(-(date.max() - date).dt.days / 90)
        weight = (weight / weight.mean()).to_numpy()
        weight = np.where(inputs["생산량"].eq(0), self.zero_weight, 1.0) * weight
        weight = weight / weight.mean()
        self.base = clone(m242_pipeline).set_params(features=RHuberFeatures(trend=self.trend))
        with threadpool_limits(limits=2):
            self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
        residual = train[POWER].to_numpy() - self.base.predict(inputs)
        weekend = date.dt.dayofweek.ge(5).to_numpy()
        assert weekend.any(), "주말 잔차 모델을 학습할 행이 필요합니다."
        self.residual = Pipeline([
            ("features", FunctionTransformer(n207_residual_features)),
            ("preprocess", ColumnTransformer([
                ("category", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                 ["시간", "weekday"]),
                ("numeric", make_pipeline(SimpleImputer(strategy="median", add_indicator=True),
                                          StandardScaler()), ["기온", "log_prod"]),
            ], transformer_weights={"numeric": self.numeric_weight})),
            ("svr", MultiOutputRegressor(SVR(
                kernel=self.kernel, degree=2, coef0=1, C=10, epsilon=1, gamma="scale", tol=1e-4))),
        ])
        # SVR 전처리·잔차 적합: 주말 학습 행만, 표본 가중치 없음
        with threadpool_limits(limits=2):
            self.residual.fit(inputs.loc[weekend], residual[weekend])
        return self

    def predict(self, frame):
        inputs = frame[["날짜", "시간", "생산량", "기온"]]
        prediction = self.base.predict(inputs)
        weekend = pd.to_datetime(inputs["날짜"].astype(str), format="%Y%m%d").dt.dayofweek.ge(5).to_numpy()
        if weekend.any():
            prediction[weekend] += self.residual.predict(inputs.loc[weekend])
        return prediction


class R192Model:
    """R192: 날짜 추세를 넣은 RWeekendModel 두 개(RBF 수치 가중 .5 / poly)를 .75/.25로 조합.
    zero_weight: Huber 생산량 0 행 가중치 (R192=1.0, R191=0.5)."""

    def __init__(self, zero_weight=1.0):
        self.zero_weight = zero_weight

    def fit(self, train):
        self.members = [
            RWeekendModel("rbf", numeric_weight=0.5, zero_weight=self.zero_weight, trend=True).fit(train),
            RWeekendModel("poly", zero_weight=self.zero_weight, trend=True).fit(train),
        ]
        return self

    def predict(self, frame):
        left, right = (member.predict(frame) for member in self.members)
        return 0.75 * left + 0.25 * right


# 전처리: 시간 열 손상 행 제외, 날짜·시간 정렬. 공유 train_df 유지
r192_train_df = train_df.loc[train_df["시간"].between(0, 23)].sort_values(["날짜", "시간"]).copy()
assert r192_train_df["날짜"].between(20210101, 20210831).all()
assert not r192_train_df.duplicated(["날짜", "시간"]).any()
assert r192_train_df[POWER].notna().all().all()

cv_rows_r192 = []
for month in VALIDATION_MONTHS:
    tr = r192_train_df.loc[r192_train_df["날짜"].between(20210101, month * 100 - 1)]
    va = r192_train_df.loc[r192_train_df["날짜"].floordiv(100).eq(month)]
    assert not tr.empty and not va.empty
    assert tr["날짜"].max() < va["날짜"].min()
    fold_model = R192Model().fit(tr)
    prediction = fold_model.predict(va[["날짜", "시간", "생산량", "기온"]])
    cv_rows_r192.append({"month": month, **metrics(va[POWER], prediction)})

cv_scores_r192 = pd.DataFrame(cv_rows_r192).set_index("month")
print("R192 월별 확장 교차검증 (중복 유지)")
print(cv_scores_r192.to_string())
print("월평균 MAE:", cv_scores_r192["MAE"].mean())
cv_results["R192"] = {"scores": cv_scores_r192}

# 최종 모델: 1~8월 전체(r192_train_df) 학습. 9월 미사용
model_5 = R192Model().fit(r192_train_df)  # predict(frame) → 입력 행 순서대로 (행 수, 4), 열=15·30·45·60분

R192 월별 확장 교차검증 (중복 유지)
              MAE       RMSE        R2
month                                 
202104  18.410742  32.527951  0.636343
202105  10.889871  23.981150  0.809056
202106  11.983098  19.894179  0.871888
202107  13.006175  23.620112  0.850347
202108   8.836524  15.212306  0.939376
월평균 MAE: 12.625282135596233


#### R100 (생산량 0: N207 / 그 외: R036)

- Huber 회귀 + 주말 잔차 RBF·poly SVR, 생산량 0 여부 분기, 월별 확장 윈도우 CV, MAE·RMSE·R²

In [12]:
# R100: 생산량 0 → N207(= M242 + 주말 잔차 RBF SVR), 그 외 → R036(= M242 + 주말 잔차 poly SVR)
# RWeekendModel(R192 셀) 사용. 구성 모델 모두 전체 fold 학습(생산량별 분할 없음)


class R100Model:
    def fit(self, train):
        self.members = [RWeekendModel("rbf").fit(train), RWeekendModel("poly").fit(train)]
        return self

    def predict(self, frame):
        left, right = (member.predict(frame) for member in self.members)
        zero = frame["생산량"].eq(0).to_numpy()[:, None]
        return np.where(zero, left, right)


# 전처리: 시간 열 손상 행 제외, 날짜·시간 정렬. 공유 train_df 유지
r100_train_df = train_df.loc[train_df["시간"].between(0, 23)].sort_values(["날짜", "시간"]).copy()

cv_rows_r100 = []
for month in VALIDATION_MONTHS:
    tr = r100_train_df.loc[r100_train_df["날짜"].between(20210101, month * 100 - 1)]
    va = r100_train_df.loc[r100_train_df["날짜"].floordiv(100).eq(month)]
    assert not tr.empty and not va.empty
    assert tr["날짜"].max() < va["날짜"].min()
    fold_model = R100Model().fit(tr)
    prediction = fold_model.predict(va[["날짜", "시간", "생산량", "기온"]])
    cv_rows_r100.append({"month": month, **metrics(va[POWER], prediction)})

cv_scores_r100 = pd.DataFrame(cv_rows_r100).set_index("month")
print("R100 월별 확장 교차검증 (중복 유지)")
print(cv_scores_r100.to_string())
print("월평균 MAE:", cv_scores_r100["MAE"].mean())
cv_results["R100"] = {"scores": cv_scores_r100}

# 최종 모델: 1~8월 전체(r100_train_df) 학습. 9월 미사용
model_6 = R100Model().fit(r100_train_df)  # predict(frame) → (행 수, 4)

R100 월별 확장 교차검증 (중복 유지)
              MAE       RMSE        R2
month                                 
202104  18.384826  32.532231  0.636248
202105  11.110556  24.005247  0.808672
202106  11.824614  19.780940  0.873342
202107  13.070920  23.722704  0.849044
202108   8.747809  15.066063  0.940536
월평균 MAE: 12.627745073120847


#### R222 (생산량 0: R192 / 그 외: R192·R036 조합)

- Huber 회귀 + 주말 잔차 RBF·poly SVR, 생산량 0 여부 분기 가중 평균, 월별 확장 윈도우 CV, MAE·RMSE·R²

In [13]:
# R222: 생산량 0 → R192, 그 외 → 0.25 × R192 + 0.75 × R036(= M242 + 주말 잔차 poly SVR)
# R192Model·RWeekendModel(R192 셀) 사용


class R222Model:
    def fit(self, train):
        self.members = [R192Model().fit(train), RWeekendModel("poly").fit(train)]
        return self

    def predict(self, frame):
        left, right = (member.predict(frame) for member in self.members)
        zero = frame["생산량"].eq(0).to_numpy()[:, None]
        return np.where(zero, left, 0.25 * left + 0.75 * right)


# 전처리: 시간 열 손상 행 제외, 날짜·시간 정렬. 공유 train_df 유지
r222_train_df = train_df.loc[train_df["시간"].between(0, 23)].sort_values(["날짜", "시간"]).copy()

cv_rows_r222 = []
for month in VALIDATION_MONTHS:
    tr = r222_train_df.loc[r222_train_df["날짜"].between(20210101, month * 100 - 1)]
    va = r222_train_df.loc[r222_train_df["날짜"].floordiv(100).eq(month)]
    assert not tr.empty and not va.empty
    assert tr["날짜"].max() < va["날짜"].min()
    fold_model = R222Model().fit(tr)
    prediction = fold_model.predict(va[["날짜", "시간", "생산량", "기온"]])
    cv_rows_r222.append({"month": month, **metrics(va[POWER], prediction)})

cv_scores_r222 = pd.DataFrame(cv_rows_r222).set_index("month")
print("R222 월별 확장 교차검증 (중복 유지)")
print(cv_scores_r222.to_string())
print("월평균 MAE:", cv_scores_r222["MAE"].mean())
cv_results["R222"] = {"scores": cv_scores_r222}

# 최종 모델: 1~8월 전체(r222_train_df) 학습. 9월 미사용
model_7 = R222Model().fit(r222_train_df)  # predict(frame) → (행 수, 4)

R222 월별 확장 교차검증 (중복 유지)
              MAE       RMSE        R2
month                                 
202104  18.404933  32.480212  0.637410
202105  10.927939  23.999896  0.808757
202106  11.876010  19.822993  0.872803
202107  13.081773  23.685937  0.849511
202108   8.891082  15.274581  0.938879
월평균 MAE: 12.636347484050972


#### R191 (R192, 생산량 0 가중치 0.5)

- Huber 회귀(날짜 추세 피처) + 주말 잔차 RBF·poly SVR, 가중 평균(.75/.25), 월별 확장 윈도우 CV, MAE·RMSE·R²

In [14]:
# R191: R192 구조, Huber 생산량 0 가중치 0.5. R192Model(R192 셀) 사용

# 전처리: 시간 열 손상 행 제외, 날짜·시간 정렬. 공유 train_df 유지
r191_train_df = train_df.loc[train_df["시간"].between(0, 23)].sort_values(["날짜", "시간"]).copy()

cv_rows_r191 = []
for month in VALIDATION_MONTHS:
    tr = r191_train_df.loc[r191_train_df["날짜"].between(20210101, month * 100 - 1)]
    va = r191_train_df.loc[r191_train_df["날짜"].floordiv(100).eq(month)]
    assert not tr.empty and not va.empty
    assert tr["날짜"].max() < va["날짜"].min()
    fold_model = R192Model(zero_weight=0.5).fit(tr)
    prediction = fold_model.predict(va[["날짜", "시간", "생산량", "기온"]])
    cv_rows_r191.append({"month": month, **metrics(va[POWER], prediction)})

cv_scores_r191 = pd.DataFrame(cv_rows_r191).set_index("month")
print("R191 월별 확장 교차검증 (중복 유지)")
print(cv_scores_r191.to_string())
print("월평균 MAE:", cv_scores_r191["MAE"].mean())
cv_results["R191"] = {"scores": cv_scores_r191}

# 최종 모델: 1~8월 전체(r191_train_df) 학습. 9월 미사용
model_8 = R192Model(zero_weight=0.5).fit(r191_train_df)  # predict(frame) → (행 수, 4)

R191 월별 확장 교차검증 (중복 유지)
              MAE       RMSE        R2
month                                 
202104  18.375800  32.579966  0.635179
202105  10.778950  23.970806  0.809221
202106  12.084041  20.043039  0.869964
202107  12.866403  23.530463  0.851481
202108   8.869561  15.305532  0.938631
월평균 MAE: 12.59495094501511


#### V081 (기온×생산 강도 + 가동 상태 피처 + 주말 잔차 SVR)

- Huber 회귀(가동 상태×시간, 기온×생산 강도 피처) + 주말 잔차 RBF·poly SVR, 가중 평균(.75/.25), 월별 확장 윈도우 CV, MAE·RMSE·R²

In [15]:
# V081 원본: scripts/research_v2/results/V081/source/
# 의존: 데이터 로딩·분할 셀의 df·cv_results만 (다른 모델 셀 독립)
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import HuberRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.svm import SVR
from threadpoolctl import threadpool_limits

V081_INPUTS = ["날짜", "시간", "생산량", "기온"]
V081_POWER = ["15분", "30분", "45분", "60분"]
V081_MONTHS = [202104, 202105, 202106, 202107, 202108]


def v081_metrics(y, pred):
    y, pred = np.asarray(y).ravel(), np.asarray(pred).ravel()
    return dict(MAE=mean_absolute_error(y, pred), RMSE=np.sqrt(mean_squared_error(y, pred)),
                R2=r2_score(y, pred))


def v081_is_weekend(frame):
    return pd.to_datetime(frame["날짜"].astype(str), format="%Y%m%d").dt.dayofweek.ge(5)


def v081_base_features(frame):
    """날짜·시간·생산량·기온 4개 열 기반 피처"""
    assert frame["시간"].between(0, 23).all(), "hour must be 0..23"
    hour = frame["시간"]
    weekday = pd.to_datetime(frame["날짜"].astype(str), format="%Y%m%d").dt.dayofweek
    log_prod = np.log1p(frame["생산량"])
    on = log_prod.gt(0).where(log_prod.notna()).astype(float)  # 가동 여부
    hinge = (log_prod - np.log1p(100)).clip(lower=0)  # 생산량 100 초과분
    x = pd.DataFrame({
        "hour_weekday": hour.astype(str) + "_" + weekday.astype(str),  # 시간×요일 범주
        "기온": frame["기온"],
        "log_prod": log_prod,
        "on": on,
        "temp_on": frame["기온"] * on,
        "temp_above20": (frame["기온"] - 20).clip(lower=0),
    }, index=frame.index)
    for day in range(7):
        x[f"prod_day{day}"] = log_prod * weekday.eq(day)
    for h in range(24):
        x[f"prod_hour{h}"] = log_prod * hour.eq(h)
    for h in range(24):
        x[f"prod_hinge_hour{h}"] = hinge * hour.eq(h)
    x["prod_sin"] = log_prod * np.sin(2 * np.pi * hour / 24)
    x["prod_cos"] = log_prod * np.cos(2 * np.pi * hour / 24)
    return x


class V081Features(TransformerMixin, BaseEstimator):
    """M242 + 학습 기준일 추세 + 가동×시간×평일/주말 + 기온×생산 강도."""

    def fit(self, frame, y=None):
        date = pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d')
        self.anchor_ = float(date.max().toordinal())
        return self

    def transform(self, frame):
        assert frame['생산량'].dropna().ge(0).all()
        x = v081_base_features(frame)
        date = pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d')
        ordinal = date.map(lambda value: value.toordinal()).astype(float)
        gate = frame['생산량'].eq(0) & date.dt.dayofweek.lt(5)
        x['bounded_date_trend'] = ((ordinal - self.anchor_) / 60).clip(-1, 0) * gate
        weekday = date.dt.dayofweek
        groups = weekday.ge(5).astype(int)
        logprod = np.log1p(frame["생산량"])
        signal = logprod.gt(0).where(logprod.notna()).astype(float)
        interactions = {
            f"state_hour_{hour}_group_{group}": signal * frame["시간"].eq(hour) * groups.eq(group)
            for hour in range(24) for group in range(2)
        }
        features = pd.concat([x, pd.DataFrame(interactions, index=frame.index)], axis=1)
        log_prod = np.log1p(frame["생산량"])
        features["temperature_intensity"] = frame["기온"] * log_prod
        features["cooling_intensity"] = (frame["기온"] - 20).clip(lower=0) * log_prod
        return features


v081_base_pipeline = Pipeline([
    ("features", FunctionTransformer(v081_base_features)),
    ("preprocess", ColumnTransformer(
        [("category", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False),
          ["hour_weekday"])],
        remainder=make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()),
    )),
    ("huber", MultiOutputRegressor(HuberRegressor(alpha=1.5, epsilon=1.05, max_iter=2000, tol=1e-7))),
])


def v081_residual_features(frame):
    """주말 잔차 보정용: 시간·요일(범주), 기온·로그 생산량"""
    assert frame["시간"].between(0, 23).all(), "hour must be 0..23"
    return pd.DataFrame({
        "시간": frame["시간"],
        "기온": frame["기온"],
        "weekday": pd.to_datetime(frame["날짜"].astype(str), format="%Y%m%d").dt.dayofweek,
        "log_prod": np.log1p(frame["생산량"]),
    }, index=frame.index)


v081_residual_pipeline = Pipeline([
    ("features", FunctionTransformer(v081_residual_features)),
    ("preprocess", ColumnTransformer(
        [("category", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["시간", "weekday"])],
        remainder=make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()),
    )),
    ("svr", MultiOutputRegressor(SVR(kernel="rbf", C=10, epsilon=1, gamma="scale", tol=1e-4))),
])


class V081Model:
    """Huber 본체 + 주말 잔차 RBF/poly SVR(.75/.25). 출력 열은 V081_POWER 순서."""

    def fit(self, train):
        inputs = train[['날짜', '시간', '생산량', '기온']]
        date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
        weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
        weight *= np.where(inputs['생산량'].eq(0), .5, 1.)
        weight /= weight.mean()
        self.base = clone(v081_base_pipeline).set_params(features=V081Features())
        self.base.fit(inputs, train[V081_POWER], huber__sample_weight=weight)
        residual = train[V081_POWER].to_numpy() - self.base.predict(inputs)
        weekend = v081_is_weekend(inputs).to_numpy()
        self.corrections = []
        for kernel, numeric_weight in [('rbf', .5), ('poly', 1.)]:
            model = clone(v081_residual_pipeline).set_params(
                preprocess__transformer_weights={'remainder': numeric_weight},
                svr=MultiOutputRegressor(SVR(kernel=kernel, degree=2, coef0=1, C=10,
                                            epsilon=1, gamma='scale', tol=1e-4)),
            )
            model.fit(inputs.loc[weekend], residual[weekend])
            self.corrections.append(model)
        return self

    def predict(self, frame):
        inputs = frame[["날짜", "시간", "생산량", "기온"]]
        prediction = self.base.predict(inputs)
        weekend = v081_is_weekend(inputs).to_numpy()
        if weekend.any():
            prediction[weekend] += (.75 * self.corrections[0].predict(inputs.loc[weekend])
                                    + .25 * self.corrections[1].predict(inputs.loc[weekend]))
        return prediction


# 원본 전처리: 1~8월, 유효 시간만, 날짜·시간 정렬, 중복 곡선 유지
v081_train_df = df.loc[
    df["날짜"].between(20210101, 20210831) & df["시간"].between(0, 23),
    V081_INPUTS + V081_POWER,
].sort_values(["날짜", "시간"]).copy()
assert len(v081_train_df) == 5784
assert not v081_train_df.duplicated(["날짜", "시간"]).any()
assert v081_train_df[V081_POWER].notna().all().all()

# 원본 월별 확장 CV: 1~3월→4월, …, 1~7월→8월. 9월 미사용
v081_cv_rows, v081_cv_predictions = [], []
with threadpool_limits(limits=2):
    for v081_month in V081_MONTHS:
        v081_tr = v081_train_df.loc[v081_train_df["날짜"].between(20210101, v081_month * 100 - 1)]
        v081_va = v081_train_df.loc[v081_train_df["날짜"].floordiv(100).eq(v081_month)]
        assert not v081_tr.empty and not v081_va.empty
        assert v081_tr["날짜"].max() < v081_va["날짜"].min()
        v081_fold_model = V081Model().fit(v081_tr)
        v081_prediction = v081_fold_model.predict(v081_va[V081_INPUTS])
        assert v081_prediction.shape == (len(v081_va), 4) and np.isfinite(v081_prediction).all()
        v081_cv_rows.append({"month": v081_month, **v081_metrics(v081_va[V081_POWER], v081_prediction)})
        v081_cv_predictions.append(
            pd.DataFrame(v081_prediction, index=v081_va.index, columns=V081_POWER).assign(month=v081_month)
        )

    # 최종 모델: CV 동일 설정, 1~8월 전체 학습
    model_9 = V081Model().fit(v081_train_df)

cv_scores_v081 = pd.DataFrame(v081_cv_rows).set_index("month")
cv_predictions_v081 = pd.concat(v081_cv_predictions).sort_index()
cv_results["V081"] = {"scores": cv_scores_v081}
print("V081 월별 확장 교차검증")
print(cv_scores_v081.to_string())
print("월평균 MAE:", cv_scores_v081["MAE"].mean())
print("연구 기록 MAE 12.31683051652715")


V081 월별 확장 교차검증
              MAE       RMSE        R2
month                                 
202104  17.958608  32.343720  0.640451
202105  10.797763  23.862900  0.810934
202106  12.244766  20.320396  0.866340
202107  12.334196  23.247550  0.855030
202108   8.248819  15.032543  0.940800
월평균 MAE: 12.31683051652715
기존 연구 기록 MAE: 12.31683051652715 (현재 실행 결과와 비교용)


#### 블렌딩 (M242 + LightGBM)

- OOF 예측 가중 평균, 구간별(전역·가동여부·가동×주말) 비중 격자 탐색, 전진식 검증, MAE·RMSE·R²

In [16]:
# 출처: 013model_blend.ipynb 5장 블렌딩. M242·LightGBM 셀의 OOF·최종 모델 재사용
# 9월 예측·평가: '9월 테스트' 셀
BLEND_GRID = np.round(np.arange(0, 1.0001, 0.05), 2)
FIRST_MONTH_W = 0.5
MIN_REGIME_ROWS = 200
ORIGINAL_MONTHS = [202107, 202108]
blend_oof_predictions = {}


# 5-1. 두 모델 검증월 예측 행 일치 확인, 전역 비중별 CV MAE
pred_m242 = cv_predictions_m242[POWER]
pred_lgbm = cv_predictions_lgbm[POWER]
assert pred_m242.index.equals(pred_lgbm.index)
assert pred_m242.index.is_unique
assert cv_predictions_m242["month"].equals(cv_predictions_lgbm["month"])
assert np.isfinite(pred_m242.to_numpy()).all() and np.isfinite(pred_lgbm.to_numpy()).all()
oof_truth = lgbm_train_df.loc[pred_m242.index, POWER]
oof_month = lgbm_train_df.loc[pred_m242.index, "날짜"].floordiv(100)


def blend_predictions(pred_a, pred_b, w):
    """w × pred_b(LightGBM) + (1 − w) × pred_a(M242). w: 스칼라 또는 행별 비중 Series"""
    return pred_b.mul(w, axis=0) + pred_a.mul(1 - w, axis=0)


grid_rows = []
for w in BLEND_GRID:
    abs_err = (oof_truth - blend_predictions(pred_m242, pred_lgbm, w)).abs().mean(axis=1)
    month_mae = abs_err.groupby(oof_month).mean()
    grid_rows.append({"LightGBM 비중": w, "월평균 MAE": month_mae.mean(),
                      "7~8월(원본)": month_mae.loc[ORIGINAL_MONTHS].mean()})
blend_grid = pd.DataFrame(grid_rows).set_index("LightGBM 비중")
print(blend_grid.round(3).to_string())
print("최소 MAE 비중 월평균", blend_grid["월평균 MAE"].idxmin(), "/ 7~8월(원본)", blend_grid["7~8월(원본)"].idxmin())


# 5-2. 블렌딩 구간·비중 선택 함수. 구간 기준: 입력 열(날짜·생산량)


def regime_all(frame):
    return pd.Series("전체", index=frame.index)


def regime_on(frame):
    return pd.Series(np.where(frame["생산량"].gt(0), "가동", "비가동"), index=frame.index)


def regime_on_weekend(frame):
    return regime_on(frame) + np.where(pd.to_datetime(frame["날짜"].astype(str), format="%Y%m%d").dt.dayofweek.ge(5), "·주말", "·평일")


BLEND_REGIMES = {"전역": regime_all, "가동여부": regime_on, "가동×주말": regime_on_weekend}
oof_regimes = {name: fn(lgbm_train_df.loc[oof_truth.index]) for name, fn in BLEND_REGIMES.items()}


def blend_mae(rows, w):
    """OOF 행 rows, LightGBM 비중 w 블렌딩 예측 MAE"""
    pred = blend_predictions(pred_m242.loc[rows], pred_lgbm.loc[rows], w)
    return float((oof_truth.loc[rows] - pred).abs().to_numpy().mean())


def choose_weights(rows, regimes):
    """rows 내 구간별 MAE 최소 비중. 행 부족 구간은 rows 전체 최적 비중
    "(기본)": 전체 최적 비중, 선택 시 없던 구간용"""
    def best(idx):
        return float(BLEND_GRID[np.argmin([blend_mae(idx, w) for w in BLEND_GRID])])

    overall = best(rows)
    sub = regimes.loc[rows]
    weights = {name: best(idx) if len(idx) >= MIN_REGIME_ROWS else overall
               for name, idx in sub.groupby(sub).groups.items()}
    return {"(기본)": overall, **weights}


def weights_for(weights, regimes):
    """구간 이름 → 비중. 선택 시 없던 구간은 기본 비중"""
    return regimes.map(weights).fillna(weights["(기본)"]).astype(float)


def weight_table(weights, regimes):
    """구간별 OOF 행 수, 적용 LightGBM 비중, 기본 비중 사용 여부"""
    counts = regimes.value_counts().sort_index()
    return pd.DataFrame({"행수": counts, "LightGBM 비중": [weights.get(r, weights["(기본)"]) for r in counts.index],
                         "기본 비중 사용": counts.lt(MIN_REGIME_ROWS)})


def register_blend(name, pred):
    """검증월 예측 등록, 월별 MAE·RMSE·R2 출력"""
    month = lgbm_train_df.loc[pred.index, "날짜"].floordiv(100)
    scores = pd.DataFrame([
        {"month": m, **metrics(lgbm_train_df.loc[idx, POWER], pred.loc[idx, POWER])}
        for m, idx in pred.groupby(month).groups.items()
    ]).set_index("month").sort_index()
    blend_oof_predictions[name], cv_results[name] = pred, {"scores": scores}
    print(f"[{name}] 월별 확장 CV")
    print(scores.round(3).to_string())
    print(f"월평균 MAE {scores['MAE'].mean():.3f} / 원본 7~8월 {scores.loc[ORIGINAL_MONTHS, 'MAE'].mean():.3f}\n")
    return scores


# 5-3. 전진식 검증: 비중 선택 포함 CV 점수, cv_results에 "Blend(구간 방식)" 등록


def forward_blend(regimes):
    """검증월 m 비중: m 이전 검증월 OOF로만 선택. 첫 검증월은 FIRST_MONTH_W"""
    names = sorted(regimes.unique())
    parts, history = [], []
    for month in VALIDATION_MONTHS:
        past, now = oof_month.index[oof_month.lt(month)], oof_month.index[oof_month.eq(month)]
        weights = choose_weights(past, regimes) if len(past) else {"(기본)": FIRST_MONTH_W}
        parts.append(blend_predictions(pred_m242.loc[now], pred_lgbm.loc[now], weights_for(weights, regimes.loc[now])))
        history.append({"month": month, **{r: weights.get(r, weights["(기본)"]) for r in names}})
    return pd.concat(parts), pd.DataFrame(history).set_index("month")


blend_weight_history = {}
for name, regimes in oof_regimes.items():
    pred, blend_weight_history[name] = forward_blend(regimes)
    register_blend(f"Blend({name})", pred)
    print(f"검증월별 LightGBM 비중 [{name}]")
    print(blend_weight_history[name].to_string(), "\n")


# 5-4. 최종 블렌딩: 전진식 CV 월평균 MAE 최소 구간 방식, 비중은 4~8월 OOF 전체로 재선택


class BlendModel:
    """M242·LightGBM 구간별 비중 블렌딩. predict(frame) → (행 수, 4)"""

    def __init__(self, model_m242, model_lgbm, weights, regime_fn):
        self.model_m242, self.model_lgbm = model_m242, model_lgbm
        self.weights, self.regime_fn = weights, regime_fn

    def predict(self, frame):
        a = pd.DataFrame(self.model_m242.predict(frame), index=frame.index, columns=POWER)
        b = pd.DataFrame(self.model_lgbm.predict(frame), index=frame.index, columns=POWER)
        return blend_predictions(a, b, weights_for(self.weights, self.regime_fn(frame))).to_numpy()


blend_cv_mae = pd.Series({name: cv_results[f"Blend({name})"]["scores"]["MAE"].mean() for name in BLEND_REGIMES})
BEST_REGIME = blend_cv_mae.idxmin()
SELECTED_MODEL = f"Blend({BEST_REGIME})"
final_blend_weights = {name: choose_weights(oof_truth.index, regimes) for name, regimes in oof_regimes.items()}
blend_models = {}
for name, weights in final_blend_weights.items():
    blend_models[f"Blend({name})"] = BlendModel(model_1, model_2,
                                                weights, BLEND_REGIMES[name])
print("전진식 CV 월평균 MAE")
print(blend_cv_mae.round(3).to_string())

# 검증월별 MAE 차이(선택 모델 − 비교 모델). 4월 Blend 비중 고정(0.5)
monthly_mae = pd.DataFrame({name: result["scores"]["MAE"] for name, result in cv_results.items()})
paired = pd.DataFrame({f"− {other}": monthly_mae[SELECTED_MODEL] - monthly_mae[other]
                       for other in ["Blend(전역)", "M242", "LightGBM"] if other != SELECTED_MODEL})
print(f"\n검증월별 MAE 차이 ({SELECTED_MODEL} − 비교 모델)")
print(paired.round(3).to_string())

print(f"\n최종 비중 [{BEST_REGIME}] (4~8월 OOF)")
print(weight_table(final_blend_weights[BEST_REGIME], oof_regimes[BEST_REGIME]).to_string())
# 민감도: 원본 7~8월 OOF만으로 선택한 비중
original_rows = oof_month.index[oof_month.isin(ORIGINAL_MONTHS)]
print("\n7~8월(원본) 기준 비중", choose_weights(original_rows, oof_regimes[BEST_REGIME]))
# CV 선택 최종 블렌딩 모델. 세 방식 모두 9월 비교 표 등록
model_10 = blend_models[SELECTED_MODEL]
print(f"\n최종 블렌딩 {SELECTED_MODEL}")
print(f"최종 LightGBM 비중 {final_blend_weights[BEST_REGIME]!r}")


             월평균 MAE  7~8월(원본)
LightGBM 비중                   
0.00          12.898    11.049
0.05          12.818    10.923
0.10          12.780    10.820
0.15          12.761    10.730
0.20          12.757    10.653
0.25          12.766    10.587
0.30          12.788    10.530
0.35          12.820    10.483
0.40          12.863    10.444
0.45          12.916    10.413
0.50          12.979    10.391
0.55          13.051    10.374
0.60          13.134    10.363
0.65          13.230    10.359
0.70          13.337    10.364
0.75          13.461    10.376
0.80          13.601    10.398
0.85          13.762    10.436
0.90          13.948    10.494
0.95          14.156    10.574
1.00          14.386    10.671
MAE가 가장 작은 비중 — 월평균: 0.2 / 7~8월(원본): 0.65
[Blend(전역)] 월별 확장 CV
           MAE    RMSE     R2
month                        
202104  19.292  31.407  0.661
202105  11.433  24.345  0.803
202106  12.044  19.721  0.874
202107  13.080  23.546  0.851
202108   8.623  14.141  0.948
월평균 MAE: 12.89

#### V305 (3개 모델의 중앙값 블렌딩)

- 구성 모델 3개(V299·V303·V258) 예측 중앙값, 월별 확장 윈도우 CV, MAE·RMSE·R²

In [17]:
# V305: 전체 파이프라인 단일 셀 (외부 연구 모듈 import 불필요)
# 원본: scripts/research_v2/cells_phase2/V305.py
# 원본 SHA256: 7e3684a5e45574a659544df5e2425583603e3e40c2610ae31fddc31f85926926
# 내부 이름 함수 범위 한정 (앞선 모델 클래스·피처·설정 보호)
# 9월 예측·평가: '9월 테스트' 셀
def train_v305(train_df):
    # %% V305: standalone cell for notebooks/01model.ipynb
    # Requires train_df; test_df is never scored by this cell.

    # Source: notebook_models.py
    from collections import Counter, defaultdict
    import pandas as pd
    from sklearn.linear_model import LinearRegression
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import HuberRegressor
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.pipeline import Pipeline, make_pipeline
    from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
    from sklearn.svm import SVR
    from threadpoolctl import threadpool_limits
    import numpy as np
    import pandas as pd
    SEED = 42
    POWER = ['15분', '30분', '45분', '60분']
    VALIDATION_MONTHS = [202104, 202105, 202106, 202107, 202108]

    def metrics(y, pred):
        y, pred = (np.asarray(y).ravel(), np.asarray(pred).ravel())
        return dict(MAE=mean_absolute_error(y, pred), RMSE=np.sqrt(mean_squared_error(y, pred)), R2=r2_score(y, pred))

    def m242_features(frame):
        """날짜·시간·생산량·기온 4개 열 기반 피처"""
        assert frame['시간'].between(0, 23).all(), 'hour must be 0..23'
        hour = frame['시간']
        weekday = pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d').dt.dayofweek
        log_prod = np.log1p(frame['생산량'])
        on = log_prod.gt(0).where(log_prod.notna()).astype(float)
        hinge = (log_prod - np.log1p(100)).clip(lower=0)
        x = pd.DataFrame({'hour_weekday': hour.astype(str) + '_' + weekday.astype(str), '기온': frame['기온'], 'log_prod': log_prod, 'on': on, 'temp_on': frame['기온'] * on, 'temp_above20': (frame['기온'] - 20).clip(lower=0)}, index=frame.index)
        for day in range(7):
            x[f'prod_day{day}'] = log_prod * weekday.eq(day)
        for h in range(24):
            x[f'prod_hour{h}'] = log_prod * hour.eq(h)
        for h in range(24):
            x[f'prod_hinge_hour{h}'] = hinge * hour.eq(h)
        x['prod_sin'] = log_prod * np.sin(2 * np.pi * hour / 24)
        x['prod_cos'] = log_prod * np.cos(2 * np.pi * hour / 24)
        return x
    m242_pipeline = Pipeline([('features', FunctionTransformer(m242_features)), ('preprocess', ColumnTransformer([('category', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), ['hour_weekday'])], remainder=make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler()))), ('huber', MultiOutputRegressor(HuberRegressor(alpha=1.5, epsilon=1.05, max_iter=2000, tol=1e-07)))])

    def m242_sample_weight(train):
        """최근 가중(반감기 90일), 생산량 0 행 2배 가중"""
        date = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
        weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
        weight = np.where(train['생산량'].eq(0), 2.0, 1.0) * weight / weight.mean()
        return weight / weight.mean()

    def fit_m242(train):
        model = clone(m242_pipeline)
        with threadpool_limits(limits=2):
            return model.fit(train, train[POWER], huber__sample_weight=m242_sample_weight(train))

    def is_weekend(frame):
        return pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d').dt.dayofweek.ge(5)

    def n207_residual_features(frame):
        """주말 잔차 보정용: 시간·요일(범주), 기온·로그 생산량"""
        assert frame['시간'].between(0, 23).all(), 'hour must be 0..23'
        return pd.DataFrame({'시간': frame['시간'], '기온': frame['기온'], 'weekday': pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d').dt.dayofweek, 'log_prod': np.log1p(frame['생산량'])}, index=frame.index)
    n207_residual_pipeline = Pipeline([('features', FunctionTransformer(n207_residual_features)), ('preprocess', ColumnTransformer([('category', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['시간', 'weekday'])], remainder=make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler()))), ('svr', MultiOutputRegressor(SVR(kernel='rbf', C=10, epsilon=1, gamma='scale', tol=0.0001)))])

    class N207Model:
        """M242(전체 행 학습) + 주말 행 잔차 보정기. predict(frame) → (행 수, 4)"""

        def fit(self, train):
            self.base = fit_m242(train)
            residual = train[POWER] - self.base.predict(train)
            weekend = is_weekend(train)
            with threadpool_limits(limits=2):
                self.residual = clone(n207_residual_pipeline).fit(train.loc[weekend], residual.loc[weekend])
            return self

        def predict(self, frame):
            pred = self.base.predict(frame)
            weekend = is_weekend(frame).to_numpy()
            if weekend.any():
                pred[weekend] += self.residual.predict(frame.loc[weekend])
            return pred
    from sklearn.base import BaseEstimator, TransformerMixin

    class RHuberFeatures(TransformerMixin, BaseEstimator):
        """날짜 추세 기준일 fit 시 고정(평가 시 재계산 없음)"""

        def __init__(self, trend=False):
            self.trend = trend

        def fit(self, frame, y=None):
            if self.trend:
                date = pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d')
                self.anchor_ = float(date.max().toordinal())
            return self

        def transform(self, frame):
            assert frame['생산량'].dropna().ge(0).all()
            x = m242_features(frame)
            if self.trend:
                date = pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d')
                ordinal = date.map(lambda value: value.toordinal()).astype(float)
                gate = frame['생산량'].eq(0) & date.dt.dayofweek.lt(5)
                x['bounded_date_trend'] = ((ordinal - self.anchor_) / 60).clip(-1, 0) * gate
            return x

    class RWeekendModel:
        """전체 학습 행 Huber + 주말 학습 행 잔차 SVR. 출력 순서 = POWER"""

        def __init__(self, kernel='rbf', numeric_weight=1.0, zero_weight=2.0, trend=False):
            self.kernel = kernel
            self.numeric_weight = numeric_weight
            self.zero_weight = zero_weight
            self.trend = trend

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days / 90)
            weight = (weight / weight.mean()).to_numpy()
            weight = np.where(inputs['생산량'].eq(0), self.zero_weight, 1.0) * weight
            weight = weight / weight.mean()
            self.base = clone(m242_pipeline).set_params(features=RHuberFeatures(trend=self.trend))
            with threadpool_limits(limits=2):
                self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = date.dt.dayofweek.ge(5).to_numpy()
            assert weekend.any(), '주말 잔차 모델을 학습할 행이 필요합니다.'
            self.residual = Pipeline([('features', FunctionTransformer(n207_residual_features)), ('preprocess', ColumnTransformer([('category', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['시간', 'weekday']), ('numeric', make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler()), ['기온', 'log_prod'])], transformer_weights={'numeric': self.numeric_weight})), ('svr', MultiOutputRegressor(SVR(kernel=self.kernel, degree=2, coef0=1, C=10, epsilon=1, gamma='scale', tol=0.0001)))])
            with threadpool_limits(limits=2):
                self.residual.fit(inputs.loc[weekend], residual[weekend])
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            prediction = self.base.predict(inputs)
            weekend = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d').dt.dayofweek.ge(5).to_numpy()
            if weekend.any():
                prediction[weekend] += self.residual.predict(inputs.loc[weekend])
            return prediction

    class R192Model:
        """R192: 날짜 추세를 넣은 RWeekendModel 두 개(RBF 수치 가중 .5 / poly)를 .75/.25로 조합.
        zero_weight: Huber 생산량 0 행 가중치 (R192=1.0, R191=0.5)."""

        def __init__(self, zero_weight=1.0):
            self.zero_weight = zero_weight

        def fit(self, train):
            self.members = [RWeekendModel('rbf', numeric_weight=0.5, zero_weight=self.zero_weight, trend=True).fit(train), RWeekendModel('poly', zero_weight=self.zero_weight, trend=True).fit(train)]
            return self

        def predict(self, frame):
            left, right = (member.predict(frame) for member in self.members)
            return 0.75 * left + 0.25 * right

    class R100Model:

        def fit(self, train):
            self.members = [RWeekendModel('rbf').fit(train), RWeekendModel('poly').fit(train)]
            return self

        def predict(self, frame):
            left, right = (member.predict(frame) for member in self.members)
            zero = frame['생산량'].eq(0).to_numpy()[:, None]
            return np.where(zero, left, right)

    # Source: phase2_regimes.py
    """Historical-fit, rowwise regimes for conditional multi-model blending."""
    import numpy as np

    def fit_regime(inputs, grouping):
        if grouping == 'temperature':
            return {'group': grouping, 'cut_points': [18.0, 26.0]}
        if grouping == 'production_band':
            production = inputs['생산량'].to_numpy(dtype=float)
            positive = production[np.isfinite(production) & (production > 0)]
            return {'group': grouping, 'positive_q25': float(np.quantile(positive, 0.25))}
        raise ValueError(grouping)

    def apply_regime(inputs, state):
        if state['group'] == 'temperature':
            values = inputs['기온'].to_numpy(dtype=float)
            groups = np.searchsorted(state['cut_points'], values, side='right')
        elif state['group'] == 'production_band':
            values = inputs['생산량'].to_numpy(dtype=float)
            groups = np.where(values <= 0, 0, np.where(values <= state['positive_q25'], 1, 2))
        else:
            raise ValueError(state['group'])
        return np.where(np.isfinite(values), groups, -1).astype(int)

    # Source: observed_state_models.py
    """Direct regression within an observed zero-production state."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from threadpoolctl import threadpool_limits
    pass

    class ObservedStateModel:

        def __init__(self, config, seed=42):
            self.region = config.get('region', 'zero')
            assert self.region in ('zero', 'weekday_zero')

        def region_mask(self, frame):
            selected = frame['생산량'].eq(0)
            if self.region == 'weekday_zero':
                date = pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d')
                selected &= date.dt.dayofweek.lt(5)
            return selected

        def fit(self, train):
            region = train.loc[self.region_mask(train)]
            inputs = region[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            weight /= weight.mean()
            self.specialist = clone(m242_pipeline).set_params(features=RHuberFeatures(trend=True))
            with threadpool_limits(limits=2):
                self.base = R192Model(zero_weight=0.5).fit(train)
                self.specialist.fit(inputs, region[POWER], huber__sample_weight=weight)
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            prediction = self.base.predict(inputs)
            selected = self.region_mask(inputs).to_numpy()
            if selected.any():
                prediction[selected] = self.specialist.predict(inputs.loc[selected])
            return prediction

    # Source: duplicate_models.py
    """Weight exact repeated training-day curves without removing any rows."""
    from collections import Counter
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from threadpoolctl import threadpool_limits
    pass

    class DuplicateModel:

        def __init__(self, config, seed=42):
            self.mode = config.get('mode', 'curve_equal')
            assert self.mode in ('curve_equal', 'curve_sqrt')

        def day_weights(self, train):
            curves = {}
            for date, day in train.groupby('날짜'):
                day = day.sort_values('시간')
                if len(day) == 24 and np.array_equal(day['시간'].to_numpy(), np.arange(24)):
                    curves[date] = day[POWER].to_numpy(dtype='<f8').tobytes()
            counts = Counter(curves.values())
            exponent = 1.0 if self.mode == 'curve_equal' else 0.5
            weights = {date: counts[curve] ** (-exponent) for date, curve in curves.items()}
            return train['날짜'].map(weights).fillna(1.0).to_numpy()

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0) * self.day_weights(train)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=RHuberFeatures(trend=True))
            weekend = is_weekend(inputs).to_numpy()
            self.residuals = []
            with threadpool_limits(limits=2):
                self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
                residual = train[POWER].to_numpy() - self.base.predict(inputs)
                for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                    correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                    correction.fit(inputs.loc[weekend], residual[weekend])
                    self.residuals.append(correction)
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            prediction = self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            if weekend.any():
                left, right = (model.predict(inputs.loc[weekend]) for model in self.residuals)
                prediction[weekend] += 0.75 * left + 0.25 * right
            return prediction

    # Source: thermal_models.py
    """H18: fixed 20-degree cooling interactions with operating state/intensity."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.pipeline import Pipeline, make_pipeline
    from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
    from sklearn.svm import SVR
    from threadpoolctl import threadpool_limits
    pass

    class ThermalFeatures(RHuberFeatures):

        def __init__(self, mode='state_hinge'):
            super().__init__(trend=True)
            self.mode = mode

        def transform(self, frame):
            features = super().transform(frame)
            log_prod = np.log1p(frame['생산량'])
            on = log_prod.gt(0).where(log_prod.notna()).astype(float)
            cooling = (frame['기온'] - 20).clip(lower=0)
            if self.mode in ('state_hinge', 'both'):
                features['cooling_on'] = cooling * on
                features['cooling_off'] = cooling * (1 - on)
            if self.mode in ('intensity', 'both'):
                features['temperature_intensity'] = frame['기온'] * log_prod
                features['cooling_intensity'] = cooling * log_prod
            if self.mode == 'daytime_hinge':
                angle = 2 * np.pi * frame['시간'] / 24
                features['cooling_on'] = cooling * on
                features['cooling_on_sin'] = cooling * on * np.sin(angle)
                features['cooling_on_cos'] = cooling * on * np.cos(angle)
            return features

    class ThermalModel:

        def __init__(self, config, seed=42):
            self.mode = config.get('mode', 'state_hinge')
            assert self.mode in ('state_hinge', 'intensity', 'daytime_hinge', 'both')

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days / 90)
            weight = (weight / weight.mean()).to_numpy()
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=ThermalFeatures(self.mode))
            weekend = date.dt.dayofweek.ge(5).to_numpy()
            self.residuals = []
            with threadpool_limits(limits=2):
                self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
                residual = train[POWER].to_numpy() - self.base.predict(inputs)
                for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                    model = Pipeline([('features', FunctionTransformer(n207_residual_features)), ('preprocess', ColumnTransformer([('category', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['시간', 'weekday']), ('numeric', make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler()), ['기온', 'log_prod'])], transformer_weights={'numeric': numeric_weight})), ('svr', MultiOutputRegressor(SVR(kernel=kernel, degree=2, coef0=1, C=10, epsilon=1, gamma='scale', tol=0.0001)))])
                    self.residuals.append(model.fit(inputs.loc[weekend], residual[weekend]))
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            prediction = self.base.predict(inputs)
            weekend = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d').dt.dayofweek.ge(5).to_numpy()
            if weekend.any():
                left, right = (model.predict(inputs.loc[weekend]) for model in self.residuals)
                prediction[weekend] += 0.75 * left + 0.25 * right
            return prediction

    # Source: state_interaction_models.py
    """H23: retain pooled training while adding state-dependent calendar responses."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.svm import SVR
    pass

    class StateInteractionFeatures(RHuberFeatures):

        def __init__(self, signal='on', grouping='weekday'):
            super().__init__(trend=True)
            self.signal, self.grouping = (signal, grouping)

        def transform(self, frame):
            features = super().transform(frame)
            weekday = pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d').dt.dayofweek
            groups = weekday if self.grouping == 'weekday' else weekday.ge(5).astype(int)
            logprod = np.log1p(frame['생산량'])
            signal = logprod if self.signal == 'logprod' else logprod.gt(0).where(logprod.notna()).astype(float)
            interactions = {f'state_hour_{hour}_group_{group}': signal * frame['시간'].eq(hour) * groups.eq(group) for hour in range(24) for group in range(7 if self.grouping == 'weekday' else 2)}
            return pd.concat([features, pd.DataFrame(interactions, index=frame.index)], axis=1)

    class StateInteractionModel:

        def __init__(self, config, seed=42):
            self.signal, self.grouping = (config.get('signal', 'on'), config.get('grouping', 'weekday'))
            assert self.signal in {'on', 'logprod'}
            assert self.grouping in {'weekday', 'weekend'}

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=StateInteractionFeatures(self.signal, self.grouping))
            self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            self.corrections = []
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                model = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr=MultiOutputRegressor(SVR(kernel=kernel, degree=2, coef0=1, C=10, epsilon=1, gamma='scale', tol=0.0001)))
                model.fit(inputs.loc[weekend], residual[weekend])
                self.corrections.append(model)
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            prediction = self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            if weekend.any():
                prediction[weekend] += 0.75 * self.corrections[0].predict(inputs.loc[weekend]) + 0.25 * self.corrections[1].predict(inputs.loc[weekend])
            return prediction

    # Source: thermal_state_combo_models.py
    """H27: original V054 intensity features plus original V070 state features."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.svm import SVR
    pass
    pass

    class ThermalStateComboFeatures(StateInteractionFeatures):

        def __init__(self):
            super().__init__(signal='on', grouping='weekend')

        def transform(self, frame):
            features = super().transform(frame)
            log_prod = np.log1p(frame['생산량'])
            features['temperature_intensity'] = frame['기온'] * log_prod
            features['cooling_intensity'] = (frame['기온'] - 20).clip(lower=0) * log_prod
            return features

    class ThermalStateComboModel(StateInteractionModel):

        def __init__(self, config, seed=42):
            assert config.get('mode', 'thermal_state') == 'thermal_state'
            super().__init__({'signal': 'on', 'grouping': 'weekend'}, seed)

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=ThermalStateComboFeatures())
            self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            self.corrections = []
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                model = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr=MultiOutputRegressor(SVR(kernel=kernel, degree=2, coef0=1, C=10, epsilon=1, gamma='scale', tol=0.0001)))
                model.fit(inputs.loc[weekend], residual[weekend])
                self.corrections.append(model)
            return self

    # Source: production_efficiency_models.py
    """H30: raw throughput response on V081, with train-only normalization."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.svm import SVR
    pass
    pass

    class ProductionEfficiencyFeatures(ThermalStateComboFeatures):

        def __init__(self, scale='global', replacement=False):
            super().__init__()
            self.scale, self.replacement = (scale, replacement)

        def fit(self, frame, y=None):
            super().fit(frame, y)
            positive = frame.loc[frame['생산량'].gt(0)].copy()
            positive['weekday'] = pd.to_datetime(positive['날짜'].astype(str), format='%Y%m%d').dt.dayofweek
            self.global_median = positive['생산량'].median()
            self.hour_medians = positive.groupby('시간')['생산량'].median()
            self.cell_medians = positive.groupby(['시간', 'weekday'])['생산량'].median()
            return self

        def transform(self, frame):
            features = super().transform(frame)
            denominator = np.full(len(frame), self.global_median)
            if self.scale == 'calendar':
                weekday = pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d').dt.dayofweek
                keys = pd.MultiIndex.from_arrays([frame['시간'], weekday], names=['시간', 'weekday'])
                denominator = self.cell_medians.reindex(keys).to_numpy()
                denominator = np.where(np.isnan(denominator), frame['시간'].map(self.hour_medians), denominator)
                denominator = np.where(np.isnan(denominator), self.global_median, denominator)
            if self.replacement:
                remove = [name for name in features if name.startswith('prod_') or name in {'log_prod', 'temperature_intensity', 'cooling_intensity'}]
                features = features.drop(columns=remove)
            throughput = frame['생산량'] / denominator
            features['throughput'] = throughput
            features['throughput_temperature'] = throughput * frame['기온']
            features['throughput_cooling'] = throughput * (frame['기온'] - 20).clip(lower=0)
            return features

    class ProductionEfficiencyModel:

        def __init__(self, config, seed=42):
            self.scale, self.mode = (config.get('scale', 'global'), config.get('mode', 'add'))
            assert self.scale in {'global', 'calendar'}
            assert self.mode in {'add', 'replace_log'}

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=ProductionEfficiencyFeatures(self.scale, self.mode == 'replace_log'))
            self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            self.corrections = []
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                model = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr=MultiOutputRegressor(SVR(kernel=kernel, degree=2, coef0=1, C=10, epsilon=1, gamma='scale', tol=0.0001)))
                self.corrections.append(model.fit(inputs.loc[weekend], residual[weekend]))
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            prediction = self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            if weekend.any():
                prediction[weekend] += 0.75 * self.corrections[0].predict(inputs.loc[weekend]) + 0.25 * self.corrections[1].predict(inputs.loc[weekend])
            return prediction

    # Source: combo_models.py
    """Combine original thermal, duplicate-weight, and zero-specialist settings."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from threadpoolctl import threadpool_limits
    pass
    pass
    pass

    class ComboModel(DuplicateModel):

        def __init__(self, config, seed=42):
            self.combination = config.get('mode', 'thermal_duplicate')
            assert self.combination in ('thermal_duplicate', 'thermal_zero', 'thermal_duplicate_zero')
            super().__init__({'mode': 'curve_sqrt'}, seed)

        def feature_transformer(self):
            return ThermalFeatures('intensity')

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            duplicate_weight = 1.0 if self.combination == 'thermal_zero' else self.day_weights(train)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0) * duplicate_weight
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=self.feature_transformer())
            weekend = is_weekend(inputs).to_numpy()
            self.residuals = []
            with threadpool_limits(limits=2):
                self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
                residual = train[POWER].to_numpy() - self.base.predict(inputs)
                for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                    correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                    correction.fit(inputs.loc[weekend], residual[weekend])
                    self.residuals.append(correction)
                self.specialist = None
                if self.combination.endswith('_zero'):
                    region = train.loc[train['생산량'].eq(0)]
                    dates = pd.to_datetime(region['날짜'].astype(str), format='%Y%m%d')
                    specialist_weight = np.exp2(-(dates.max() - dates).dt.days.to_numpy() / 90)
                    specialist_weight /= specialist_weight.mean()
                    self.specialist = clone(m242_pipeline).set_params(features=RHuberFeatures(trend=True))
                    self.specialist.fit(region[['날짜', '시간', '생산량', '기온']], region[POWER], huber__sample_weight=specialist_weight)
            return self

        def predict(self, frame):
            prediction = super().predict(frame)
            if self.specialist is not None:
                zero = frame['생산량'].eq(0).to_numpy()
                if zero.any():
                    prediction[zero] = self.specialist.predict(frame.loc[zero, ['날짜', '시간', '생산량', '기온']])
            return prediction

    # Source: thermal_balance_models.py
    """Balance sparse thermal support with one fixed inverse-square-root rule."""
    import numpy as np
    pass
    pass

    class ThermalBalanceModel(ComboModel):

        def __init__(self, config, seed=42):
            super().__init__({'mode': 'thermal_duplicate'}, seed)
            self.grouping = config.get('grouping', 'temperature')
            assert self.grouping in ('temperature', 'state_temperature')

        def feature_transformer(self):
            return ThermalStateComboFeatures()

        def day_weights(self, train):
            group = train['기온'].floordiv(5).astype('Int64').astype(str)
            if self.grouping == 'state_temperature':
                group = train['생산량'].eq(0).astype(str) + '_' + group
            self.thermal_counts_ = group.value_counts().to_dict()
            weight = 1.0 / np.sqrt(group.map(self.thermal_counts_))
            return (weight / weight.mean()).to_numpy()

    # Source: neighbor_models.py
    """H20: fixed-size local conditional medians of training targets or residuals."""
    import numpy as np
    from sklearn.base import clone
    from sklearn.neighbors import NearestNeighbors
    pass
    pass

    class NeighborModel:

        def __init__(self, config, seed=42):
            self.mode = config.get('mode', 'direct')
            self.representation = config.get('representation', 'compact')
            assert self.mode in {'direct', 'residual'}
            assert self.representation in {'compact', 'engineered'}

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            self.targets = train[POWER].to_numpy(copy=True)
            self.base = None
            if self.mode == 'residual':
                self.base = ThermalStateComboModel({'mode': 'thermal_state'}).fit(train)
                self.targets = self.targets - self.base.predict(inputs)
            template = n207_residual_pipeline if self.representation == 'compact' else m242_pipeline
            self.preprocess = clone(template[:-1])
            features = self.preprocess.fit_transform(inputs)
            self.neighbors = NearestNeighbors(n_neighbors=31, metric='manhattan', algorithm='brute', n_jobs=1)
            self.neighbors.fit(features)
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            features = self.preprocess.transform(inputs)
            indices = np.concatenate([self.neighbors.kneighbors(row[None, :], return_distance=False) for row in features])
            prediction = np.median(self.targets[indices], axis=1)
            if self.base is not None:
                prediction += self.base.predict(inputs)
            return prediction

    # Source: poisson_models.py
    """H58: one fixed log-link Poisson-deviance alternative on V081 inputs."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.linear_model import PoissonRegressor
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.svm import SVR
    pass
    pass

    class PoissonModel:

        def __init__(self, config, seed=42):
            assert config.get('mode', 'poisson') == 'poisson'

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            target = train[POWER].to_numpy()
            assert np.isfinite(target).all() and (target >= 0).all()
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=ThermalStateComboFeatures(), huber=MultiOutputRegressor(PoissonRegressor(alpha=1.0, max_iter=1000, tol=1e-07)))
            self.base.fit(inputs, target, huber__sample_weight=weight)
            residual = target - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            self.corrections = []
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                model = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr=MultiOutputRegressor(SVR(kernel=kernel, degree=2, coef0=1, C=10, epsilon=1, gamma='scale', tol=0.0001)))
                self.corrections.append(model.fit(inputs.loc[weekend], residual[weekend]))
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            prediction = self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            if weekend.any():
                prediction[weekend] += 0.75 * self.corrections[0].predict(inputs.loc[weekend]) + 0.25 * self.corrections[1].predict(inputs.loc[weekend])
            return prediction

    # Source: radial_huber_models.py
    """H57: direct four-output Huber with one robust influence per observed hour."""
    import warnings
    import numpy as np
    import pandas as pd
    from scipy.optimize import minimize
    from sklearn.base import BaseEstimator, RegressorMixin, clone
    from sklearn.exceptions import ConvergenceWarning
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.svm import SVR
    pass
    pass
    pass

    def radial_huber_objective(parameters, matrix, target, weight):
        coefficients = parameters[:-5].reshape(matrix.shape[1], 4)
        intercept, scale = (parameters[-5:-1], parameters[-1])
        residual = (matrix @ coefficients + intercept - target) / scale
        radius = np.sqrt(np.mean(residual ** 2, axis=1))
        influence = np.minimum(1.0, 1.05 / np.maximum(radius, np.finfo(float).tiny))
        rho = np.where(radius <= 1.05, radius ** 2, 2 * 1.05 * radius - 1.05 ** 2)
        loss = 4 * (weight @ (scale * (1 + rho))) + 1.5 * np.sum(coefficients ** 2)
        score = weight[:, None] * influence[:, None] * residual
        gradient = np.r_[(2 * matrix.T @ score + 3 * coefficients).ravel(), 2 * score.sum(axis=0), 4 * (weight @ (1 + rho - 2 * radius * np.minimum(radius, 1.05)))]
        return (float(loss), gradient)

    class RadialHuberRegressor(RegressorMixin, BaseEstimator):

        def fit(self, matrix, target, sample_weight=None):
            target = np.asarray(target)
            weight = np.ones(len(target)) if sample_weight is None else np.asarray(sample_weight)
            initial = np.zeros(matrix.shape[1] * 4 + 5)
            initial[-5:-1] = np.median(target, axis=0)
            initial[-1] = max(float(np.median(np.sqrt(np.mean((target - initial[-5:-1]) ** 2, axis=1)))), 1.0)
            bounds = [(None, None)] * (len(initial) - 1) + [(np.finfo(float).eps * 10, None)]
            result = minimize(radial_huber_objective, initial, args=(matrix, target, weight), method='L-BFGS-B', jac=True, bounds=bounds, options={'maxiter': 2000, 'gtol': 1e-07, 'ftol': 1e-10})
            if result.status == 2:
                raise ValueError('Radial Huber optimization failed: ' + str(result.message))
            if not result.success:
                warnings.warn(str(result.message), ConvergenceWarning)
            self.coef_ = result.x[:-5].reshape(matrix.shape[1], 4)
            self.intercept_, self.scale_ = (result.x[-5:-1], result.x[-1])
            self.training_summary_ = dict(success=bool(result.success), iterations=int(result.nit), message=str(result.message), objective=float(result.fun), scale=float(self.scale_))
            return self

        def predict(self, matrix):
            return matrix @ self.coef_ + self.intercept_

    class RadialHuberModel(StateInteractionModel):

        def __init__(self, config, seed=42):
            assert config.get('loss', 'radial') == 'radial'

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=ThermalStateComboFeatures(), huber=RadialHuberRegressor())
            self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            self.training_summary_ = self.base.named_steps['huber'].training_summary_
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            self.corrections = []
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr=MultiOutputRegressor(SVR(kernel=kernel, degree=2, coef0=1, C=10, epsilon=1, gamma='scale', tol=0.0001)))
                correction.fit(inputs.loc[weekend], residual[weekend])
                self.corrections.append(correction)
            return self

    # Source: tree_direct_models.py
    """H66: fixed direct LightGBM on V081 features and its equal-weight blend."""
    from importlib.metadata import version
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.multioutput import MultiOutputRegressor
    from threadpoolctl import threadpool_limits
    pass
    pass

    class TreeDirectModel:

        def __init__(self, config, seed=42):
            self.mode = config.get('mode', 'standalone')
            self.seed = seed
            assert self.mode in ('standalone', 'blend')

        def fit(self, train):
            from lightgbm import LGBMRegressor
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0)
            weight /= weight.mean()
            self.preprocess = clone(m242_pipeline[:-1]).set_params(features=ThermalStateComboFeatures())
            estimator = LGBMRegressor(objective='regression_l1', n_estimators=300, max_depth=5, num_leaves=15, min_child_samples=40, learning_rate=0.03, reg_lambda=10.0, random_state=self.seed, n_jobs=2, device_type='cpu', force_col_wise=True, verbosity=-1)
            with threadpool_limits(limits=2):
                matrix = self.preprocess.fit_transform(inputs)
                self.tree = MultiOutputRegressor(estimator, n_jobs=1)
                self.tree.fit(matrix, train[POWER], sample_weight=weight)
                self.base = ThermalStateComboModel({'mode': 'thermal_state'}, self.seed).fit(train) if self.mode == 'blend' else None
            self.training_summary_ = dict(lightgbm_version=version('lightgbm'), seed=self.seed, mode=self.mode, parameters=estimator.get_params())
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            prediction = self.tree.predict(self.preprocess.transform(inputs))
            return prediction if self.base is None else 0.5 * prediction + 0.5 * self.base.predict(inputs)

    # Source: phase2_quality_combo_models.py
    """Preserve V096 thermal support weights and V028 duplicate-curve weights."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.preprocessing import RobustScaler
    from threadpoolctl import threadpool_limits
    pass
    pass
    pass
    pass
    pass
    pass
    pass

    class Phase2QualityComboModel(ThermalBalanceModel):

        def __init__(self, config, seed=42):
            self.quality_mode = config.get('mode', 'state_temperature_duplicate')
            assert self.quality_mode in ('state_temperature_duplicate', 'raw_throughput', 'radial', 'features_r191', 'features_v054', 'features_v070', 'radial_v070', 'radial_v070_recent3', 'radial_v070_recent3_iqr')
            super().__init__({'grouping': 'state_temperature'}, seed)

        def feature_transformer(self):
            if self.quality_mode == 'raw_throughput':
                return ProductionEfficiencyFeatures(scale='global', replacement=False)
            if self.quality_mode == 'features_r191':
                return RHuberFeatures(trend=True)
            if self.quality_mode == 'features_v054':
                return ThermalFeatures(mode='intensity')
            if self.quality_mode in ('features_v070', 'radial_v070', 'radial_v070_recent3', 'radial_v070_recent3_iqr'):
                return StateInteractionFeatures(signal='on', grouping='weekend')
            return super().feature_transformer()

        def day_weights(self, train):
            return ThermalBalanceModel.day_weights(self, train) * DuplicateModel.day_weights(self, train)

        def fit(self, train):
            if self.quality_mode not in ('radial', 'radial_v070', 'radial_v070_recent3', 'radial_v070_recent3_iqr'):
                return super().fit(train)
            if self.quality_mode in ('radial_v070_recent3', 'radial_v070_recent3_iqr'):
                dates = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
                start = (dates.max().to_period('M') - 2).start_time
                train = train.loc[dates >= start]
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0) * self.day_weights(train)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=self.feature_transformer(), huber=RadialHuberRegressor())
            if self.quality_mode == 'radial_v070_recent3_iqr':
                self.base.set_params(preprocess__remainder__standardscaler=RobustScaler(quantile_range=(25.0, 75.0)))
            weekend = is_weekend(inputs).to_numpy()
            self.residuals, self.specialist = ([], None)
            with threadpool_limits(limits=2):
                self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
                residual = train[POWER].to_numpy() - self.base.predict(inputs)
                for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                    correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                    correction.fit(inputs.loc[weekend], residual[weekend])
                    self.residuals.append(correction)
            self.training_summary_ = self.base.named_steps['huber'].training_summary_
            return self

    # Source: phase2_window_models.py
    """V081 members with calendar training windows or genuinely uniform weights."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    pass
    pass
    pass

    class Phase2WindowModel(ThermalStateComboModel):

        def __init__(self, config, seed=42):
            super().__init__({'mode': 'thermal_state'}, seed)
            self.window = config['window']
            self.base_name, self.seed = (config.get('base', 'V081'), seed)
            assert self.base_name in ('V081', 'V217', 'V271', 'V239')
            assert self.window in ('recent1', 'recent2', 'recent3', 'unweighted', 'no_age')
            assert self.base_name != 'V217' or self.window in ('recent3', 'no_age')
            assert self.base_name not in ('V271', 'V239') or self.window == 'recent3'
            assert self.window != 'no_age' or self.base_name == 'V217'

        def fit(self, train):
            if self.window.startswith('recent'):
                months = int(self.window[-1])
                dates = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
                start = (dates.max().to_period('M') - (months - 1)).start_time
                train = train.loc[dates >= start]
                if self.base_name in ('V217', 'V271', 'V239'):
                    mode = {'V217': 'state_temperature_duplicate', 'V271': 'features_v070', 'V239': 'raw_throughput'}[self.base_name]
                    model = Phase2QualityComboModel({'mode': mode}, self.seed).fit(train)
                    self.base, self.corrections = (model.base, model.residuals)
                else:
                    super().fit(train)
            else:
                inputs = train[['날짜', '시간', '생산량', '기온']]
                weight = np.ones(len(train))
                if self.window == 'no_age':
                    quality = Phase2QualityComboModel({'mode': 'state_temperature_duplicate'}, self.seed)
                    weight = np.where(inputs['생산량'].eq(0), 0.5, 1.0) * quality.day_weights(train)
                    weight /= weight.mean()
                self.base = clone(m242_pipeline).set_params(features=ThermalStateComboFeatures())
                self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
                residual = train[POWER].to_numpy() - self.base.predict(inputs)
                weekend = is_weekend(inputs).to_numpy()
                self.corrections = []
                for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                    correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                    self.corrections.append(correction.fit(inputs.loc[weekend], residual[weekend]))
            row_weight = 'ones' if self.window == 'unweighted' else 'V217_thermal_duplicate_zero0.5_no_age' if self.window == 'no_age' else self.base_name + '_age90_zero0.5'
            self.training_summary_ = dict(window=self.window, base=self.base_name, rows=len(train), train_start=int(train['날짜'].min()), train_end=int(train['날짜'].max()), row_weight=row_weight)
            return self

    # Source: phase2_representation_models.py
    """V054/V070 Huber representations with zero or full RBF weekend correction."""
    pass
    pass
    pass

    class Phase2RepresentationModel:

        def __init__(self, config, seed=42):
            self.representation = config['representation']
            self.weekend_correction = config['weekend_correction']
            self.seed = seed
            assert self.representation in ('V054', 'V070')
            assert self.weekend_correction in ('none', 'rbf')

        def fit(self, train):
            original = ThermalModel({'mode': 'intensity'}, self.seed) if self.representation == 'V054' else StateInteractionModel({'signal': 'on', 'grouping': 'weekend'}, self.seed)
            original.fit(train)
            self.base = original.base
            corrections = original.residuals if self.representation == 'V054' else original.corrections
            self.correction = corrections[0] if self.weekend_correction == 'rbf' else None
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            prediction = self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            if self.correction is not None and weekend.any():
                prediction[weekend] += self.correction.predict(inputs.loc[weekend])
            return prediction

    # Source: phase2_loss_models.py
    """Conditional-median loss on V081 features with V081 or V217 weights."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.linear_model import QuantileRegressor
    from sklearn.multioutput import MultiOutputRegressor
    pass
    pass

    class Phase2MedianModel(Phase2QualityComboModel):

        def __init__(self, config, seed=42):
            self.loss_base = config.get('base', 'V081')
            assert self.loss_base in ('V081', 'V217')
            super().__init__({'mode': 'state_temperature_duplicate'}, seed)

        def day_weights(self, train):
            return np.ones(len(train)) if self.loss_base == 'V081' else super().day_weights(train)

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            dates = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(dates.max() - dates).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0) * self.day_weights(train)
            weight /= weight.mean()
            alpha = 1.5 / len(train)
            estimator = MultiOutputRegressor(QuantileRegressor(quantile=0.5, alpha=alpha, solver='highs'), n_jobs=1)
            self.base = clone(m242_pipeline).set_params(features=self.feature_transformer(), huber=estimator)
            self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            self.residuals, self.specialist = ([], None)
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                correction.fit(inputs.loc[weekend], residual[weekend])
                self.residuals.append(correction)
            self.training_summary_ = dict(base=self.loss_base, loss='median_pinball', alpha=alpha, summed_loss_l1_penalty=1.5, training_rows=len(train), weekend_correction='original 0.75 RBF + 0.25 polynomial SVR')
            return self

    # Source: phase2_scaling_models.py
    """Fixed IQR scaling of numeric V081 features; original Huber and SVR settings."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.preprocessing import RobustScaler
    pass
    pass

    class Phase2ScalingModel(Phase2QualityComboModel):

        def __init__(self, config, seed=42):
            self.scaling_base = config.get('base', 'V081')
            assert self.scaling_base in ('V081', 'V217', 'V239', 'V258', 'V271', 'V299', 'V321')
            mode = 'raw_throughput' if self.scaling_base in ('V239', 'V321') else 'features_v070' if self.scaling_base in ('V271', 'V299') else 'state_temperature_duplicate'
            super().__init__({'mode': mode}, seed)

        def day_weights(self, train):
            return np.ones(len(train)) if self.scaling_base == 'V081' else super().day_weights(train)

        def fit(self, train):
            if self.scaling_base in ('V258', 'V299', 'V321'):
                dates = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
                start = (dates.max().to_period('M') - 2).start_time
                train = train.loc[dates >= start]
            inputs = train[['날짜', '시간', '생산량', '기온']]
            dates = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(dates.max() - dates).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0) * self.day_weights(train)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=self.feature_transformer(), preprocess__remainder__standardscaler=RobustScaler(quantile_range=(25.0, 75.0)))
            self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            self.residuals, self.specialist = ([], None)
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                correction.fit(inputs.loc[weekend], residual[weekend])
                self.residuals.append(correction)
            self.training_summary_ = dict(base=self.scaling_base, scaler='RobustScaler', quantile_range=[25.0, 75.0], replaced_step='preprocess__remainder__standardscaler', other_preprocessing='unchanged categorical one-hot, imputer and weekend SVR pipelines')
            if self.scaling_base in ('V258', 'V299', 'V321'):
                self.training_summary_.update(window='recent3', rows=len(train), train_start=int(train['날짜'].min()), train_end=int(train['날짜'].max()))
            return self

    # Source: phase2_weighted_scaling_models.py
    """Use the Huber training weights for numeric feature means and variances too."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.preprocessing import StandardScaler
    pass
    pass
    pass

    class FoldWeightedStandardScaler(StandardScaler):

        def __init__(self, training_weight):
            self.training_weight = training_weight
            super().__init__()

        def fit(self, matrix, y=None, sample_weight=None):
            assert len(matrix) == len(self.training_weight)
            return super().fit(matrix, y, sample_weight=self.training_weight)

    class Phase2WeightedScalingModel(Phase2QualityComboModel):

        def __init__(self, config, seed=42):
            self.scaling_base = config['base']
            self.loss = config.get('loss', 'independent')
            assert self.scaling_base in ('V217', 'V271', 'V258', 'V299')
            assert self.loss in ('independent', 'radial')
            mode = 'features_v070' if self.scaling_base in ('V271', 'V299') else 'state_temperature_duplicate'
            super().__init__({'mode': mode}, seed)

        def fit(self, train):
            if self.scaling_base in ('V258', 'V299'):
                dates = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
                start = (dates.max().to_period('M') - 2).start_time
                train = train.loc[dates >= start]
            inputs = train[['날짜', '시간', '생산량', '기온']]
            dates = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(dates.max() - dates).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0) * self.day_weights(train)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=self.feature_transformer(), preprocess__remainder__standardscaler=FoldWeightedStandardScaler(weight))
            if self.loss == 'radial':
                self.base.set_params(huber=RadialHuberRegressor())
            self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            self.residuals, self.specialist = ([], None)
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                correction.fit(inputs.loc[weekend], residual[weekend])
                self.residuals.append(correction)
            self.training_summary_ = dict(base=self.scaling_base, numeric_scaler='StandardScaler fitted with exact Huber sample weights', categorical_and_imputation='unchanged and unweighted', weekend_correction='original unweighted 0.75 RBF + 0.25 polynomial SVR', training_rows=len(train), train_start=int(train['날짜'].min()), train_end=int(train['날짜'].max()))
            if self.loss == 'radial':
                self.training_summary_.update(loss='original radial Huber', solver=self.base.named_steps['huber'].training_summary_)
            return self

    # Source: phase2_strong_representation_models.py
    """Exact strong bases with no weekend SVR or the existing RBF SVR alone."""
    pass
    pass
    pass
    pass
    pass
    pass

    class Phase2StrongRepresentationModel(Phase2RepresentationModel):

        def __init__(self, config, seed=42):
            self.representation = config['representation']
            self.weekend_correction = config['weekend_correction']
            self.seed = seed
            assert self.representation in ('V081', 'V217', 'V239', 'V258', 'V266', 'V271', 'V299', 'V303', 'V317', 'V321', 'V351', 'V353')
            assert self.weekend_correction in ('none', 'rbf')
            assert self.representation in ('V081', 'V217') or self.weekend_correction == 'rbf'

        def fit(self, train):
            if self.representation == 'V081':
                original = ThermalStateComboModel({'mode': 'thermal_state'}, self.seed)
            elif self.representation in ('V258', 'V299', 'V321'):
                base = {'V258': 'V217', 'V299': 'V271', 'V321': 'V239'}[self.representation]
                original = Phase2WindowModel({'base': base, 'window': 'recent3'}, self.seed)
            elif self.representation in ('V266', 'V303', 'V317'):
                base = {'V266': 'V217', 'V303': 'V271', 'V317': 'V299'}[self.representation]
                original = Phase2ScalingModel({'base': base}, self.seed)
            elif self.representation in ('V351', 'V353'):
                base = 'V299' if self.representation == 'V353' else 'V271'
                original = Phase2WeightedScalingModel({'base': base}, self.seed)
            else:
                mode = 'raw_throughput' if self.representation == 'V239' else 'features_v070' if self.representation == 'V271' else 'state_temperature_duplicate'
                original = Phase2QualityComboModel({'mode': mode}, self.seed)
            original.fit(train)
            self.base = original.base
            corrections = original.corrections if self.representation in ('V081', 'V258', 'V299', 'V321') else original.residuals
            self.correction = corrections[0] if self.weekend_correction == 'rbf' else None
            self.training_summary_ = dict(representation=self.representation, weekend_correction=self.weekend_correction, base_fit='unchanged original features, Huber and sample weights')
            return self

    # Source: phase2_weekpart_models.py
    """Independent weekday/weekend Huber responses on V081 or V217 training weights."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    pass
    pass

    class Phase2WeekpartModel(Phase2QualityComboModel):

        def __init__(self, config, seed=42):
            self.weight_base = config.get('base', 'V081')
            assert self.weight_base in ('V081', 'V217')
            super().__init__({'mode': 'state_temperature_duplicate'}, seed)

        def day_weights(self, train):
            return np.ones(len(train)) if self.weight_base == 'V081' else super().day_weights(train)

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            dates = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(dates.max() - dates).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0) * self.day_weights(train)
            weight /= weight.mean()
            weekend = is_weekend(inputs).to_numpy()
            self.hubers = []
            for part in [False, True]:
                selected = weekend == part
                model = clone(m242_pipeline).set_params(features=self.feature_transformer())
                model.fit(inputs.loc[selected], train.loc[selected, POWER], huber__sample_weight=weight[selected])
                self.hubers.append(model)
            residual = train.loc[weekend, POWER].to_numpy() - self.hubers[1].predict(inputs.loc[weekend])
            self.residuals = []
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                correction.fit(inputs.loc[weekend], residual)
                self.residuals.append(correction)
            self.training_summary_ = dict(base=self.weight_base, weekday_rows=int((~weekend).sum()), weekend_rows=int(weekend.sum()), weight_scope='original full-fold weights sliced unchanged into weekday/weekend', weekend_correction='original 0.75 RBF + 0.25 polynomial SVR')
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            weekend = is_weekend(inputs).to_numpy()
            prediction = np.empty((len(inputs), 4))
            for part, model in enumerate(self.hubers):
                selected = weekend == bool(part)
                if selected.any():
                    prediction[selected] = model.predict(inputs.loc[selected])
            if weekend.any():
                left, right = (model.predict(inputs.loc[weekend]) for model in self.residuals)
                prediction[weekend] += 0.75 * left + 0.25 * right
            return prediction

    # Source: phase2_weighted_weekend_models.py
    """Apply V217 training-weight factors to its existing weekend SVRs."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    pass
    pass

    class Phase2WeightedWeekendModel(Phase2QualityComboModel):

        def __init__(self, config, seed=42):
            self.weekend_weight = config['weight_policy']
            assert self.weekend_weight in ('recency', 'quality', 'combined')
            super().__init__({'mode': 'state_temperature_duplicate'}, seed)

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            dates = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            recency = np.exp2(-(dates.max() - dates).dt.days.to_numpy() / 90)
            quality = self.day_weights(train)
            weight = recency.copy()
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0) * quality
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=self.feature_transformer())
            self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            weekend_weight = {'recency': recency, 'quality': quality, 'combined': weight}[self.weekend_weight][weekend].copy()
            weekend_weight /= weekend_weight.mean()
            self.residuals, self.specialist = ([], None)
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                correction.fit(inputs.loc[weekend], residual[weekend], svr__sample_weight=weekend_weight)
                self.residuals.append(correction)
            self.training_summary_ = dict(base='V217', weight_policy=self.weekend_weight, weight_normalization='mean one within training weekends', quality_support='all fold-training rows', weekend_rows=int(weekend.sum()), correction='original 0.75 RBF + 0.25 polynomial SVR')
            return self

    # Source: phase2_bootstrap_models.py
    """Median of five V217 fits with Poisson day/week weights on Huber training."""
    import numpy as np
    import pandas as pd
    pass

    class PoissonWeightedV217(Phase2QualityComboModel):

        def __init__(self, block, seed, member_index):
            super().__init__({'mode': 'state_temperature_duplicate'}, seed)
            self.block, self.seed, self.member_index = (block, seed, member_index)

        def day_weights(self, train):
            original = super().day_weights(train)
            dates = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
            if self.block == 'day':
                labels = train['날짜'].astype(str)
            else:
                calendar = dates.dt.isocalendar()
                labels = calendar['year'].astype(str) + '-' + calendar['week'].astype(str)
            block_ids = np.sort(labels.unique())
            rng = np.random.default_rng(np.random.SeedSequence([self.seed, self.member_index]))
            counts = rng.poisson(1, len(block_ids))
            assert counts.sum() > 0
            self.block_counts_ = dict(zip(block_ids, counts.tolist()))
            return original * labels.map(self.block_counts_).to_numpy()

    class Phase2BootstrapModel:

        def __init__(self, config, seed=42):
            self.block, self.seed = (config.get('block', 'day'), seed)
            assert self.block in ('day', 'week')

        def fit(self, train):
            self.members = [PoissonWeightedV217(self.block, self.seed, index).fit(train) for index in range(5)]
            self.training_summary_ = dict(base='V217', block=self.block, members=5, seed=self.seed, seed_rule='SeedSequence([seed, member_index])', aggregation='median', block_counts=[model.block_counts_ for model in self.members], weight_scope='Poisson counts multiply original full-fold V217 Huber weights', weekend_correction='original unweighted SVRs on each member Huber residuals')
            return self

        def predict(self, frame):
            return np.median(np.stack([model.predict(frame) for model in self.members]), axis=0)

    # Source: phase2_linear_svr_models.py
    """Fixed L2-regularized absolute-loss linear members on V271 features."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.preprocessing import RobustScaler
    from sklearn.svm import LinearSVR
    pass
    pass

    class Phase2LinearSVRModel(Phase2QualityComboModel):

        def __init__(self, config, seed=42):
            super().__init__({'mode': 'features_v070'}, seed)
            self.window = config.get('window', 'full')
            self.scaler = config.get('scaler', 'standard')
            self.seed = seed
            assert self.window in ('full', 'recent3')
            assert self.scaler in ('standard', 'iqr')

        def fit(self, train):
            if self.window == 'recent3':
                dates = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
                start = (dates.max().to_period('M') - 2).start_time
                train = train.loc[dates >= start]
            inputs = train[['날짜', '시간', '생산량', '기온']]
            dates = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(dates.max() - dates).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0) * self.day_weights(train)
            weight /= weight.mean()
            estimator = MultiOutputRegressor(LinearSVR(C=1.0, epsilon=0.0, loss='epsilon_insensitive', dual=True, tol=0.0001, max_iter=20000, random_state=self.seed), n_jobs=1)
            self.base = clone(m242_pipeline).set_params(features=self.feature_transformer(), huber=estimator)
            if self.scaler == 'iqr':
                self.base.set_params(preprocess__remainder__standardscaler=RobustScaler(quantile_range=(25.0, 75.0)))
            self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            self.residuals, self.specialist = ([], None)
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                correction.fit(inputs.loc[weekend], residual[weekend])
                self.residuals.append(correction)
            iterations = [int(model.n_iter_) for model in self.base.named_steps['huber'].estimators_]
            self.training_summary_ = dict(base='V271', window=self.window, scaler=self.scaler, C=1.0, epsilon=0.0, loss='epsilon_insensitive', penalty='L2', max_iter=20000, seed=self.seed, seed_confirmation_required=True, n_iter=iterations, reached_iteration_limit=any((count >= 20000 for count in iterations)), training_rows=len(train), train_start=int(train['날짜'].min()), train_end=int(train['날짜'].max()), weekend_correction='original 0.75 RBF + 0.25 polynomial SVR')
            return self

    # Source: phase2_linear_tree_models.py
    """Piecewise-linear Huber boosting replaces the strong member's base learner.

    LightGBM 4.6.0 permits Huber with serial linear trees and excludes regression_l1:
    https://lightgbm.readthedocs.io/en/v4.6.0/Parameters.html#linear_tree
    """
    from importlib.metadata import version
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.compose import TransformedTargetRegressor
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.preprocessing import StandardScaler
    pass
    pass

    class Phase2LinearTreeModel(Phase2QualityComboModel):

        def __init__(self, config, seed=42):
            self.base_name, self.history = (config['base'], config['history'])
            self.seed = seed
            assert self.base_name in ('V217', 'V271')
            assert self.history in ('full', 'recent3')
            mode = 'features_v070' if self.base_name == 'V271' else 'state_temperature_duplicate'
            super().__init__({'mode': mode}, seed)

        def fit(self, train):
            from lightgbm import LGBMRegressor
            if self.history == 'recent3':
                dates = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
                train = train.loc[dates >= (dates.max().to_period('M') - 2).start_time]
            inputs = train[['날짜', '시간', '생산량', '기온']]
            dates = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(dates.max() - dates).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0) * self.day_weights(train)
            weight /= weight.mean()
            parameters = dict(objective='huber', alpha=0.9, n_estimators=100, num_leaves=4, min_child_samples=100, learning_rate=0.05, linear_tree=True, linear_lambda=10.0, reg_lambda=10.0, tree_learner='serial', device_type='cpu', n_jobs=2, random_state=self.seed, deterministic=True, force_col_wise=True, zero_as_missing=False, verbosity=-1)
            estimator = TransformedTargetRegressor(regressor=MultiOutputRegressor(LGBMRegressor(**parameters), n_jobs=1), transformer=StandardScaler())
            self.base = clone(m242_pipeline).set_params(features=self.feature_transformer(), huber=estimator)
            self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            self.residuals, self.specialist = ([], None)
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                correction.fit(inputs.loc[weekend], residual[weekend])
                self.residuals.append(correction)
            target_scaler = self.base.named_steps['huber'].transformer_
            self.training_summary_ = dict(base=self.base_name, history=self.history, rows=len(train), train_start=int(train['날짜'].min()), train_end=int(train['날짜'].max()), lightgbm_version=version('lightgbm'), parameters=parameters, target_mean=target_scaler.mean_.tolist(), target_scale=target_scaler.scale_.tolist(), weekend_correction='original 0.75 RBF + 0.25 polynomial SVR')
            return self

    # Source: phase2_quality_support_models.py
    """Preserve state/calendar mass while retaining within-group quality weights."""
    import numpy as np
    import pandas as pd
    pass

    class Phase2QualitySupportModel(Phase2QualityComboModel):

        def __init__(self, config, seed=42):
            super().__init__({'mode': 'features_v070'}, seed)
            self.balance = config['balance']
            self.window = config.get('window', 'full')
            assert self.balance in ('state', 'weekpart')
            assert self.window in ('full', 'recent3')

        def day_weights(self, train):
            quality = super().day_weights(train)
            dates = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
            original = np.exp2(-(dates.max() - dates).dt.days.to_numpy() / 90)
            original *= np.where(train['생산량'].eq(0), 0.5, 1.0)
            group = train['생산량'].gt(0) if self.balance == 'state' else dates.dt.dayofweek.ge(5)
            original = pd.Series(original, index=train.index)
            weighted = original * quality
            normalizer = weighted.groupby(group).sum() / original.groupby(group).sum()
            adjusted = quality / group.map(normalizer).to_numpy()
            self.weight_audit_ = dict(balance=self.balance, group_normalizers={str(k): float(v) for k, v in normalizer.items()}, original_group_mass={str(k): float(v) for k, v in original.groupby(group).sum().items()}, adjusted_group_mass={str(k): float(v) for k, v in (original * adjusted).groupby(group).sum().items()})
            return adjusted

        def fit(self, train):
            if self.window == 'recent3':
                dates = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
                train = train.loc[dates >= (dates.max().to_period('M') - 2).start_time]
            super().fit(train)
            self.training_summary_ = dict(base='V271', window=self.window, train_start=int(train['날짜'].min()), train_end=int(train['날짜'].max()), rows=len(train), **self.weight_audit_)
            return self

    # Source: phase2_distinct_support_models.py
    """V271 thermal support counts distinct daily-curve mass, not repeat count."""
    import numpy as np
    import pandas as pd
    pass
    pass

    class Phase2DistinctSupportModel(Phase2QualityComboModel):

        def __init__(self, config, seed=42):
            super().__init__({'mode': 'features_v070'}, seed)
            self.support_grouping = config.get('grouping', 'state_temperature')
            assert self.support_grouping in ('state_temperature', 'state_temperature_weekpart', 'state_temperature_daytime', 'temperature_production')

        def day_weights(self, train):
            duplicate = DuplicateModel.day_weights(self, train)
            temperature = train['기온'].floordiv(5).astype('Int64').astype(str)
            group = train['생산량'].eq(0).astype(str) + '_' + temperature
            summary = dict(base='V271', grouping=self.support_grouping, support_mass='sum of inverse exact-curve multiplicity', per_row_duplicate_weight='original inverse square root multiplicity', temperature_width=5, support_scope='current training rows only')
            if self.support_grouping == 'state_temperature_weekpart':
                date = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
                group = group + '_' + date.dt.dayofweek.ge(5).astype(str)
            elif self.support_grouping == 'state_temperature_daytime':
                group = group + '_' + train['시간'].between(8, 18).astype(str)
                summary['daytime_hours'] = [8, 18]
            elif self.support_grouping == 'temperature_production':
                production = train['생산량']
                self.positive_median_ = float(production.loc[production > 0].median())
                tier = pd.Series(np.where(production <= 0, 0, np.where(production <= self.positive_median_, 1, 2)), index=train.index)
                group = temperature + '_' + tier.astype(str)
                summary['positive_production_median'] = self.positive_median_
            support = pd.Series(duplicate ** 2, index=train.index).groupby(group).sum()
            thermal = 1.0 / np.sqrt(group.map(support).to_numpy(dtype=float))
            thermal /= thermal.mean()
            self.training_summary_ = dict(summary, group_support=support.to_dict())
            return thermal * duplicate

    # Source: phase2_production_representation_models.py
    """Two fixed production-response assumptions on the strong V271 base."""
    import numpy as np
    import pandas as pd
    pass
    pass

    class Phase2ProductionFeatures(StateInteractionFeatures):

        def __init__(self, representation):
            super().__init__(signal='on', grouping='weekend')
            self.representation = representation

        def transform(self, frame):
            features = super().transform(frame)
            hinge_columns = [f'prod_hinge_hour{hour}' for hour in range(24)]
            if self.representation == 'no_hinge':
                return features.drop(columns=hinge_columns)
            production = frame['생산량'].to_numpy(dtype=float)
            original = np.log1p(production)
            transformed = np.arcsinh(production / 100.0)
            ratio = np.divide(transformed, original, out=np.zeros(len(frame)), where=original > 0)
            columns = ['log_prod', 'prod_sin', 'prod_cos']
            columns += [f'prod_day{day}' for day in range(7)]
            columns += [f'prod_hour{hour}' for hour in range(24)]
            features[columns] = features[columns].mul(ratio, axis=0)
            original_hinge = np.maximum(original - np.log1p(100.0), 0.0)
            transformed_hinge = np.maximum(transformed - np.arcsinh(1.0), 0.0)
            hinge_ratio = np.divide(transformed_hinge, original_hinge, out=np.zeros(len(frame)), where=original_hinge > 0)
            features[hinge_columns] = features[hinge_columns].mul(hinge_ratio, axis=0)
            return features

    class Phase2ProductionRepresentationModel(Phase2QualityComboModel):

        def __init__(self, config, seed=42):
            self.representation, self.history = (config['representation'], config['history'])
            assert self.representation in ('no_hinge', 'asinh100')
            assert self.history in ('full', 'recent3')
            super().__init__({'mode': 'features_v070'}, seed)

        def feature_transformer(self):
            return Phase2ProductionFeatures(self.representation)

        def fit(self, train):
            if self.history == 'recent3':
                dates = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
                train = train.loc[dates >= (dates.max().to_period('M') - 2).start_time]
            super().fit(train)
            self.training_summary_ = dict(base='V271', representation=self.representation, history=self.history, rows=len(train), train_start=int(train['날짜'].min()), train_end=int(train['날짜'].max()), production_reference=100.0, other_settings='original V271 quality weights, Huber and weekend SVRs')
            return self

    # Source: phase2_clock_pool_models.py
    """Pool response gates across clock hours without changing calendar baseload."""
    import numpy as np
    import pandas as pd
    pass
    pass

    class ClockPoolFeatures(StateInteractionFeatures):

        def __init__(self, blocks='state'):
            super().__init__(signal='on', grouping='weekend')
            self.blocks = blocks

        def transform(self, frame):
            features = super().transform(frame)
            hour = frame['시간'].to_numpy()
            responses = {}
            if self.blocks in ('state', 'state_hinge'):
                features = features.drop(columns=[name for name in features if name.startswith('state_hour_')])
                weekday = pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d').dt.dayofweek
                for group in (0, 1):
                    responses[f'clock_state_{group}'] = features['on'] * weekday.ge(5).eq(bool(group))
            if self.blocks in ('hinge', 'state_hinge'):
                features = features.drop(columns=[name for name in features if name.startswith('prod_hinge_hour')])
                responses['clock_hinge'] = (np.log1p(frame['생산량']) - np.log1p(100)).clip(lower=0)
            for name, response in responses.items():
                features[name] = response
                for harmonic in (1, 2):
                    angle = 2 * np.pi * harmonic * hour / 24
                    features[f'{name}_sin{harmonic}'] = response * np.sin(angle)
                    features[f'{name}_cos{harmonic}'] = response * np.cos(angle)
            return features

    class Phase2ClockPoolModel(Phase2QualityComboModel):

        def __init__(self, config, seed=42):
            super().__init__({'mode': 'features_v070'}, seed)
            self.clock_blocks = config.get('blocks', 'state')
            assert self.clock_blocks in ('state', 'hinge', 'state_hinge')
            self.training_summary_ = dict(base='V271', pooled_blocks=self.clock_blocks, clock_harmonics=[1, 2], main_hour_weekday_category='unchanged', quality_weights='original V271', weekend_correction='original V271')

        def feature_transformer(self):
            return ClockPoolFeatures(self.clock_blocks)

    # Source: phase2_geometry_combo_models.py
    """Apply V353 weighted feature geometry to four preserved recent-three designs."""
    pass
    pass
    pass
    pass

    class Phase2GeometryComboModel(Phase2WeightedScalingModel):

        def __init__(self, config, seed=42):
            super().__init__({'base': 'V299', 'loss': config.get('loss', 'independent')}, seed)
            self.variant = config['variant']
            self.weekend_correction = config.get('weekend_correction', 'original')
            self.seed = seed
            assert self.variant in ('no_hinge', 'asinh100', 'state_neutral', 'weekpart_neutral')
            assert self.weekend_correction in ('original', 'rbf')

        def feature_transformer(self):
            if self.variant in ('no_hinge', 'asinh100'):
                return Phase2ProductionFeatures(self.variant)
            return super().feature_transformer()

        def day_weights(self, train):
            if self.variant.endswith('_neutral'):
                model = Phase2QualitySupportModel({'balance': self.variant.removesuffix('_neutral'), 'window': 'recent3'}, self.seed)
                weights = model.day_weights(train)
                self.weight_audit_ = model.weight_audit_
                return weights
            return super().day_weights(train)

        def fit(self, train):
            super().fit(train)
            self.training_summary_['structural_combination'] = self.variant
            if self.variant.endswith('_neutral'):
                self.training_summary_['quality_normalization'] = self.weight_audit_
            if self.weekend_correction == 'rbf':
                self.training_summary_['weekend_correction'] = 'original RBF SVR alone'
            return self

        def predict(self, frame):
            if self.weekend_correction == 'original':
                return super().predict(frame)
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            prediction = self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            if weekend.any():
                prediction[weekend] += self.residuals[0].predict(inputs.loc[weekend])
            return prediction

    # Source: phase2_categorical_models.py
    """Symmetric calendar regularization on the fixed strong V353 pipeline."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import OneHotEncoder
    pass
    pass
    pass

    class Phase2CategoricalModel(Phase2QualityComboModel):

        def __init__(self, config, seed=42):
            super().__init__({'mode': 'features_v070'}, seed)
            self.coding = config.get('coding', 'symmetric')
            assert self.coding in ('symmetric', 'weighted_variance')

        def fit(self, train):
            dates = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
            start = (dates.max().to_period('M') - 2).start_time
            train = train.loc[dates >= start]
            inputs = train[['날짜', '시간', '생산량', '기온']]
            dates = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(dates.max() - dates).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0) * self.day_weights(train)
            weight /= weight.mean()
            category = OneHotEncoder(drop=None, handle_unknown='ignore', sparse_output=False)
            if self.coding == 'weighted_variance':
                category = make_pipeline(category, FoldWeightedStandardScaler(weight))
            self.base = clone(m242_pipeline).set_params(features=self.feature_transformer(), preprocess__category=category, preprocess__remainder__standardscaler=FoldWeightedStandardScaler(weight))
            self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            self.residuals, self.specialist = ([], None)
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                correction.fit(inputs.loc[weekend], residual[weekend])
                self.residuals.append(correction)
            self.training_summary_ = dict(base='V353', coding=self.coding, window='recent3', category='all hour-weekday indicators; no privileged omitted category', category_scaling='weighted StandardScaler' if self.coding == 'weighted_variance' else 'none', numeric_scaling='original V353 weighted StandardScaler', training_rows=len(train), train_start=int(train['날짜'].min()), train_end=int(train['날짜'].max()), weekend_correction='original 0.75 RBF + 0.25 polynomial SVR')
            return self

    # Source: phase2_weekend_scaling_models.py
    """Change only the numeric geometry inside the existing two weekend SVRs."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.preprocessing import RobustScaler
    pass
    pass
    pass

    class Phase2WeekendScalingModel(Phase2WeightedScalingModel):

        def __init__(self, config, seed=42):
            self.member, self.weekend_scaler = (config['base'], config['weekend_scaler'])
            self.seed = seed
            assert self.member in ('V351', 'V353', 'V365', 'V366', 'V367', 'V368')
            assert self.weekend_scaler in ('weighted', 'iqr')
            assert self.member in ('V351', 'V353') or self.weekend_scaler == 'iqr'
            super().__init__({'base': 'V271' if self.member == 'V351' else 'V299'}, seed)

        def fit(self, train):
            if self.member != 'V351':
                dates = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
                train = train.loc[dates >= (dates.max().to_period('M') - 2).start_time]
            if self.member in ('V365', 'V366', 'V367', 'V368'):
                variant = {'V365': 'no_hinge', 'V366': 'asinh100', 'V367': 'state_neutral', 'V368': 'weekpart_neutral'}[self.member]
                model = Phase2GeometryComboModel({'variant': variant}, self.seed).fit(train)
                self.base, self.specialist = (model.base, None)
                self.training_summary_ = dict(model.training_summary_)
            else:
                super().fit(train)
            inputs = train[['날짜', '시간', '생산량', '기온']]
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            if self.weekend_scaler == 'weighted':
                dates = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
                weight = np.exp2(-(dates.max() - dates).dt.days.to_numpy() / 90)
                weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0) * self.day_weights(train)
                weight = weight[weekend]
                weight /= weight.mean()
                scaler = FoldWeightedStandardScaler(weight)
            else:
                scaler = RobustScaler(quantile_range=(25.0, 75.0))
            self.residuals = []
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, preprocess__remainder__standardscaler=clone(scaler), svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                correction.fit(inputs.loc[weekend], residual[weekend])
                self.residuals.append(correction)
            self.training_summary_.update(member=self.member, weekend_scaler=self.weekend_scaler, weekend_svr_loss='unchanged unweighted', replaced_step='weekend SVRs preprocess__remainder__standardscaler')
            return self

    # Source: phase2_joint_geometry_models.py
    """Combine preserved production features and quality-mass normalization."""
    pass
    pass

    class Phase2JointGeometryModel(Phase2GeometryComboModel):

        def __init__(self, config, seed=42):
            features = config['features_variant']
            assert features in ('no_hinge', 'asinh100')
            super().__init__({'variant': features}, seed)
            self.quality_balance = config['quality_balance']
            assert self.quality_balance in ('state', 'weekpart')

        def day_weights(self, train):
            model = Phase2QualitySupportModel({'balance': self.quality_balance, 'window': 'recent3'}, self.seed)
            weights = model.day_weights(train)
            self.weight_audit_ = model.weight_audit_
            return weights

        def fit(self, train):
            super().fit(train)
            self.training_summary_.update(features_variant=self.variant, quality_balance=self.quality_balance, quality_normalization=self.weight_audit_)
            return self

    # Source: phase2_catboost_models.py
    """Two fixed CatBoost losses provide new error patterns for phase-two pooling."""
    from importlib.metadata import version
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.multioutput import MultiOutputRegressor
    pass
    pass

    class Phase2CatBoostModel:

        def __init__(self, config, seed=42):
            self.mode = config.get('mode', 'joint')
            self.seed = seed
            assert self.mode in ('joint', 'mae')

        def fit(self, train):
            from catboost import CatBoostRegressor
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0)
            weight /= weight.mean()
            self.preprocess = clone(m242_pipeline[:-1]).set_params(features=ThermalStateComboFeatures())
            matrix = self.preprocess.fit_transform(inputs)
            parameters = dict(iterations=500, depth=4, learning_rate=0.04, l2_leaf_reg=10.0, random_seed=self.seed, thread_count=2, task_type='CPU', allow_writing_files=False, verbose=False, loss_function='MultiRMSE' if self.mode == 'joint' else 'MAE')
            estimator = CatBoostRegressor(**parameters)
            self.tree = estimator if self.mode == 'joint' else MultiOutputRegressor(estimator, n_jobs=1)
            self.tree.fit(matrix, train[POWER], sample_weight=weight)
            self.training_summary_ = dict(catboost_version=version('catboost'), seed=self.seed, mode=self.mode, parameters=parameters)
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            return self.tree.predict(self.preprocess.transform(inputs))

    # Source: phase2_components.py
    """Frozen phase-one components reused by forward-chain phase-two ensembles."""
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    COMPONENTS = {'V019': (ObservedStateModel, {'kind': 'observed_state', 'region': 'zero'}), 'V028': (DuplicateModel, {'kind': 'duplicate', 'mode': 'curve_sqrt'}), 'V054': (ThermalModel, {'kind': 'thermal', 'mode': 'intensity'}), 'V070': (StateInteractionModel, {'kind': 'state_interaction', 'signal': 'on', 'grouping': 'weekend'}), 'V081': (ThermalStateComboModel, {'kind': 'thermal_state_combo', 'mode': 'thermal_state'}), 'V089': (ProductionEfficiencyModel, {'kind': 'production_efficiency', 'scale': 'global', 'mode': 'add'}), 'V096': (ThermalBalanceModel, {'kind': 'thermal_balance', 'grouping': 'state_temperature', 'base': 'V081'}), 'V102': (NeighborModel, {'kind': 'neighbor', 'mode': 'direct', 'representation': 'engineered'}), 'V154': (PoissonModel, {'kind': 'poisson', 'mode': 'poisson'}), 'V159': (RadialHuberModel, {'kind': 'radial_huber', 'loss': 'radial'}), 'V174': (TreeDirectModel, {'kind': 'tree_direct', 'mode': 'standalone'}), 'V217': (Phase2QualityComboModel, {'kind': 'phase2_quality_combo', 'mode': 'state_temperature_duplicate'}), 'V239': (Phase2QualityComboModel, {'kind': 'phase2_quality_combo', 'mode': 'raw_throughput'}), 'V240': (Phase2QualityComboModel, {'kind': 'phase2_quality_combo', 'mode': 'radial'}), 'V241': (Phase2WindowModel, {'kind': 'phase2_window', 'window': 'recent1'}), 'V242': (Phase2WindowModel, {'kind': 'phase2_window', 'window': 'recent2'}), 'V243': (Phase2WindowModel, {'kind': 'phase2_window', 'window': 'recent3'}), 'V244': (Phase2WindowModel, {'kind': 'phase2_window', 'window': 'unweighted'}), 'V245': (Phase2RepresentationModel, {'kind': 'phase2_representation', 'representation': 'V054', 'weekend_correction': 'none'}), 'V246': (Phase2RepresentationModel, {'kind': 'phase2_representation', 'representation': 'V054', 'weekend_correction': 'rbf'}), 'V247': (Phase2RepresentationModel, {'kind': 'phase2_representation', 'representation': 'V070', 'weekend_correction': 'none'}), 'V248': (Phase2RepresentationModel, {'kind': 'phase2_representation', 'representation': 'V070', 'weekend_correction': 'rbf'}), 'V249': (Phase2MedianModel, {'kind': 'phase2_median_loss', 'base': 'V081'}), 'V250': (Phase2MedianModel, {'kind': 'phase2_median_loss', 'base': 'V217'}), 'V258': (Phase2WindowModel, {'kind': 'phase2_window', 'base': 'V217', 'window': 'recent3'}), 'V259': (Phase2WindowModel, {'kind': 'phase2_window', 'base': 'V217', 'window': 'no_age'}), 'V260': (Phase2StrongRepresentationModel, {'kind': 'phase2_strong_representation', 'representation': 'V081', 'weekend_correction': 'none'}), 'V261': (Phase2StrongRepresentationModel, {'kind': 'phase2_strong_representation', 'representation': 'V081', 'weekend_correction': 'rbf'}), 'V262': (Phase2StrongRepresentationModel, {'kind': 'phase2_strong_representation', 'representation': 'V217', 'weekend_correction': 'none'}), 'V263': (Phase2StrongRepresentationModel, {'kind': 'phase2_strong_representation', 'representation': 'V217', 'weekend_correction': 'rbf'}), 'V265': (Phase2ScalingModel, {'kind': 'phase2_scaling', 'base': 'V081'}), 'V266': (Phase2ScalingModel, {'kind': 'phase2_scaling', 'base': 'V217'}), 'V267': (Phase2WeekpartModel, {'kind': 'phase2_weekpart', 'base': 'V081'}), 'V268': (Phase2WeekpartModel, {'kind': 'phase2_weekpart', 'base': 'V217'}), 'V269': (Phase2QualityComboModel, {'kind': 'phase2_quality_combo', 'mode': 'features_r191'}), 'V270': (Phase2QualityComboModel, {'kind': 'phase2_quality_combo', 'mode': 'features_v054'}), 'V271': (Phase2QualityComboModel, {'kind': 'phase2_quality_combo', 'mode': 'features_v070'}), 'V277': (Phase2WeightedWeekendModel, {'kind': 'phase2_weighted_weekend', 'weight_policy': 'recency'}), 'V278': (Phase2WeightedWeekendModel, {'kind': 'phase2_weighted_weekend', 'weight_policy': 'quality'}), 'V279': (Phase2WeightedWeekendModel, {'kind': 'phase2_weighted_weekend', 'weight_policy': 'combined'}), 'V286': (Phase2BootstrapModel, {'kind': 'phase2_bootstrap', 'block': 'day', 'seed_confirmation_required': True}), 'V287': (Phase2BootstrapModel, {'kind': 'phase2_bootstrap', 'block': 'week', 'seed_confirmation_required': True}), 'V299': (Phase2WindowModel, {'kind': 'phase2_window', 'base': 'V271', 'window': 'recent3'}), 'V300': (Phase2StrongRepresentationModel, {'kind': 'phase2_strong_representation', 'representation': 'V258', 'weekend_correction': 'rbf'}), 'V301': (Phase2StrongRepresentationModel, {'kind': 'phase2_strong_representation', 'representation': 'V271', 'weekend_correction': 'rbf'}), 'V302': (Phase2ScalingModel, {'kind': 'phase2_scaling', 'base': 'V258'}), 'V303': (Phase2ScalingModel, {'kind': 'phase2_scaling', 'base': 'V271'}), 'V304': (Phase2StrongRepresentationModel, {'kind': 'phase2_strong_representation', 'representation': 'V266', 'weekend_correction': 'rbf'}), 'V317': (Phase2ScalingModel, {'kind': 'phase2_scaling', 'base': 'V299'}), 'V318': (Phase2StrongRepresentationModel, {'kind': 'phase2_strong_representation', 'representation': 'V299', 'weekend_correction': 'rbf'}), 'V319': (Phase2QualityComboModel, {'kind': 'phase2_quality_combo', 'mode': 'radial_v070'}), 'V320': (Phase2QualityComboModel, {'kind': 'phase2_quality_combo', 'mode': 'radial_v070_recent3'}), 'V321': (Phase2WindowModel, {'kind': 'phase2_window', 'base': 'V239', 'window': 'recent3'}), 'V322': (Phase2ScalingModel, {'kind': 'phase2_scaling', 'base': 'V239'}), 'V323': (Phase2StrongRepresentationModel, {'kind': 'phase2_strong_representation', 'representation': 'V239', 'weekend_correction': 'rbf'}), 'V324': (Phase2StrongRepresentationModel, {'kind': 'phase2_strong_representation', 'representation': 'V303', 'weekend_correction': 'rbf'}), 'V325': (Phase2LinearSVRModel, {'kind': 'phase2_linear_svr', 'window': 'full', 'scaler': 'standard', 'seed_confirmation_required': True}), 'V326': (Phase2LinearSVRModel, {'kind': 'phase2_linear_svr', 'window': 'full', 'scaler': 'iqr', 'seed_confirmation_required': True}), 'V327': (Phase2LinearSVRModel, {'kind': 'phase2_linear_svr', 'window': 'recent3', 'scaler': 'standard', 'seed_confirmation_required': True}), 'V328': (Phase2LinearSVRModel, {'kind': 'phase2_linear_svr', 'window': 'recent3', 'scaler': 'iqr', 'seed_confirmation_required': True}), 'V330': (Phase2LinearTreeModel, {'kind': 'phase2_linear_tree', 'base': 'V217', 'history': 'full', 'seed_confirmation_required': True}), 'V331': (Phase2LinearTreeModel, {'kind': 'phase2_linear_tree', 'base': 'V217', 'history': 'recent3', 'seed_confirmation_required': True}), 'V332': (Phase2LinearTreeModel, {'kind': 'phase2_linear_tree', 'base': 'V271', 'history': 'full', 'seed_confirmation_required': True}), 'V333': (Phase2LinearTreeModel, {'kind': 'phase2_linear_tree', 'base': 'V271', 'history': 'recent3', 'seed_confirmation_required': True}), 'V334': (Phase2StrongRepresentationModel, {'kind': 'phase2_strong_representation', 'representation': 'V317', 'weekend_correction': 'rbf'}), 'V335': (Phase2ScalingModel, {'kind': 'phase2_scaling', 'base': 'V321'}), 'V336': (Phase2StrongRepresentationModel, {'kind': 'phase2_strong_representation', 'representation': 'V321', 'weekend_correction': 'rbf'}), 'V337': (Phase2QualityComboModel, {'kind': 'phase2_quality_combo', 'mode': 'radial_v070_recent3_iqr'}), 'V338': (Phase2DistinctSupportModel, {'kind': 'phase2_distinct_support', 'grouping': 'state_temperature'}), 'V339': (Phase2DistinctSupportModel, {'kind': 'phase2_distinct_support', 'grouping': 'state_temperature_weekpart'}), 'V340': (Phase2DistinctSupportModel, {'kind': 'phase2_distinct_support', 'grouping': 'state_temperature_daytime'}), 'V341': (Phase2DistinctSupportModel, {'kind': 'phase2_distinct_support', 'grouping': 'temperature_production'}), 'V342': (Phase2ProductionRepresentationModel, {'kind': 'phase2_production_representation', 'representation': 'no_hinge', 'history': 'full'}), 'V343': (Phase2ProductionRepresentationModel, {'kind': 'phase2_production_representation', 'representation': 'no_hinge', 'history': 'recent3'}), 'V344': (Phase2ProductionRepresentationModel, {'kind': 'phase2_production_representation', 'representation': 'asinh100', 'history': 'full'}), 'V345': (Phase2ProductionRepresentationModel, {'kind': 'phase2_production_representation', 'representation': 'asinh100', 'history': 'recent3'}), 'V349': (Phase2QualitySupportModel, {'kind': 'phase2_quality_support', 'balance': 'weekpart', 'window': 'recent3'}), 'V350': (Phase2WeightedScalingModel, {'kind': 'phase2_weighted_scaling', 'base': 'V217'}), 'V351': (Phase2WeightedScalingModel, {'kind': 'phase2_weighted_scaling', 'base': 'V271'}), 'V352': (Phase2WeightedScalingModel, {'kind': 'phase2_weighted_scaling', 'base': 'V258'}), 'V353': (Phase2WeightedScalingModel, {'kind': 'phase2_weighted_scaling', 'base': 'V299'}), 'V354': (Phase2QualitySupportModel, {'kind': 'phase2_quality_support', 'balance': 'state', 'window': 'full'}), 'V355': (Phase2QualitySupportModel, {'kind': 'phase2_quality_support', 'balance': 'state', 'window': 'recent3'}), 'V356': (Phase2QualitySupportModel, {'kind': 'phase2_quality_support', 'balance': 'weekpart', 'window': 'full'}), 'V357': (Phase2ClockPoolModel, {'kind': 'phase2_clock_pool', 'blocks': 'state'}), 'V358': (Phase2ClockPoolModel, {'kind': 'phase2_clock_pool', 'blocks': 'hinge'}), 'V359': (Phase2ClockPoolModel, {'kind': 'phase2_clock_pool', 'blocks': 'state_hinge'}), 'V361': (Phase2StrongRepresentationModel, {'kind': 'phase2_strong_representation', 'representation': 'V351', 'weekend_correction': 'rbf'}), 'V362': (Phase2StrongRepresentationModel, {'kind': 'phase2_strong_representation', 'representation': 'V353', 'weekend_correction': 'rbf'}), 'V363': (Phase2WeightedScalingModel, {'kind': 'phase2_weighted_scaling', 'base': 'V271', 'loss': 'radial'}), 'V364': (Phase2WeightedScalingModel, {'kind': 'phase2_weighted_scaling', 'base': 'V299', 'loss': 'radial'}), 'V365': (Phase2GeometryComboModel, {'kind': 'phase2_geometry_combo', 'variant': 'no_hinge'}), 'V366': (Phase2GeometryComboModel, {'kind': 'phase2_geometry_combo', 'variant': 'asinh100'}), 'V367': (Phase2GeometryComboModel, {'kind': 'phase2_geometry_combo', 'variant': 'state_neutral'}), 'V368': (Phase2GeometryComboModel, {'kind': 'phase2_geometry_combo', 'variant': 'weekpart_neutral'}), 'V369': (Phase2CategoricalModel, {'kind': 'phase2_categorical', 'coding': 'symmetric'}), 'V370': (Phase2CategoricalModel, {'kind': 'phase2_categorical', 'coding': 'weighted_variance'}), 'V371': (Phase2WeekendScalingModel, {'kind': 'phase2_weekend_scaling', 'base': 'V351', 'weekend_scaler': 'weighted'}), 'V372': (Phase2WeekendScalingModel, {'kind': 'phase2_weekend_scaling', 'base': 'V353', 'weekend_scaler': 'weighted'}), 'V373': (Phase2WeekendScalingModel, {'kind': 'phase2_weekend_scaling', 'base': 'V351', 'weekend_scaler': 'iqr'}), 'V374': (Phase2WeekendScalingModel, {'kind': 'phase2_weekend_scaling', 'base': 'V353', 'weekend_scaler': 'iqr'}), 'V376': (Phase2GeometryComboModel, {'kind': 'phase2_geometry_combo', 'variant': 'no_hinge', 'weekend_correction': 'rbf'}), 'V377': (Phase2GeometryComboModel, {'kind': 'phase2_geometry_combo', 'variant': 'weekpart_neutral', 'weekend_correction': 'rbf'}), 'V378': (Phase2GeometryComboModel, {'kind': 'phase2_geometry_combo', 'variant': 'no_hinge', 'loss': 'radial'}), 'V379': (Phase2GeometryComboModel, {'kind': 'phase2_geometry_combo', 'variant': 'weekpart_neutral', 'loss': 'radial'}), 'V380': (Phase2JointGeometryModel, {'kind': 'phase2_joint_geometry', 'features_variant': 'no_hinge', 'quality_balance': 'state'}), 'V381': (Phase2JointGeometryModel, {'kind': 'phase2_joint_geometry', 'features_variant': 'no_hinge', 'quality_balance': 'weekpart'}), 'V382': (Phase2JointGeometryModel, {'kind': 'phase2_joint_geometry', 'features_variant': 'asinh100', 'quality_balance': 'state'}), 'V383': (Phase2JointGeometryModel, {'kind': 'phase2_joint_geometry', 'features_variant': 'asinh100', 'quality_balance': 'weekpart'}), 'V385': (Phase2WeekendScalingModel, {'kind': 'phase2_weekend_scaling', 'base': 'V365', 'weekend_scaler': 'iqr'}), 'V386': (Phase2WeekendScalingModel, {'kind': 'phase2_weekend_scaling', 'base': 'V366', 'weekend_scaler': 'iqr'}), 'V387': (Phase2WeekendScalingModel, {'kind': 'phase2_weekend_scaling', 'base': 'V368', 'weekend_scaler': 'iqr'}), 'V388': (Phase2WeekendScalingModel, {'kind': 'phase2_weekend_scaling', 'base': 'V367', 'weekend_scaler': 'iqr'})}
    V305_CONFIG = {'kind': 'phase2_blend', 'phase': 2, 'reference': 'V217', 'strategy': 'median', 'hypothesis': 'P2_H41', 'members': ['V299', 'V303', 'V258']}

    class M242Component:

        def fit(self, train):
            self.model = fit_m242(train)
            return self

        def predict(self, frame):
            return self.model.predict(frame)

    def make_component(name, seed=42):
        if name == 'M242':
            return M242Component()
        if name == 'N207':
            return N207Model()
        if name == 'R191':
            return R192Model(zero_weight=0.5)
        if name == 'R100':
            return R100Model()
        if name in ('V176', 'V177'):
            pass
            return Phase2CatBoostModel({'mode': 'joint' if name == 'V176' else 'mae'}, seed)
        if name == 'V305':
            pass
            return Phase2BlendModel(dict(V305_CONFIG), seed)
        model, config = COMPONENTS[name]
        return model(dict(config), seed)

    def fit_component(name, train, seed=42):
        return make_component(name, seed).fit(train)

    # Source: phase2_blend_models.py
    """Chronological, rowwise multi-model blending; callers supply past OOF only."""
    import numpy as np
    import pandas as pd
    from scipy import sparse
    from scipy.optimize import linprog, minimize
    from scipy.stats import rankdata
    from sklearn.linear_model import Ridge
    pass
    pass
    INPUTS = ['날짜', '시간', '생산량', '기온', '풍속', '습도', '강수량']

    def _members(config):
        return list(config.get('members', config.get('components', [])))

    def _prior(config, count):
        if 'prior_weights' in config:
            weights = np.asarray(config['prior_weights'], dtype=float)
        elif config.get('prior', 'anchor') == 'uniform':
            weights = np.ones(count)
        else:
            members = _members(config)
            anchor = config.get('anchor', 'V081')
            weights = np.zeros(count)
            weights[members.index(anchor) if anchor in members else 0] = 1.0
        assert weights.shape == (count,) and (weights >= 0).all() and (weights.sum() > 0)
        return weights / weights.sum()

    def _groups(frame, grouping):
        date = pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d')
        on = frame['생산량'].to_numpy() > 0
        weekend = date.dt.dayofweek.to_numpy() >= 5
        hour = frame['시간'].to_numpy()
        band = np.where(hour < 8, 0, np.where(hour <= 18, 1, 2))
        return {'zero': on.astype(int), 'weekend': weekend.astype(int), 'state_weekend': on.astype(int) * 2 + weekend, 'hour_band': band, 'state_hour': on.astype(int) * 3 + band}[grouping]

    def _indicators(frame):
        date = pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d')
        return np.column_stack([np.ones(len(frame)), frame['생산량'].to_numpy() > 0, date.dt.dayofweek.to_numpy() >= 5, frame['시간'].between(8, 18).to_numpy()]).astype(float)

    def _family_predictions(predictions, members, families, representative):
        flattened = [member for family in families.values() for member in family]
        assert len(flattened) == len(members) and sorted(flattened) == sorted(members)
        assert representative in ('mean', 'median')
        reduce = np.mean if representative == 'mean' else np.median
        return np.stack([reduce(predictions[:, [members.index(member) for member in family], :], axis=1) for family in families.values()], axis=1)

    def _conditional_groups(state, predictions, inputs):
        if 'regime' in state:
            return apply_regime(inputs, state['regime'])
        if state['group'] == 'disagreement':
            anchor = predictions[:, state['disagreement_anchor_index'], :]
            disagreement = np.abs(predictions - anchor[:, None, :]).mean(axis=(1, 2))
            return (disagreement > state['disagreement_threshold']).astype(int)
        return _groups(inputs, state['group'])

    def _flat(prediction, target, row_weight):
        matrix = prediction.transpose(0, 2, 1).reshape(-1, prediction.shape[1])
        weights = np.repeat(row_weight, prediction.shape[2])
        return (matrix, target.reshape(-1), weights / weights.sum())

    def _simplex(prediction, target, row_weight, prior, config, strategy):
        matrix, truth, weight = _flat(prediction, target, row_weight)
        count, models = matrix.shape
        penalty = float(config.get('regularization', 0.25 if strategy == 'simplex_mae' else 0.1))
        if strategy == 'simplex_mae':
            identity = sparse.eye(count, format='csr')
            zeros = sparse.csr_matrix((count, models))
            model_eye = sparse.eye(models, format='csr')
            padding = sparse.csr_matrix((models, count))
            constraints = sparse.vstack([sparse.hstack([matrix, -identity, zeros]), sparse.hstack([-matrix, -identity, zeros]), sparse.hstack([model_eye, padding, -model_eye]), sparse.hstack([-model_eye, padding, -model_eye])], format='csr')
            result = linprog(np.r_[np.zeros(models), weight, np.full(models, penalty)], A_ub=constraints, b_ub=np.r_[truth, -truth, prior, -prior], A_eq=sparse.csr_matrix(np.r_[np.ones(models), np.zeros(count + models)][None]), b_eq=[1.0], bounds=(0.0, None), method='highs')
            assert result.success, result.message
            weights = result.x[:models]
            details = dict(objective=float(result.fun), solver='highs', regularization=penalty)
        else:
            scale = max(float(np.sqrt(np.average((matrix @ prior - truth) ** 2, weights=weight))), 1.0)

            def objective(weights):
                error = matrix @ weights - truth
                value = np.sum(weight * error ** 2) / scale ** 2 + penalty * np.sum((weights - prior) ** 2)
                gradient = 2 * matrix.T @ (weight * error) / scale ** 2 + 2 * penalty * (weights - prior)
                return (value, gradient)
            result = minimize(objective, prior, jac=True, method='SLSQP', bounds=[(0.0, 1.0)] * models, constraints={'type': 'eq', 'fun': lambda w: w.sum() - 1.0, 'jac': lambda w: np.ones(models)}, options={'ftol': 1e-10, 'maxiter': 500})
            assert result.success, result.message
            weights = result.x
            details = dict(objective=float(result.fun), solver='SLSQP', regularization=penalty, scale=scale)
        weights = np.maximum(weights, 0.0)
        weights /= weights.sum()
        return (weights, details)

    def fit_meta(predictions, targets, inputs, config):
        """Fit on strictly historical OOF supplied by the caller; no outer truth."""
        predictions, targets = (np.asarray(predictions), np.asarray(targets))
        assert predictions.ndim == 3 and predictions.shape[2] == 4
        assert targets.shape == (len(inputs), 4) and len(predictions) == len(inputs)
        assert len(inputs) and np.isfinite(predictions).all() and np.isfinite(targets).all()
        months = inputs['날짜'].to_numpy() // 100
        assert months.min() >= 202103 and months.max() <= 202108
        history = config.get('history', 'expanding')
        assert history in ('expanding', 'recent3')
        if history == 'recent3':
            selected = np.isin(months, np.unique(months)[-3:])
            predictions, targets = (predictions[selected], targets[selected])
            inputs, months = (inputs.iloc[np.flatnonzero(selected)], months[selected])
        strategy = config.get('strategy', config.get('method', 'mean'))
        members = _members(config)
        assert len(members) == predictions.shape[1]
        prior = _prior(config, len(members))
        _, inverse, counts = np.unique(months, return_inverse=True, return_counts=True)
        row_weight = 1.0 / counts[inverse]
        state = dict(strategy=strategy, members=members, training_rows=len(inputs), training_months=[int(m) for m in np.unique(months)], month_row_counts={str(m): int((months == m).sum()) for m in np.unique(months)}, prior=prior.tolist(), weighting='equal_month', history=history)
        if strategy in ('mean', 'median', 'trimmed'):
            state['weights'] = (np.ones(len(members)) / len(members)).tolist()
        elif strategy == 'fixed':
            weights = np.asarray(config['weights'], dtype=float)
            assert weights.shape == prior.shape and (weights >= 0).all() and (weights.sum() > 0)
            state['weights'] = (weights / weights.sum()).tolist()
        elif strategy in ('performance', 'filtered_mean'):
            mae = np.average(np.abs(predictions - targets[:, None, :]).mean(axis=2), axis=0, weights=row_weight)
            state['past_mae'] = dict(zip(members, mae.tolist()))
            if strategy == 'performance':
                rule = config.get('performance', 'inverse_mae')
                assert rule in ('inverse_mae', 'rank')
                weights = 1.0 / (np.maximum(mae, 1e-08) if rule == 'inverse_mae' else rankdata(mae, method='average'))
                weights /= weights.sum()
                state.update(performance=rule, performance_weights=weights.tolist(), prior_shrink=0.5, weights=(0.5 * prior + 0.5 * weights).tolist())
            else:
                anchor = config.get('anchor', 'V081')
                index = members.index(anchor) if anchor in members else 0
                eligible = mae <= mae[index]
                eligible[index] = True
                representative = config.get('representative', 'mean')
                assert representative in ('mean', 'median')
                state.update(anchor=members[index], representative=representative, eligible_indices=np.flatnonzero(eligible).tolist(), eligible_members=[name for name, keep in zip(members, eligible) if keep], weights=(eligible / eligible.sum()).tolist())
        elif strategy == 'per_target':
            base_strategy = config.get('base_strategy', 'simplex_mae')
            assert base_strategy in ('simplex_mae', 'ridgeconvex')
            state.update(weights=[], optimization=[], base_strategy=base_strategy)
            for target in range(4):
                weights, detail = _simplex(predictions[:, :, target:target + 1], targets[:, target:target + 1], row_weight, prior, config, base_strategy)
                state['weights'].append(weights.tolist())
                state['optimization'].append(detail)
        elif strategy == 'hierarchical':
            families = config['families']
            representative = config.get('representative', 'mean')
            grouped = _family_predictions(predictions, members, families, representative)
            if config.get('prior') == 'uniform' and 'prior_weights' not in config:
                family_prior = np.ones(len(families)) / len(families)
            else:
                family_prior = np.array([sum((prior[members.index(member)] for member in family)) for family in families.values()])
            base_strategy = config.get('base_strategy', 'simplex_mae')
            assert base_strategy in ('simplex_mae', 'ridgeconvex')
            weights, detail = _simplex(grouped, targets, row_weight, family_prior, config, base_strategy)
            state.update(families=families, representative=representative, base_strategy=base_strategy, family_prior=family_prior.tolist(), weights=weights.tolist(), optimization=detail)
        elif strategy in ('simplex_mae', 'ridgeconvex', 'conditional'):
            base_strategy = config.get('base_strategy', 'simplex_mae') if strategy == 'conditional' else strategy
            weights, state['optimization'] = _simplex(predictions, targets, row_weight, prior, config, base_strategy)
            state['weights'] = weights.tolist()
            if strategy == 'conditional':
                grouping = config.get('group', 'state_weekend')
                state.update(group=grouping, groups={}, group_shrink_rows=720, minimum_group_rows=96)
                if grouping in ('temperature', 'production_band'):
                    state['regime'] = fit_regime(inputs, grouping)
                if grouping == 'disagreement':
                    anchor = config.get('anchor', 'V081')
                    index = members.index(anchor) if anchor in members else 0
                    disagreement = np.abs(predictions - predictions[:, index:index + 1, :]).mean(axis=(1, 2))
                    state.update(disagreement_anchor_index=index, disagreement_anchor=members[index], disagreement_quantile=0.75, disagreement_threshold=float(np.quantile(disagreement, 0.75)))
                groups = _conditional_groups(state, predictions, inputs)
                for group in np.unique(groups):
                    if group < 0:
                        continue
                    selected = groups == group
                    rows = int(selected.sum())
                    if rows < state['minimum_group_rows']:
                        continue
                    local, detail = _simplex(predictions[selected], targets[selected], row_weight[selected], weights, config, base_strategy)
                    shrink = rows / (rows + state['group_shrink_rows'])
                    state['groups'][str(int(group))] = dict(rows=rows, shrink=shrink, weights=(weights * (1 - shrink) + local * shrink).tolist(), optimization=detail)
        elif strategy == 'ridge':
            base = np.einsum('nkt,k->nt', predictions, prior)
            difference = (predictions - base[:, None, :]).transpose(0, 2, 1).reshape(-1, len(members))
            scale = np.maximum(difference.std(axis=0), 1.0)
            design = np.column_stack([difference / scale, np.repeat(_indicators(inputs), 4, axis=0)])
            alpha = float(config.get('alpha', 10000.0))
            estimator = Ridge(alpha=alpha, fit_intercept=False)
            estimator.fit(design, (targets - base).reshape(-1), sample_weight=np.repeat(row_weight / row_weight.mean(), 4))
            state.update(scale=scale.tolist(), coefficient=estimator.coef_.tolist(), alpha=alpha)
        else:
            raise ValueError(strategy)
        return state

    def predict_meta(state, predictions, inputs):
        predictions = np.asarray(predictions)
        strategy = state['strategy']
        if strategy == 'median':
            return np.median(predictions, axis=1)
        if strategy == 'trimmed':
            assert predictions.shape[1] >= 3
            return np.sort(predictions, axis=1)[:, 1:-1, :].mean(axis=1)
        if strategy == 'filtered_mean':
            reduce = np.mean if state['representative'] == 'mean' else np.median
            return reduce(predictions[:, state['eligible_indices'], :], axis=1)
        if strategy == 'per_target':
            return np.einsum('nkt,tk->nt', predictions, state['weights'])
        if strategy == 'hierarchical':
            grouped = _family_predictions(predictions, state['members'], state['families'], state['representative'])
            return np.einsum('nkt,k->nt', grouped, state['weights'])
        if strategy == 'ridge':
            base = np.einsum('nkt,k->nt', predictions, state['prior'])
            difference = (predictions - base[:, None, :]).transpose(0, 2, 1).reshape(-1, predictions.shape[1])
            design = np.column_stack([difference / state['scale'], np.repeat(_indicators(inputs), 4, axis=0)])
            return base + (design @ state['coefficient']).reshape(-1, 4)
        prediction = np.einsum('nkt,k->nt', predictions, state['weights'])
        if strategy == 'conditional':
            groups = _conditional_groups(state, predictions, inputs)
            for group, fitted in state['groups'].items():
                selected = groups == int(group)
                prediction[selected] = np.einsum('nkt,k->nt', predictions[selected], fitted['weights'])
        return prediction

    class Phase2Blend:
        """Small cache-harness adapter; all state is JSON serializable."""

        def __init__(self, config, seed=42):
            self.config, self.seed = (config, seed)

        def fit(self, inputs, predictions, targets):
            self.state_ = fit_meta(predictions, targets, inputs, self.config)
            return self

        def predict(self, inputs, predictions):
            return predict_meta(self.state_, predictions, inputs)

        def audit(self):
            return self.state_

    class Phase2BlendModel:
        """Fixed heads need no meta labels; learned heads use forward-chain rows."""

        def __init__(self, config, seed=42):
            self.config, self.seed = (config, seed)

        def fit(self, train):
            pass
            assert train['날짜'].between(20210101, 20210831).all()
            self.members_ = _members(self.config)
            strategy = self.config.get('strategy', self.config.get('method', 'mean'))
            if strategy in ('mean', 'median', 'trimmed', 'fixed'):
                weights = np.asarray(self.config['weights'], dtype=float) if strategy == 'fixed' else np.ones(len(self.members_))
                assert weights.shape == (len(self.members_),) and (weights >= 0).all() and (weights.sum() > 0)
                self.meta_ = Phase2Blend(self.config, self.seed)
                self.meta_.state_ = dict(strategy=strategy, members=self.members_, weights=(weights / weights.sum()).tolist(), training_rows=0, training_months=[], meta_training_required=False)
            else:
                months = train['날짜'].to_numpy() // 100
                frames, predictions, targets = ([], [], [])
                for month in sorted(set(months)):
                    if month < 202103:
                        continue
                    past, valid = (train.loc[months < month], train.loc[months == month])
                    prediction = np.stack([make_component(name, self.seed).fit(past).predict(valid[INPUTS]) for name in self.members_], axis=1)
                    frames.append(valid[INPUTS])
                    predictions.append(prediction)
                    targets.append(valid[POWER].to_numpy())
                self.meta_ = Phase2Blend(self.config, self.seed).fit(pd.concat(frames), np.concatenate(predictions), np.concatenate(targets))
            self.models_ = [make_component(name, self.seed).fit(train) for name in self.members_]
            self.training_summary_ = self.meta_.audit()
            return self

        def predict(self, frame):
            predictions = np.stack([model.predict(frame[INPUTS]) for model in self.models_], axis=1)
            return self.meta_.predict(frame[INPUTS], predictions)

    NAME, CONFIG, SEED = 'V305', {'kind': 'phase2_blend', 'phase': 2, 'reference': 'V217', 'strategy': 'median', 'hypothesis': 'P2_H41', 'members': ['V299', 'V303', 'V258']}, 42
    INPUTS = ['날짜', '시간', '생산량', '기온', '풍속', '습도', '강수량']
    np.random.seed(SEED)
    filtered_train = train_df.loc[
        train_df['날짜'].between(20210101, 20210831) & train_df['시간'].between(0, 23)
    ].sort_values(['날짜', '시간']).copy()
    assert len(filtered_train) == 5784
    assert not filtered_train.duplicated(['날짜', '시간']).any()
    assert filtered_train[POWER].notna().all().all()

    def fit_exported(frame):
        return Phase2BlendModel(CONFIG, SEED).fit(frame)

    # 1~3월→4월, …, 1~7월→8월. 구성 모델 학습 기간·가중치 규칙 원본 그대로
    scores, predictions = [], []
    with threadpool_limits(limits=2):
        for month in VALIDATION_MONTHS:
            tr = filtered_train.loc[filtered_train['날짜'].between(20210101, month * 100 - 1)]
            va = filtered_train.loc[filtered_train['날짜'].floordiv(100).eq(month)]
            assert not tr.empty and not va.empty
            assert tr['날짜'].max() < va['날짜'].min()
            fold_model = fit_exported(tr)
            prediction = fold_model.predict(va[INPUTS])
            assert prediction.shape == (len(va), 4) and np.isfinite(prediction).all()
            scores.append(dict(month=month, **metrics(va[POWER], prediction)))
            predictions.append(pd.DataFrame(prediction, index=va.index, columns=POWER).assign(month=month))
            print(f"{NAME} {month}: MAE={scores[-1]['MAE']:.6f}")
            del fold_model
        # 최종 모델: 1~8월 데이터 전달, 구성 모델별 학습 범위 적용
        final_model = fit_exported(filtered_train)
    return pd.DataFrame(scores).set_index('month'), pd.concat(predictions).sort_index(), final_model


cv_scores_v305, cv_predictions_v305, model_11 = train_v305(train_df)
cv_results["V305"] = {"scores": cv_scores_v305}
print("V305 월별 확장 교차검증")
print(cv_scores_v305.to_string())
print("월평균 MAE:", cv_scores_v305["MAE"].mean())


V305 202104: MAE=16.820242
V305 202105: MAE=10.616630
V305 202106: MAE=11.855017
V305 202107: MAE=12.329532
V305 202108: MAE=8.302649
V305 월별 확장 교차검증
              MAE       RMSE        R2
month                                 
202104  16.820242  28.587588  0.719112
202105  10.616630  23.966722  0.809286
202106  11.855017  20.212683  0.867753
202107  12.329532  23.268473  0.854769
202108   8.302649  15.001639  0.941043
월평균 MAE: 11.98481388419361


#### V384 (규칙으로 선택된 54개 모델의 중앙값 블렌딩)

- 구성 모델 54개 예측 중앙값, 월별 확장 윈도우 CV, MAE·RMSE·R²

In [18]:
# V384: 전체 파이프라인 단일 셀 (외부 연구 모듈 import 불필요)
# 원본: scripts/research_v2/cells_phase2/V384.py
# 원본 SHA256: 787ce2067eea25c0125c99c4d02f6c45db5e76ae7c0b2fda5ee83f0426692142
# 내부 이름 함수 범위 한정 (앞선 모델 클래스·피처·설정 보호)
# 9월 예측·평가: '9월 테스트' 셀
def train_v384(train_df):
    # %% V384: standalone cell for notebooks/01model.ipynb
    # Requires train_df; test_df is never scored by this cell.

    # Source: notebook_models.py
    from collections import Counter, defaultdict
    import pandas as pd
    from sklearn.linear_model import LinearRegression
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import HuberRegressor
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.pipeline import Pipeline, make_pipeline
    from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
    from sklearn.svm import SVR
    from threadpoolctl import threadpool_limits
    import numpy as np
    import pandas as pd
    SEED = 42
    POWER = ['15분', '30분', '45분', '60분']
    VALIDATION_MONTHS = [202104, 202105, 202106, 202107, 202108]

    def metrics(y, pred):
        y, pred = (np.asarray(y).ravel(), np.asarray(pred).ravel())
        return dict(MAE=mean_absolute_error(y, pred), RMSE=np.sqrt(mean_squared_error(y, pred)), R2=r2_score(y, pred))

    def m242_features(frame):
        """날짜·시간·생산량·기온 4개 열 기반 피처"""
        assert frame['시간'].between(0, 23).all(), 'hour must be 0..23'
        hour = frame['시간']
        weekday = pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d').dt.dayofweek
        log_prod = np.log1p(frame['생산량'])
        on = log_prod.gt(0).where(log_prod.notna()).astype(float)
        hinge = (log_prod - np.log1p(100)).clip(lower=0)
        x = pd.DataFrame({'hour_weekday': hour.astype(str) + '_' + weekday.astype(str), '기온': frame['기온'], 'log_prod': log_prod, 'on': on, 'temp_on': frame['기온'] * on, 'temp_above20': (frame['기온'] - 20).clip(lower=0)}, index=frame.index)
        for day in range(7):
            x[f'prod_day{day}'] = log_prod * weekday.eq(day)
        for h in range(24):
            x[f'prod_hour{h}'] = log_prod * hour.eq(h)
        for h in range(24):
            x[f'prod_hinge_hour{h}'] = hinge * hour.eq(h)
        x['prod_sin'] = log_prod * np.sin(2 * np.pi * hour / 24)
        x['prod_cos'] = log_prod * np.cos(2 * np.pi * hour / 24)
        return x
    m242_pipeline = Pipeline([('features', FunctionTransformer(m242_features)), ('preprocess', ColumnTransformer([('category', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), ['hour_weekday'])], remainder=make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler()))), ('huber', MultiOutputRegressor(HuberRegressor(alpha=1.5, epsilon=1.05, max_iter=2000, tol=1e-07)))])

    def m242_sample_weight(train):
        """최근 가중(반감기 90일), 생산량 0 행 2배 가중"""
        date = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
        weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
        weight = np.where(train['생산량'].eq(0), 2.0, 1.0) * weight / weight.mean()
        return weight / weight.mean()

    def fit_m242(train):
        model = clone(m242_pipeline)
        with threadpool_limits(limits=2):
            return model.fit(train, train[POWER], huber__sample_weight=m242_sample_weight(train))

    def is_weekend(frame):
        return pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d').dt.dayofweek.ge(5)

    def n207_residual_features(frame):
        """주말 잔차 보정용: 시간·요일(범주), 기온·로그 생산량"""
        assert frame['시간'].between(0, 23).all(), 'hour must be 0..23'
        return pd.DataFrame({'시간': frame['시간'], '기온': frame['기온'], 'weekday': pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d').dt.dayofweek, 'log_prod': np.log1p(frame['생산량'])}, index=frame.index)
    n207_residual_pipeline = Pipeline([('features', FunctionTransformer(n207_residual_features)), ('preprocess', ColumnTransformer([('category', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['시간', 'weekday'])], remainder=make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler()))), ('svr', MultiOutputRegressor(SVR(kernel='rbf', C=10, epsilon=1, gamma='scale', tol=0.0001)))])

    class N207Model:
        """M242(전체 행 학습) + 주말 행 잔차 보정기. predict(frame) → (행 수, 4)"""

        def fit(self, train):
            self.base = fit_m242(train)
            residual = train[POWER] - self.base.predict(train)
            weekend = is_weekend(train)
            with threadpool_limits(limits=2):
                self.residual = clone(n207_residual_pipeline).fit(train.loc[weekend], residual.loc[weekend])
            return self

        def predict(self, frame):
            pred = self.base.predict(frame)
            weekend = is_weekend(frame).to_numpy()
            if weekend.any():
                pred[weekend] += self.residual.predict(frame.loc[weekend])
            return pred
    from sklearn.base import BaseEstimator, TransformerMixin

    class RHuberFeatures(TransformerMixin, BaseEstimator):
        """날짜 추세 기준일 fit 시 고정(평가 시 재계산 없음)"""

        def __init__(self, trend=False):
            self.trend = trend

        def fit(self, frame, y=None):
            if self.trend:
                date = pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d')
                self.anchor_ = float(date.max().toordinal())
            return self

        def transform(self, frame):
            assert frame['생산량'].dropna().ge(0).all()
            x = m242_features(frame)
            if self.trend:
                date = pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d')
                ordinal = date.map(lambda value: value.toordinal()).astype(float)
                gate = frame['생산량'].eq(0) & date.dt.dayofweek.lt(5)
                x['bounded_date_trend'] = ((ordinal - self.anchor_) / 60).clip(-1, 0) * gate
            return x

    class RWeekendModel:
        """전체 학습 행 Huber + 주말 학습 행 잔차 SVR. 출력 순서 = POWER"""

        def __init__(self, kernel='rbf', numeric_weight=1.0, zero_weight=2.0, trend=False):
            self.kernel = kernel
            self.numeric_weight = numeric_weight
            self.zero_weight = zero_weight
            self.trend = trend

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days / 90)
            weight = (weight / weight.mean()).to_numpy()
            weight = np.where(inputs['생산량'].eq(0), self.zero_weight, 1.0) * weight
            weight = weight / weight.mean()
            self.base = clone(m242_pipeline).set_params(features=RHuberFeatures(trend=self.trend))
            with threadpool_limits(limits=2):
                self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = date.dt.dayofweek.ge(5).to_numpy()
            assert weekend.any(), '주말 잔차 모델을 학습할 행이 필요합니다.'
            self.residual = Pipeline([('features', FunctionTransformer(n207_residual_features)), ('preprocess', ColumnTransformer([('category', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['시간', 'weekday']), ('numeric', make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler()), ['기온', 'log_prod'])], transformer_weights={'numeric': self.numeric_weight})), ('svr', MultiOutputRegressor(SVR(kernel=self.kernel, degree=2, coef0=1, C=10, epsilon=1, gamma='scale', tol=0.0001)))])
            with threadpool_limits(limits=2):
                self.residual.fit(inputs.loc[weekend], residual[weekend])
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            prediction = self.base.predict(inputs)
            weekend = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d').dt.dayofweek.ge(5).to_numpy()
            if weekend.any():
                prediction[weekend] += self.residual.predict(inputs.loc[weekend])
            return prediction

    class R192Model:
        """R192: 날짜 추세를 넣은 RWeekendModel 두 개(RBF 수치 가중 .5 / poly)를 .75/.25로 조합.
        zero_weight: Huber 생산량 0 행 가중치 (R192=1.0, R191=0.5)."""

        def __init__(self, zero_weight=1.0):
            self.zero_weight = zero_weight

        def fit(self, train):
            self.members = [RWeekendModel('rbf', numeric_weight=0.5, zero_weight=self.zero_weight, trend=True).fit(train), RWeekendModel('poly', zero_weight=self.zero_weight, trend=True).fit(train)]
            return self

        def predict(self, frame):
            left, right = (member.predict(frame) for member in self.members)
            return 0.75 * left + 0.25 * right

    class R100Model:

        def fit(self, train):
            self.members = [RWeekendModel('rbf').fit(train), RWeekendModel('poly').fit(train)]
            return self

        def predict(self, frame):
            left, right = (member.predict(frame) for member in self.members)
            zero = frame['생산량'].eq(0).to_numpy()[:, None]
            return np.where(zero, left, right)

    # Source: phase2_regimes.py
    """Historical-fit, rowwise regimes for conditional multi-model blending."""
    import numpy as np

    def fit_regime(inputs, grouping):
        if grouping == 'temperature':
            return {'group': grouping, 'cut_points': [18.0, 26.0]}
        if grouping == 'production_band':
            production = inputs['생산량'].to_numpy(dtype=float)
            positive = production[np.isfinite(production) & (production > 0)]
            return {'group': grouping, 'positive_q25': float(np.quantile(positive, 0.25))}
        raise ValueError(grouping)

    def apply_regime(inputs, state):
        if state['group'] == 'temperature':
            values = inputs['기온'].to_numpy(dtype=float)
            groups = np.searchsorted(state['cut_points'], values, side='right')
        elif state['group'] == 'production_band':
            values = inputs['생산량'].to_numpy(dtype=float)
            groups = np.where(values <= 0, 0, np.where(values <= state['positive_q25'], 1, 2))
        else:
            raise ValueError(state['group'])
        return np.where(np.isfinite(values), groups, -1).astype(int)

    # Source: observed_state_models.py
    """Direct regression within an observed zero-production state."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from threadpoolctl import threadpool_limits
    pass

    class ObservedStateModel:

        def __init__(self, config, seed=42):
            self.region = config.get('region', 'zero')
            assert self.region in ('zero', 'weekday_zero')

        def region_mask(self, frame):
            selected = frame['생산량'].eq(0)
            if self.region == 'weekday_zero':
                date = pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d')
                selected &= date.dt.dayofweek.lt(5)
            return selected

        def fit(self, train):
            region = train.loc[self.region_mask(train)]
            inputs = region[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            weight /= weight.mean()
            self.specialist = clone(m242_pipeline).set_params(features=RHuberFeatures(trend=True))
            with threadpool_limits(limits=2):
                self.base = R192Model(zero_weight=0.5).fit(train)
                self.specialist.fit(inputs, region[POWER], huber__sample_weight=weight)
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            prediction = self.base.predict(inputs)
            selected = self.region_mask(inputs).to_numpy()
            if selected.any():
                prediction[selected] = self.specialist.predict(inputs.loc[selected])
            return prediction

    # Source: duplicate_models.py
    """Weight exact repeated training-day curves without removing any rows."""
    from collections import Counter
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from threadpoolctl import threadpool_limits
    pass

    class DuplicateModel:

        def __init__(self, config, seed=42):
            self.mode = config.get('mode', 'curve_equal')
            assert self.mode in ('curve_equal', 'curve_sqrt')

        def day_weights(self, train):
            curves = {}
            for date, day in train.groupby('날짜'):
                day = day.sort_values('시간')
                if len(day) == 24 and np.array_equal(day['시간'].to_numpy(), np.arange(24)):
                    curves[date] = day[POWER].to_numpy(dtype='<f8').tobytes()
            counts = Counter(curves.values())
            exponent = 1.0 if self.mode == 'curve_equal' else 0.5
            weights = {date: counts[curve] ** (-exponent) for date, curve in curves.items()}
            return train['날짜'].map(weights).fillna(1.0).to_numpy()

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0) * self.day_weights(train)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=RHuberFeatures(trend=True))
            weekend = is_weekend(inputs).to_numpy()
            self.residuals = []
            with threadpool_limits(limits=2):
                self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
                residual = train[POWER].to_numpy() - self.base.predict(inputs)
                for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                    correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                    correction.fit(inputs.loc[weekend], residual[weekend])
                    self.residuals.append(correction)
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            prediction = self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            if weekend.any():
                left, right = (model.predict(inputs.loc[weekend]) for model in self.residuals)
                prediction[weekend] += 0.75 * left + 0.25 * right
            return prediction

    # Source: thermal_models.py
    """H18: fixed 20-degree cooling interactions with operating state/intensity."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.pipeline import Pipeline, make_pipeline
    from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
    from sklearn.svm import SVR
    from threadpoolctl import threadpool_limits
    pass

    class ThermalFeatures(RHuberFeatures):

        def __init__(self, mode='state_hinge'):
            super().__init__(trend=True)
            self.mode = mode

        def transform(self, frame):
            features = super().transform(frame)
            log_prod = np.log1p(frame['생산량'])
            on = log_prod.gt(0).where(log_prod.notna()).astype(float)
            cooling = (frame['기온'] - 20).clip(lower=0)
            if self.mode in ('state_hinge', 'both'):
                features['cooling_on'] = cooling * on
                features['cooling_off'] = cooling * (1 - on)
            if self.mode in ('intensity', 'both'):
                features['temperature_intensity'] = frame['기온'] * log_prod
                features['cooling_intensity'] = cooling * log_prod
            if self.mode == 'daytime_hinge':
                angle = 2 * np.pi * frame['시간'] / 24
                features['cooling_on'] = cooling * on
                features['cooling_on_sin'] = cooling * on * np.sin(angle)
                features['cooling_on_cos'] = cooling * on * np.cos(angle)
            return features

    class ThermalModel:

        def __init__(self, config, seed=42):
            self.mode = config.get('mode', 'state_hinge')
            assert self.mode in ('state_hinge', 'intensity', 'daytime_hinge', 'both')

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days / 90)
            weight = (weight / weight.mean()).to_numpy()
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=ThermalFeatures(self.mode))
            weekend = date.dt.dayofweek.ge(5).to_numpy()
            self.residuals = []
            with threadpool_limits(limits=2):
                self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
                residual = train[POWER].to_numpy() - self.base.predict(inputs)
                for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                    model = Pipeline([('features', FunctionTransformer(n207_residual_features)), ('preprocess', ColumnTransformer([('category', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['시간', 'weekday']), ('numeric', make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler()), ['기온', 'log_prod'])], transformer_weights={'numeric': numeric_weight})), ('svr', MultiOutputRegressor(SVR(kernel=kernel, degree=2, coef0=1, C=10, epsilon=1, gamma='scale', tol=0.0001)))])
                    self.residuals.append(model.fit(inputs.loc[weekend], residual[weekend]))
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            prediction = self.base.predict(inputs)
            weekend = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d').dt.dayofweek.ge(5).to_numpy()
            if weekend.any():
                left, right = (model.predict(inputs.loc[weekend]) for model in self.residuals)
                prediction[weekend] += 0.75 * left + 0.25 * right
            return prediction

    # Source: state_interaction_models.py
    """H23: retain pooled training while adding state-dependent calendar responses."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.svm import SVR
    pass

    class StateInteractionFeatures(RHuberFeatures):

        def __init__(self, signal='on', grouping='weekday'):
            super().__init__(trend=True)
            self.signal, self.grouping = (signal, grouping)

        def transform(self, frame):
            features = super().transform(frame)
            weekday = pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d').dt.dayofweek
            groups = weekday if self.grouping == 'weekday' else weekday.ge(5).astype(int)
            logprod = np.log1p(frame['생산량'])
            signal = logprod if self.signal == 'logprod' else logprod.gt(0).where(logprod.notna()).astype(float)
            interactions = {f'state_hour_{hour}_group_{group}': signal * frame['시간'].eq(hour) * groups.eq(group) for hour in range(24) for group in range(7 if self.grouping == 'weekday' else 2)}
            return pd.concat([features, pd.DataFrame(interactions, index=frame.index)], axis=1)

    class StateInteractionModel:

        def __init__(self, config, seed=42):
            self.signal, self.grouping = (config.get('signal', 'on'), config.get('grouping', 'weekday'))
            assert self.signal in {'on', 'logprod'}
            assert self.grouping in {'weekday', 'weekend'}

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=StateInteractionFeatures(self.signal, self.grouping))
            self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            self.corrections = []
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                model = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr=MultiOutputRegressor(SVR(kernel=kernel, degree=2, coef0=1, C=10, epsilon=1, gamma='scale', tol=0.0001)))
                model.fit(inputs.loc[weekend], residual[weekend])
                self.corrections.append(model)
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            prediction = self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            if weekend.any():
                prediction[weekend] += 0.75 * self.corrections[0].predict(inputs.loc[weekend]) + 0.25 * self.corrections[1].predict(inputs.loc[weekend])
            return prediction

    # Source: thermal_state_combo_models.py
    """H27: original V054 intensity features plus original V070 state features."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.svm import SVR
    pass
    pass

    class ThermalStateComboFeatures(StateInteractionFeatures):

        def __init__(self):
            super().__init__(signal='on', grouping='weekend')

        def transform(self, frame):
            features = super().transform(frame)
            log_prod = np.log1p(frame['생산량'])
            features['temperature_intensity'] = frame['기온'] * log_prod
            features['cooling_intensity'] = (frame['기온'] - 20).clip(lower=0) * log_prod
            return features

    class ThermalStateComboModel(StateInteractionModel):

        def __init__(self, config, seed=42):
            assert config.get('mode', 'thermal_state') == 'thermal_state'
            super().__init__({'signal': 'on', 'grouping': 'weekend'}, seed)

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=ThermalStateComboFeatures())
            self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            self.corrections = []
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                model = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr=MultiOutputRegressor(SVR(kernel=kernel, degree=2, coef0=1, C=10, epsilon=1, gamma='scale', tol=0.0001)))
                model.fit(inputs.loc[weekend], residual[weekend])
                self.corrections.append(model)
            return self

    # Source: production_efficiency_models.py
    """H30: raw throughput response on V081, with train-only normalization."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.svm import SVR
    pass
    pass

    class ProductionEfficiencyFeatures(ThermalStateComboFeatures):

        def __init__(self, scale='global', replacement=False):
            super().__init__()
            self.scale, self.replacement = (scale, replacement)

        def fit(self, frame, y=None):
            super().fit(frame, y)
            positive = frame.loc[frame['생산량'].gt(0)].copy()
            positive['weekday'] = pd.to_datetime(positive['날짜'].astype(str), format='%Y%m%d').dt.dayofweek
            self.global_median = positive['생산량'].median()
            self.hour_medians = positive.groupby('시간')['생산량'].median()
            self.cell_medians = positive.groupby(['시간', 'weekday'])['생산량'].median()
            return self

        def transform(self, frame):
            features = super().transform(frame)
            denominator = np.full(len(frame), self.global_median)
            if self.scale == 'calendar':
                weekday = pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d').dt.dayofweek
                keys = pd.MultiIndex.from_arrays([frame['시간'], weekday], names=['시간', 'weekday'])
                denominator = self.cell_medians.reindex(keys).to_numpy()
                denominator = np.where(np.isnan(denominator), frame['시간'].map(self.hour_medians), denominator)
                denominator = np.where(np.isnan(denominator), self.global_median, denominator)
            if self.replacement:
                remove = [name for name in features if name.startswith('prod_') or name in {'log_prod', 'temperature_intensity', 'cooling_intensity'}]
                features = features.drop(columns=remove)
            throughput = frame['생산량'] / denominator
            features['throughput'] = throughput
            features['throughput_temperature'] = throughput * frame['기온']
            features['throughput_cooling'] = throughput * (frame['기온'] - 20).clip(lower=0)
            return features

    class ProductionEfficiencyModel:

        def __init__(self, config, seed=42):
            self.scale, self.mode = (config.get('scale', 'global'), config.get('mode', 'add'))
            assert self.scale in {'global', 'calendar'}
            assert self.mode in {'add', 'replace_log'}

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=ProductionEfficiencyFeatures(self.scale, self.mode == 'replace_log'))
            self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            self.corrections = []
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                model = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr=MultiOutputRegressor(SVR(kernel=kernel, degree=2, coef0=1, C=10, epsilon=1, gamma='scale', tol=0.0001)))
                self.corrections.append(model.fit(inputs.loc[weekend], residual[weekend]))
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            prediction = self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            if weekend.any():
                prediction[weekend] += 0.75 * self.corrections[0].predict(inputs.loc[weekend]) + 0.25 * self.corrections[1].predict(inputs.loc[weekend])
            return prediction

    # Source: combo_models.py
    """Combine original thermal, duplicate-weight, and zero-specialist settings."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from threadpoolctl import threadpool_limits
    pass
    pass
    pass

    class ComboModel(DuplicateModel):

        def __init__(self, config, seed=42):
            self.combination = config.get('mode', 'thermal_duplicate')
            assert self.combination in ('thermal_duplicate', 'thermal_zero', 'thermal_duplicate_zero')
            super().__init__({'mode': 'curve_sqrt'}, seed)

        def feature_transformer(self):
            return ThermalFeatures('intensity')

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            duplicate_weight = 1.0 if self.combination == 'thermal_zero' else self.day_weights(train)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0) * duplicate_weight
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=self.feature_transformer())
            weekend = is_weekend(inputs).to_numpy()
            self.residuals = []
            with threadpool_limits(limits=2):
                self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
                residual = train[POWER].to_numpy() - self.base.predict(inputs)
                for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                    correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                    correction.fit(inputs.loc[weekend], residual[weekend])
                    self.residuals.append(correction)
                self.specialist = None
                if self.combination.endswith('_zero'):
                    region = train.loc[train['생산량'].eq(0)]
                    dates = pd.to_datetime(region['날짜'].astype(str), format='%Y%m%d')
                    specialist_weight = np.exp2(-(dates.max() - dates).dt.days.to_numpy() / 90)
                    specialist_weight /= specialist_weight.mean()
                    self.specialist = clone(m242_pipeline).set_params(features=RHuberFeatures(trend=True))
                    self.specialist.fit(region[['날짜', '시간', '생산량', '기온']], region[POWER], huber__sample_weight=specialist_weight)
            return self

        def predict(self, frame):
            prediction = super().predict(frame)
            if self.specialist is not None:
                zero = frame['생산량'].eq(0).to_numpy()
                if zero.any():
                    prediction[zero] = self.specialist.predict(frame.loc[zero, ['날짜', '시간', '생산량', '기온']])
            return prediction

    # Source: thermal_balance_models.py
    """Balance sparse thermal support with one fixed inverse-square-root rule."""
    import numpy as np
    pass
    pass

    class ThermalBalanceModel(ComboModel):

        def __init__(self, config, seed=42):
            super().__init__({'mode': 'thermal_duplicate'}, seed)
            self.grouping = config.get('grouping', 'temperature')
            assert self.grouping in ('temperature', 'state_temperature')

        def feature_transformer(self):
            return ThermalStateComboFeatures()

        def day_weights(self, train):
            group = train['기온'].floordiv(5).astype('Int64').astype(str)
            if self.grouping == 'state_temperature':
                group = train['생산량'].eq(0).astype(str) + '_' + group
            self.thermal_counts_ = group.value_counts().to_dict()
            weight = 1.0 / np.sqrt(group.map(self.thermal_counts_))
            return (weight / weight.mean()).to_numpy()

    # Source: neighbor_models.py
    """H20: fixed-size local conditional medians of training targets or residuals."""
    import numpy as np
    from sklearn.base import clone
    from sklearn.neighbors import NearestNeighbors
    pass
    pass

    class NeighborModel:

        def __init__(self, config, seed=42):
            self.mode = config.get('mode', 'direct')
            self.representation = config.get('representation', 'compact')
            assert self.mode in {'direct', 'residual'}
            assert self.representation in {'compact', 'engineered'}

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            self.targets = train[POWER].to_numpy(copy=True)
            self.base = None
            if self.mode == 'residual':
                self.base = ThermalStateComboModel({'mode': 'thermal_state'}).fit(train)
                self.targets = self.targets - self.base.predict(inputs)
            template = n207_residual_pipeline if self.representation == 'compact' else m242_pipeline
            self.preprocess = clone(template[:-1])
            features = self.preprocess.fit_transform(inputs)
            self.neighbors = NearestNeighbors(n_neighbors=31, metric='manhattan', algorithm='brute', n_jobs=1)
            self.neighbors.fit(features)
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            features = self.preprocess.transform(inputs)
            indices = np.concatenate([self.neighbors.kneighbors(row[None, :], return_distance=False) for row in features])
            prediction = np.median(self.targets[indices], axis=1)
            if self.base is not None:
                prediction += self.base.predict(inputs)
            return prediction

    # Source: poisson_models.py
    """H58: one fixed log-link Poisson-deviance alternative on V081 inputs."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.linear_model import PoissonRegressor
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.svm import SVR
    pass
    pass

    class PoissonModel:

        def __init__(self, config, seed=42):
            assert config.get('mode', 'poisson') == 'poisson'

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            target = train[POWER].to_numpy()
            assert np.isfinite(target).all() and (target >= 0).all()
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=ThermalStateComboFeatures(), huber=MultiOutputRegressor(PoissonRegressor(alpha=1.0, max_iter=1000, tol=1e-07)))
            self.base.fit(inputs, target, huber__sample_weight=weight)
            residual = target - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            self.corrections = []
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                model = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr=MultiOutputRegressor(SVR(kernel=kernel, degree=2, coef0=1, C=10, epsilon=1, gamma='scale', tol=0.0001)))
                self.corrections.append(model.fit(inputs.loc[weekend], residual[weekend]))
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            prediction = self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            if weekend.any():
                prediction[weekend] += 0.75 * self.corrections[0].predict(inputs.loc[weekend]) + 0.25 * self.corrections[1].predict(inputs.loc[weekend])
            return prediction

    # Source: radial_huber_models.py
    """H57: direct four-output Huber with one robust influence per observed hour."""
    import warnings
    import numpy as np
    import pandas as pd
    from scipy.optimize import minimize
    from sklearn.base import BaseEstimator, RegressorMixin, clone
    from sklearn.exceptions import ConvergenceWarning
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.svm import SVR
    pass
    pass
    pass

    def radial_huber_objective(parameters, matrix, target, weight):
        coefficients = parameters[:-5].reshape(matrix.shape[1], 4)
        intercept, scale = (parameters[-5:-1], parameters[-1])
        residual = (matrix @ coefficients + intercept - target) / scale
        radius = np.sqrt(np.mean(residual ** 2, axis=1))
        influence = np.minimum(1.0, 1.05 / np.maximum(radius, np.finfo(float).tiny))
        rho = np.where(radius <= 1.05, radius ** 2, 2 * 1.05 * radius - 1.05 ** 2)
        loss = 4 * (weight @ (scale * (1 + rho))) + 1.5 * np.sum(coefficients ** 2)
        score = weight[:, None] * influence[:, None] * residual
        gradient = np.r_[(2 * matrix.T @ score + 3 * coefficients).ravel(), 2 * score.sum(axis=0), 4 * (weight @ (1 + rho - 2 * radius * np.minimum(radius, 1.05)))]
        return (float(loss), gradient)

    class RadialHuberRegressor(RegressorMixin, BaseEstimator):

        def fit(self, matrix, target, sample_weight=None):
            target = np.asarray(target)
            weight = np.ones(len(target)) if sample_weight is None else np.asarray(sample_weight)
            initial = np.zeros(matrix.shape[1] * 4 + 5)
            initial[-5:-1] = np.median(target, axis=0)
            initial[-1] = max(float(np.median(np.sqrt(np.mean((target - initial[-5:-1]) ** 2, axis=1)))), 1.0)
            bounds = [(None, None)] * (len(initial) - 1) + [(np.finfo(float).eps * 10, None)]
            result = minimize(radial_huber_objective, initial, args=(matrix, target, weight), method='L-BFGS-B', jac=True, bounds=bounds, options={'maxiter': 2000, 'gtol': 1e-07, 'ftol': 1e-10})
            if result.status == 2:
                raise ValueError('Radial Huber optimization failed: ' + str(result.message))
            if not result.success:
                warnings.warn(str(result.message), ConvergenceWarning)
            self.coef_ = result.x[:-5].reshape(matrix.shape[1], 4)
            self.intercept_, self.scale_ = (result.x[-5:-1], result.x[-1])
            self.training_summary_ = dict(success=bool(result.success), iterations=int(result.nit), message=str(result.message), objective=float(result.fun), scale=float(self.scale_))
            return self

        def predict(self, matrix):
            return matrix @ self.coef_ + self.intercept_

    class RadialHuberModel(StateInteractionModel):

        def __init__(self, config, seed=42):
            assert config.get('loss', 'radial') == 'radial'

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=ThermalStateComboFeatures(), huber=RadialHuberRegressor())
            self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            self.training_summary_ = self.base.named_steps['huber'].training_summary_
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            self.corrections = []
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr=MultiOutputRegressor(SVR(kernel=kernel, degree=2, coef0=1, C=10, epsilon=1, gamma='scale', tol=0.0001)))
                correction.fit(inputs.loc[weekend], residual[weekend])
                self.corrections.append(correction)
            return self

    # Source: tree_direct_models.py
    """H66: fixed direct LightGBM on V081 features and its equal-weight blend."""
    from importlib.metadata import version
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.multioutput import MultiOutputRegressor
    from threadpoolctl import threadpool_limits
    pass
    pass

    class TreeDirectModel:

        def __init__(self, config, seed=42):
            self.mode = config.get('mode', 'standalone')
            self.seed = seed
            assert self.mode in ('standalone', 'blend')

        def fit(self, train):
            from lightgbm import LGBMRegressor
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0)
            weight /= weight.mean()
            self.preprocess = clone(m242_pipeline[:-1]).set_params(features=ThermalStateComboFeatures())
            estimator = LGBMRegressor(objective='regression_l1', n_estimators=300, max_depth=5, num_leaves=15, min_child_samples=40, learning_rate=0.03, reg_lambda=10.0, random_state=self.seed, n_jobs=2, device_type='cpu', force_col_wise=True, verbosity=-1)
            with threadpool_limits(limits=2):
                matrix = self.preprocess.fit_transform(inputs)
                self.tree = MultiOutputRegressor(estimator, n_jobs=1)
                self.tree.fit(matrix, train[POWER], sample_weight=weight)
                self.base = ThermalStateComboModel({'mode': 'thermal_state'}, self.seed).fit(train) if self.mode == 'blend' else None
            self.training_summary_ = dict(lightgbm_version=version('lightgbm'), seed=self.seed, mode=self.mode, parameters=estimator.get_params())
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            prediction = self.tree.predict(self.preprocess.transform(inputs))
            return prediction if self.base is None else 0.5 * prediction + 0.5 * self.base.predict(inputs)

    # Source: phase2_quality_combo_models.py
    """Preserve V096 thermal support weights and V028 duplicate-curve weights."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.preprocessing import RobustScaler
    from threadpoolctl import threadpool_limits
    pass
    pass
    pass
    pass
    pass
    pass
    pass

    class Phase2QualityComboModel(ThermalBalanceModel):

        def __init__(self, config, seed=42):
            self.quality_mode = config.get('mode', 'state_temperature_duplicate')
            assert self.quality_mode in ('state_temperature_duplicate', 'raw_throughput', 'radial', 'features_r191', 'features_v054', 'features_v070', 'radial_v070', 'radial_v070_recent3', 'radial_v070_recent3_iqr')
            super().__init__({'grouping': 'state_temperature'}, seed)

        def feature_transformer(self):
            if self.quality_mode == 'raw_throughput':
                return ProductionEfficiencyFeatures(scale='global', replacement=False)
            if self.quality_mode == 'features_r191':
                return RHuberFeatures(trend=True)
            if self.quality_mode == 'features_v054':
                return ThermalFeatures(mode='intensity')
            if self.quality_mode in ('features_v070', 'radial_v070', 'radial_v070_recent3', 'radial_v070_recent3_iqr'):
                return StateInteractionFeatures(signal='on', grouping='weekend')
            return super().feature_transformer()

        def day_weights(self, train):
            return ThermalBalanceModel.day_weights(self, train) * DuplicateModel.day_weights(self, train)

        def fit(self, train):
            if self.quality_mode not in ('radial', 'radial_v070', 'radial_v070_recent3', 'radial_v070_recent3_iqr'):
                return super().fit(train)
            if self.quality_mode in ('radial_v070_recent3', 'radial_v070_recent3_iqr'):
                dates = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
                start = (dates.max().to_period('M') - 2).start_time
                train = train.loc[dates >= start]
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0) * self.day_weights(train)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=self.feature_transformer(), huber=RadialHuberRegressor())
            if self.quality_mode == 'radial_v070_recent3_iqr':
                self.base.set_params(preprocess__remainder__standardscaler=RobustScaler(quantile_range=(25.0, 75.0)))
            weekend = is_weekend(inputs).to_numpy()
            self.residuals, self.specialist = ([], None)
            with threadpool_limits(limits=2):
                self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
                residual = train[POWER].to_numpy() - self.base.predict(inputs)
                for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                    correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                    correction.fit(inputs.loc[weekend], residual[weekend])
                    self.residuals.append(correction)
            self.training_summary_ = self.base.named_steps['huber'].training_summary_
            return self

    # Source: phase2_window_models.py
    """V081 members with calendar training windows or genuinely uniform weights."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    pass
    pass
    pass

    class Phase2WindowModel(ThermalStateComboModel):

        def __init__(self, config, seed=42):
            super().__init__({'mode': 'thermal_state'}, seed)
            self.window = config['window']
            self.base_name, self.seed = (config.get('base', 'V081'), seed)
            assert self.base_name in ('V081', 'V217', 'V271', 'V239')
            assert self.window in ('recent1', 'recent2', 'recent3', 'unweighted', 'no_age')
            assert self.base_name != 'V217' or self.window in ('recent3', 'no_age')
            assert self.base_name not in ('V271', 'V239') or self.window == 'recent3'
            assert self.window != 'no_age' or self.base_name == 'V217'

        def fit(self, train):
            if self.window.startswith('recent'):
                months = int(self.window[-1])
                dates = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
                start = (dates.max().to_period('M') - (months - 1)).start_time
                train = train.loc[dates >= start]
                if self.base_name in ('V217', 'V271', 'V239'):
                    mode = {'V217': 'state_temperature_duplicate', 'V271': 'features_v070', 'V239': 'raw_throughput'}[self.base_name]
                    model = Phase2QualityComboModel({'mode': mode}, self.seed).fit(train)
                    self.base, self.corrections = (model.base, model.residuals)
                else:
                    super().fit(train)
            else:
                inputs = train[['날짜', '시간', '생산량', '기온']]
                weight = np.ones(len(train))
                if self.window == 'no_age':
                    quality = Phase2QualityComboModel({'mode': 'state_temperature_duplicate'}, self.seed)
                    weight = np.where(inputs['생산량'].eq(0), 0.5, 1.0) * quality.day_weights(train)
                    weight /= weight.mean()
                self.base = clone(m242_pipeline).set_params(features=ThermalStateComboFeatures())
                self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
                residual = train[POWER].to_numpy() - self.base.predict(inputs)
                weekend = is_weekend(inputs).to_numpy()
                self.corrections = []
                for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                    correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                    self.corrections.append(correction.fit(inputs.loc[weekend], residual[weekend]))
            row_weight = 'ones' if self.window == 'unweighted' else 'V217_thermal_duplicate_zero0.5_no_age' if self.window == 'no_age' else self.base_name + '_age90_zero0.5'
            self.training_summary_ = dict(window=self.window, base=self.base_name, rows=len(train), train_start=int(train['날짜'].min()), train_end=int(train['날짜'].max()), row_weight=row_weight)
            return self

    # Source: phase2_representation_models.py
    """V054/V070 Huber representations with zero or full RBF weekend correction."""
    pass
    pass
    pass

    class Phase2RepresentationModel:

        def __init__(self, config, seed=42):
            self.representation = config['representation']
            self.weekend_correction = config['weekend_correction']
            self.seed = seed
            assert self.representation in ('V054', 'V070')
            assert self.weekend_correction in ('none', 'rbf')

        def fit(self, train):
            original = ThermalModel({'mode': 'intensity'}, self.seed) if self.representation == 'V054' else StateInteractionModel({'signal': 'on', 'grouping': 'weekend'}, self.seed)
            original.fit(train)
            self.base = original.base
            corrections = original.residuals if self.representation == 'V054' else original.corrections
            self.correction = corrections[0] if self.weekend_correction == 'rbf' else None
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            prediction = self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            if self.correction is not None and weekend.any():
                prediction[weekend] += self.correction.predict(inputs.loc[weekend])
            return prediction

    # Source: phase2_loss_models.py
    """Conditional-median loss on V081 features with V081 or V217 weights."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.linear_model import QuantileRegressor
    from sklearn.multioutput import MultiOutputRegressor
    pass
    pass

    class Phase2MedianModel(Phase2QualityComboModel):

        def __init__(self, config, seed=42):
            self.loss_base = config.get('base', 'V081')
            assert self.loss_base in ('V081', 'V217')
            super().__init__({'mode': 'state_temperature_duplicate'}, seed)

        def day_weights(self, train):
            return np.ones(len(train)) if self.loss_base == 'V081' else super().day_weights(train)

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            dates = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(dates.max() - dates).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0) * self.day_weights(train)
            weight /= weight.mean()
            alpha = 1.5 / len(train)
            estimator = MultiOutputRegressor(QuantileRegressor(quantile=0.5, alpha=alpha, solver='highs'), n_jobs=1)
            self.base = clone(m242_pipeline).set_params(features=self.feature_transformer(), huber=estimator)
            self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            self.residuals, self.specialist = ([], None)
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                correction.fit(inputs.loc[weekend], residual[weekend])
                self.residuals.append(correction)
            self.training_summary_ = dict(base=self.loss_base, loss='median_pinball', alpha=alpha, summed_loss_l1_penalty=1.5, training_rows=len(train), weekend_correction='original 0.75 RBF + 0.25 polynomial SVR')
            return self

    # Source: phase2_scaling_models.py
    """Fixed IQR scaling of numeric V081 features; original Huber and SVR settings."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.preprocessing import RobustScaler
    pass
    pass

    class Phase2ScalingModel(Phase2QualityComboModel):

        def __init__(self, config, seed=42):
            self.scaling_base = config.get('base', 'V081')
            assert self.scaling_base in ('V081', 'V217', 'V239', 'V258', 'V271', 'V299', 'V321')
            mode = 'raw_throughput' if self.scaling_base in ('V239', 'V321') else 'features_v070' if self.scaling_base in ('V271', 'V299') else 'state_temperature_duplicate'
            super().__init__({'mode': mode}, seed)

        def day_weights(self, train):
            return np.ones(len(train)) if self.scaling_base == 'V081' else super().day_weights(train)

        def fit(self, train):
            if self.scaling_base in ('V258', 'V299', 'V321'):
                dates = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
                start = (dates.max().to_period('M') - 2).start_time
                train = train.loc[dates >= start]
            inputs = train[['날짜', '시간', '생산량', '기온']]
            dates = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(dates.max() - dates).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0) * self.day_weights(train)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=self.feature_transformer(), preprocess__remainder__standardscaler=RobustScaler(quantile_range=(25.0, 75.0)))
            self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            self.residuals, self.specialist = ([], None)
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                correction.fit(inputs.loc[weekend], residual[weekend])
                self.residuals.append(correction)
            self.training_summary_ = dict(base=self.scaling_base, scaler='RobustScaler', quantile_range=[25.0, 75.0], replaced_step='preprocess__remainder__standardscaler', other_preprocessing='unchanged categorical one-hot, imputer and weekend SVR pipelines')
            if self.scaling_base in ('V258', 'V299', 'V321'):
                self.training_summary_.update(window='recent3', rows=len(train), train_start=int(train['날짜'].min()), train_end=int(train['날짜'].max()))
            return self

    # Source: phase2_weighted_scaling_models.py
    """Use the Huber training weights for numeric feature means and variances too."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.preprocessing import StandardScaler
    pass
    pass
    pass

    class FoldWeightedStandardScaler(StandardScaler):

        def __init__(self, training_weight):
            self.training_weight = training_weight
            super().__init__()

        def fit(self, matrix, y=None, sample_weight=None):
            assert len(matrix) == len(self.training_weight)
            return super().fit(matrix, y, sample_weight=self.training_weight)

    class Phase2WeightedScalingModel(Phase2QualityComboModel):

        def __init__(self, config, seed=42):
            self.scaling_base = config['base']
            self.loss = config.get('loss', 'independent')
            assert self.scaling_base in ('V217', 'V271', 'V258', 'V299')
            assert self.loss in ('independent', 'radial')
            mode = 'features_v070' if self.scaling_base in ('V271', 'V299') else 'state_temperature_duplicate'
            super().__init__({'mode': mode}, seed)

        def fit(self, train):
            if self.scaling_base in ('V258', 'V299'):
                dates = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
                start = (dates.max().to_period('M') - 2).start_time
                train = train.loc[dates >= start]
            inputs = train[['날짜', '시간', '생산량', '기온']]
            dates = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(dates.max() - dates).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0) * self.day_weights(train)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=self.feature_transformer(), preprocess__remainder__standardscaler=FoldWeightedStandardScaler(weight))
            if self.loss == 'radial':
                self.base.set_params(huber=RadialHuberRegressor())
            self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            self.residuals, self.specialist = ([], None)
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                correction.fit(inputs.loc[weekend], residual[weekend])
                self.residuals.append(correction)
            self.training_summary_ = dict(base=self.scaling_base, numeric_scaler='StandardScaler fitted with exact Huber sample weights', categorical_and_imputation='unchanged and unweighted', weekend_correction='original unweighted 0.75 RBF + 0.25 polynomial SVR', training_rows=len(train), train_start=int(train['날짜'].min()), train_end=int(train['날짜'].max()))
            if self.loss == 'radial':
                self.training_summary_.update(loss='original radial Huber', solver=self.base.named_steps['huber'].training_summary_)
            return self

    # Source: phase2_strong_representation_models.py
    """Exact strong bases with no weekend SVR or the existing RBF SVR alone."""
    pass
    pass
    pass
    pass
    pass
    pass

    class Phase2StrongRepresentationModel(Phase2RepresentationModel):

        def __init__(self, config, seed=42):
            self.representation = config['representation']
            self.weekend_correction = config['weekend_correction']
            self.seed = seed
            assert self.representation in ('V081', 'V217', 'V239', 'V258', 'V266', 'V271', 'V299', 'V303', 'V317', 'V321', 'V351', 'V353')
            assert self.weekend_correction in ('none', 'rbf')
            assert self.representation in ('V081', 'V217') or self.weekend_correction == 'rbf'

        def fit(self, train):
            if self.representation == 'V081':
                original = ThermalStateComboModel({'mode': 'thermal_state'}, self.seed)
            elif self.representation in ('V258', 'V299', 'V321'):
                base = {'V258': 'V217', 'V299': 'V271', 'V321': 'V239'}[self.representation]
                original = Phase2WindowModel({'base': base, 'window': 'recent3'}, self.seed)
            elif self.representation in ('V266', 'V303', 'V317'):
                base = {'V266': 'V217', 'V303': 'V271', 'V317': 'V299'}[self.representation]
                original = Phase2ScalingModel({'base': base}, self.seed)
            elif self.representation in ('V351', 'V353'):
                base = 'V299' if self.representation == 'V353' else 'V271'
                original = Phase2WeightedScalingModel({'base': base}, self.seed)
            else:
                mode = 'raw_throughput' if self.representation == 'V239' else 'features_v070' if self.representation == 'V271' else 'state_temperature_duplicate'
                original = Phase2QualityComboModel({'mode': mode}, self.seed)
            original.fit(train)
            self.base = original.base
            corrections = original.corrections if self.representation in ('V081', 'V258', 'V299', 'V321') else original.residuals
            self.correction = corrections[0] if self.weekend_correction == 'rbf' else None
            self.training_summary_ = dict(representation=self.representation, weekend_correction=self.weekend_correction, base_fit='unchanged original features, Huber and sample weights')
            return self

    # Source: phase2_weekpart_models.py
    """Independent weekday/weekend Huber responses on V081 or V217 training weights."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    pass
    pass

    class Phase2WeekpartModel(Phase2QualityComboModel):

        def __init__(self, config, seed=42):
            self.weight_base = config.get('base', 'V081')
            assert self.weight_base in ('V081', 'V217')
            super().__init__({'mode': 'state_temperature_duplicate'}, seed)

        def day_weights(self, train):
            return np.ones(len(train)) if self.weight_base == 'V081' else super().day_weights(train)

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            dates = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(dates.max() - dates).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0) * self.day_weights(train)
            weight /= weight.mean()
            weekend = is_weekend(inputs).to_numpy()
            self.hubers = []
            for part in [False, True]:
                selected = weekend == part
                model = clone(m242_pipeline).set_params(features=self.feature_transformer())
                model.fit(inputs.loc[selected], train.loc[selected, POWER], huber__sample_weight=weight[selected])
                self.hubers.append(model)
            residual = train.loc[weekend, POWER].to_numpy() - self.hubers[1].predict(inputs.loc[weekend])
            self.residuals = []
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                correction.fit(inputs.loc[weekend], residual)
                self.residuals.append(correction)
            self.training_summary_ = dict(base=self.weight_base, weekday_rows=int((~weekend).sum()), weekend_rows=int(weekend.sum()), weight_scope='original full-fold weights sliced unchanged into weekday/weekend', weekend_correction='original 0.75 RBF + 0.25 polynomial SVR')
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            weekend = is_weekend(inputs).to_numpy()
            prediction = np.empty((len(inputs), 4))
            for part, model in enumerate(self.hubers):
                selected = weekend == bool(part)
                if selected.any():
                    prediction[selected] = model.predict(inputs.loc[selected])
            if weekend.any():
                left, right = (model.predict(inputs.loc[weekend]) for model in self.residuals)
                prediction[weekend] += 0.75 * left + 0.25 * right
            return prediction

    # Source: phase2_weighted_weekend_models.py
    """Apply V217 training-weight factors to its existing weekend SVRs."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    pass
    pass

    class Phase2WeightedWeekendModel(Phase2QualityComboModel):

        def __init__(self, config, seed=42):
            self.weekend_weight = config['weight_policy']
            assert self.weekend_weight in ('recency', 'quality', 'combined')
            super().__init__({'mode': 'state_temperature_duplicate'}, seed)

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            dates = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            recency = np.exp2(-(dates.max() - dates).dt.days.to_numpy() / 90)
            quality = self.day_weights(train)
            weight = recency.copy()
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0) * quality
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=self.feature_transformer())
            self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            weekend_weight = {'recency': recency, 'quality': quality, 'combined': weight}[self.weekend_weight][weekend].copy()
            weekend_weight /= weekend_weight.mean()
            self.residuals, self.specialist = ([], None)
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                correction.fit(inputs.loc[weekend], residual[weekend], svr__sample_weight=weekend_weight)
                self.residuals.append(correction)
            self.training_summary_ = dict(base='V217', weight_policy=self.weekend_weight, weight_normalization='mean one within training weekends', quality_support='all fold-training rows', weekend_rows=int(weekend.sum()), correction='original 0.75 RBF + 0.25 polynomial SVR')
            return self

    # Source: phase2_bootstrap_models.py
    """Median of five V217 fits with Poisson day/week weights on Huber training."""
    import numpy as np
    import pandas as pd
    pass

    class PoissonWeightedV217(Phase2QualityComboModel):

        def __init__(self, block, seed, member_index):
            super().__init__({'mode': 'state_temperature_duplicate'}, seed)
            self.block, self.seed, self.member_index = (block, seed, member_index)

        def day_weights(self, train):
            original = super().day_weights(train)
            dates = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
            if self.block == 'day':
                labels = train['날짜'].astype(str)
            else:
                calendar = dates.dt.isocalendar()
                labels = calendar['year'].astype(str) + '-' + calendar['week'].astype(str)
            block_ids = np.sort(labels.unique())
            rng = np.random.default_rng(np.random.SeedSequence([self.seed, self.member_index]))
            counts = rng.poisson(1, len(block_ids))
            assert counts.sum() > 0
            self.block_counts_ = dict(zip(block_ids, counts.tolist()))
            return original * labels.map(self.block_counts_).to_numpy()

    class Phase2BootstrapModel:

        def __init__(self, config, seed=42):
            self.block, self.seed = (config.get('block', 'day'), seed)
            assert self.block in ('day', 'week')

        def fit(self, train):
            self.members = [PoissonWeightedV217(self.block, self.seed, index).fit(train) for index in range(5)]
            self.training_summary_ = dict(base='V217', block=self.block, members=5, seed=self.seed, seed_rule='SeedSequence([seed, member_index])', aggregation='median', block_counts=[model.block_counts_ for model in self.members], weight_scope='Poisson counts multiply original full-fold V217 Huber weights', weekend_correction='original unweighted SVRs on each member Huber residuals')
            return self

        def predict(self, frame):
            return np.median(np.stack([model.predict(frame) for model in self.members]), axis=0)

    # Source: phase2_linear_svr_models.py
    """Fixed L2-regularized absolute-loss linear members on V271 features."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.preprocessing import RobustScaler
    from sklearn.svm import LinearSVR
    pass
    pass

    class Phase2LinearSVRModel(Phase2QualityComboModel):

        def __init__(self, config, seed=42):
            super().__init__({'mode': 'features_v070'}, seed)
            self.window = config.get('window', 'full')
            self.scaler = config.get('scaler', 'standard')
            self.seed = seed
            assert self.window in ('full', 'recent3')
            assert self.scaler in ('standard', 'iqr')

        def fit(self, train):
            if self.window == 'recent3':
                dates = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
                start = (dates.max().to_period('M') - 2).start_time
                train = train.loc[dates >= start]
            inputs = train[['날짜', '시간', '생산량', '기온']]
            dates = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(dates.max() - dates).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0) * self.day_weights(train)
            weight /= weight.mean()
            estimator = MultiOutputRegressor(LinearSVR(C=1.0, epsilon=0.0, loss='epsilon_insensitive', dual=True, tol=0.0001, max_iter=20000, random_state=self.seed), n_jobs=1)
            self.base = clone(m242_pipeline).set_params(features=self.feature_transformer(), huber=estimator)
            if self.scaler == 'iqr':
                self.base.set_params(preprocess__remainder__standardscaler=RobustScaler(quantile_range=(25.0, 75.0)))
            self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            self.residuals, self.specialist = ([], None)
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                correction.fit(inputs.loc[weekend], residual[weekend])
                self.residuals.append(correction)
            iterations = [int(model.n_iter_) for model in self.base.named_steps['huber'].estimators_]
            self.training_summary_ = dict(base='V271', window=self.window, scaler=self.scaler, C=1.0, epsilon=0.0, loss='epsilon_insensitive', penalty='L2', max_iter=20000, seed=self.seed, seed_confirmation_required=True, n_iter=iterations, reached_iteration_limit=any((count >= 20000 for count in iterations)), training_rows=len(train), train_start=int(train['날짜'].min()), train_end=int(train['날짜'].max()), weekend_correction='original 0.75 RBF + 0.25 polynomial SVR')
            return self

    # Source: phase2_linear_tree_models.py
    """Piecewise-linear Huber boosting replaces the strong member's base learner.

    LightGBM 4.6.0 permits Huber with serial linear trees and excludes regression_l1:
    https://lightgbm.readthedocs.io/en/v4.6.0/Parameters.html#linear_tree
    """
    from importlib.metadata import version
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.compose import TransformedTargetRegressor
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.preprocessing import StandardScaler
    pass
    pass

    class Phase2LinearTreeModel(Phase2QualityComboModel):

        def __init__(self, config, seed=42):
            self.base_name, self.history = (config['base'], config['history'])
            self.seed = seed
            assert self.base_name in ('V217', 'V271')
            assert self.history in ('full', 'recent3')
            mode = 'features_v070' if self.base_name == 'V271' else 'state_temperature_duplicate'
            super().__init__({'mode': mode}, seed)

        def fit(self, train):
            from lightgbm import LGBMRegressor
            if self.history == 'recent3':
                dates = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
                train = train.loc[dates >= (dates.max().to_period('M') - 2).start_time]
            inputs = train[['날짜', '시간', '생산량', '기온']]
            dates = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(dates.max() - dates).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0) * self.day_weights(train)
            weight /= weight.mean()
            parameters = dict(objective='huber', alpha=0.9, n_estimators=100, num_leaves=4, min_child_samples=100, learning_rate=0.05, linear_tree=True, linear_lambda=10.0, reg_lambda=10.0, tree_learner='serial', device_type='cpu', n_jobs=2, random_state=self.seed, deterministic=True, force_col_wise=True, zero_as_missing=False, verbosity=-1)
            estimator = TransformedTargetRegressor(regressor=MultiOutputRegressor(LGBMRegressor(**parameters), n_jobs=1), transformer=StandardScaler())
            self.base = clone(m242_pipeline).set_params(features=self.feature_transformer(), huber=estimator)
            self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            self.residuals, self.specialist = ([], None)
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                correction.fit(inputs.loc[weekend], residual[weekend])
                self.residuals.append(correction)
            target_scaler = self.base.named_steps['huber'].transformer_
            self.training_summary_ = dict(base=self.base_name, history=self.history, rows=len(train), train_start=int(train['날짜'].min()), train_end=int(train['날짜'].max()), lightgbm_version=version('lightgbm'), parameters=parameters, target_mean=target_scaler.mean_.tolist(), target_scale=target_scaler.scale_.tolist(), weekend_correction='original 0.75 RBF + 0.25 polynomial SVR')
            return self

    # Source: phase2_quality_support_models.py
    """Preserve state/calendar mass while retaining within-group quality weights."""
    import numpy as np
    import pandas as pd
    pass

    class Phase2QualitySupportModel(Phase2QualityComboModel):

        def __init__(self, config, seed=42):
            super().__init__({'mode': 'features_v070'}, seed)
            self.balance = config['balance']
            self.window = config.get('window', 'full')
            assert self.balance in ('state', 'weekpart')
            assert self.window in ('full', 'recent3')

        def day_weights(self, train):
            quality = super().day_weights(train)
            dates = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
            original = np.exp2(-(dates.max() - dates).dt.days.to_numpy() / 90)
            original *= np.where(train['생산량'].eq(0), 0.5, 1.0)
            group = train['생산량'].gt(0) if self.balance == 'state' else dates.dt.dayofweek.ge(5)
            original = pd.Series(original, index=train.index)
            weighted = original * quality
            normalizer = weighted.groupby(group).sum() / original.groupby(group).sum()
            adjusted = quality / group.map(normalizer).to_numpy()
            self.weight_audit_ = dict(balance=self.balance, group_normalizers={str(k): float(v) for k, v in normalizer.items()}, original_group_mass={str(k): float(v) for k, v in original.groupby(group).sum().items()}, adjusted_group_mass={str(k): float(v) for k, v in (original * adjusted).groupby(group).sum().items()})
            return adjusted

        def fit(self, train):
            if self.window == 'recent3':
                dates = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
                train = train.loc[dates >= (dates.max().to_period('M') - 2).start_time]
            super().fit(train)
            self.training_summary_ = dict(base='V271', window=self.window, train_start=int(train['날짜'].min()), train_end=int(train['날짜'].max()), rows=len(train), **self.weight_audit_)
            return self

    # Source: phase2_distinct_support_models.py
    """V271 thermal support counts distinct daily-curve mass, not repeat count."""
    import numpy as np
    import pandas as pd
    pass
    pass

    class Phase2DistinctSupportModel(Phase2QualityComboModel):

        def __init__(self, config, seed=42):
            super().__init__({'mode': 'features_v070'}, seed)
            self.support_grouping = config.get('grouping', 'state_temperature')
            assert self.support_grouping in ('state_temperature', 'state_temperature_weekpart', 'state_temperature_daytime', 'temperature_production')

        def day_weights(self, train):
            duplicate = DuplicateModel.day_weights(self, train)
            temperature = train['기온'].floordiv(5).astype('Int64').astype(str)
            group = train['생산량'].eq(0).astype(str) + '_' + temperature
            summary = dict(base='V271', grouping=self.support_grouping, support_mass='sum of inverse exact-curve multiplicity', per_row_duplicate_weight='original inverse square root multiplicity', temperature_width=5, support_scope='current training rows only')
            if self.support_grouping == 'state_temperature_weekpart':
                date = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
                group = group + '_' + date.dt.dayofweek.ge(5).astype(str)
            elif self.support_grouping == 'state_temperature_daytime':
                group = group + '_' + train['시간'].between(8, 18).astype(str)
                summary['daytime_hours'] = [8, 18]
            elif self.support_grouping == 'temperature_production':
                production = train['생산량']
                self.positive_median_ = float(production.loc[production > 0].median())
                tier = pd.Series(np.where(production <= 0, 0, np.where(production <= self.positive_median_, 1, 2)), index=train.index)
                group = temperature + '_' + tier.astype(str)
                summary['positive_production_median'] = self.positive_median_
            support = pd.Series(duplicate ** 2, index=train.index).groupby(group).sum()
            thermal = 1.0 / np.sqrt(group.map(support).to_numpy(dtype=float))
            thermal /= thermal.mean()
            self.training_summary_ = dict(summary, group_support=support.to_dict())
            return thermal * duplicate

    # Source: phase2_production_representation_models.py
    """Two fixed production-response assumptions on the strong V271 base."""
    import numpy as np
    import pandas as pd
    pass
    pass

    class Phase2ProductionFeatures(StateInteractionFeatures):

        def __init__(self, representation):
            super().__init__(signal='on', grouping='weekend')
            self.representation = representation

        def transform(self, frame):
            features = super().transform(frame)
            hinge_columns = [f'prod_hinge_hour{hour}' for hour in range(24)]
            if self.representation == 'no_hinge':
                return features.drop(columns=hinge_columns)
            production = frame['생산량'].to_numpy(dtype=float)
            original = np.log1p(production)
            transformed = np.arcsinh(production / 100.0)
            ratio = np.divide(transformed, original, out=np.zeros(len(frame)), where=original > 0)
            columns = ['log_prod', 'prod_sin', 'prod_cos']
            columns += [f'prod_day{day}' for day in range(7)]
            columns += [f'prod_hour{hour}' for hour in range(24)]
            features[columns] = features[columns].mul(ratio, axis=0)
            original_hinge = np.maximum(original - np.log1p(100.0), 0.0)
            transformed_hinge = np.maximum(transformed - np.arcsinh(1.0), 0.0)
            hinge_ratio = np.divide(transformed_hinge, original_hinge, out=np.zeros(len(frame)), where=original_hinge > 0)
            features[hinge_columns] = features[hinge_columns].mul(hinge_ratio, axis=0)
            return features

    class Phase2ProductionRepresentationModel(Phase2QualityComboModel):

        def __init__(self, config, seed=42):
            self.representation, self.history = (config['representation'], config['history'])
            assert self.representation in ('no_hinge', 'asinh100')
            assert self.history in ('full', 'recent3')
            super().__init__({'mode': 'features_v070'}, seed)

        def feature_transformer(self):
            return Phase2ProductionFeatures(self.representation)

        def fit(self, train):
            if self.history == 'recent3':
                dates = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
                train = train.loc[dates >= (dates.max().to_period('M') - 2).start_time]
            super().fit(train)
            self.training_summary_ = dict(base='V271', representation=self.representation, history=self.history, rows=len(train), train_start=int(train['날짜'].min()), train_end=int(train['날짜'].max()), production_reference=100.0, other_settings='original V271 quality weights, Huber and weekend SVRs')
            return self

    # Source: phase2_clock_pool_models.py
    """Pool response gates across clock hours without changing calendar baseload."""
    import numpy as np
    import pandas as pd
    pass
    pass

    class ClockPoolFeatures(StateInteractionFeatures):

        def __init__(self, blocks='state'):
            super().__init__(signal='on', grouping='weekend')
            self.blocks = blocks

        def transform(self, frame):
            features = super().transform(frame)
            hour = frame['시간'].to_numpy()
            responses = {}
            if self.blocks in ('state', 'state_hinge'):
                features = features.drop(columns=[name for name in features if name.startswith('state_hour_')])
                weekday = pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d').dt.dayofweek
                for group in (0, 1):
                    responses[f'clock_state_{group}'] = features['on'] * weekday.ge(5).eq(bool(group))
            if self.blocks in ('hinge', 'state_hinge'):
                features = features.drop(columns=[name for name in features if name.startswith('prod_hinge_hour')])
                responses['clock_hinge'] = (np.log1p(frame['생산량']) - np.log1p(100)).clip(lower=0)
            for name, response in responses.items():
                features[name] = response
                for harmonic in (1, 2):
                    angle = 2 * np.pi * harmonic * hour / 24
                    features[f'{name}_sin{harmonic}'] = response * np.sin(angle)
                    features[f'{name}_cos{harmonic}'] = response * np.cos(angle)
            return features

    class Phase2ClockPoolModel(Phase2QualityComboModel):

        def __init__(self, config, seed=42):
            super().__init__({'mode': 'features_v070'}, seed)
            self.clock_blocks = config.get('blocks', 'state')
            assert self.clock_blocks in ('state', 'hinge', 'state_hinge')
            self.training_summary_ = dict(base='V271', pooled_blocks=self.clock_blocks, clock_harmonics=[1, 2], main_hour_weekday_category='unchanged', quality_weights='original V271', weekend_correction='original V271')

        def feature_transformer(self):
            return ClockPoolFeatures(self.clock_blocks)

    # Source: phase2_geometry_combo_models.py
    """Apply V353 weighted feature geometry to four preserved recent-three designs."""
    pass
    pass
    pass
    pass

    class Phase2GeometryComboModel(Phase2WeightedScalingModel):

        def __init__(self, config, seed=42):
            super().__init__({'base': 'V299', 'loss': config.get('loss', 'independent')}, seed)
            self.variant = config['variant']
            self.weekend_correction = config.get('weekend_correction', 'original')
            self.seed = seed
            assert self.variant in ('no_hinge', 'asinh100', 'state_neutral', 'weekpart_neutral')
            assert self.weekend_correction in ('original', 'rbf')

        def feature_transformer(self):
            if self.variant in ('no_hinge', 'asinh100'):
                return Phase2ProductionFeatures(self.variant)
            return super().feature_transformer()

        def day_weights(self, train):
            if self.variant.endswith('_neutral'):
                model = Phase2QualitySupportModel({'balance': self.variant.removesuffix('_neutral'), 'window': 'recent3'}, self.seed)
                weights = model.day_weights(train)
                self.weight_audit_ = model.weight_audit_
                return weights
            return super().day_weights(train)

        def fit(self, train):
            super().fit(train)
            self.training_summary_['structural_combination'] = self.variant
            if self.variant.endswith('_neutral'):
                self.training_summary_['quality_normalization'] = self.weight_audit_
            if self.weekend_correction == 'rbf':
                self.training_summary_['weekend_correction'] = 'original RBF SVR alone'
            return self

        def predict(self, frame):
            if self.weekend_correction == 'original':
                return super().predict(frame)
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            prediction = self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            if weekend.any():
                prediction[weekend] += self.residuals[0].predict(inputs.loc[weekend])
            return prediction

    # Source: phase2_categorical_models.py
    """Symmetric calendar regularization on the fixed strong V353 pipeline."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import OneHotEncoder
    pass
    pass
    pass

    class Phase2CategoricalModel(Phase2QualityComboModel):

        def __init__(self, config, seed=42):
            super().__init__({'mode': 'features_v070'}, seed)
            self.coding = config.get('coding', 'symmetric')
            assert self.coding in ('symmetric', 'weighted_variance')

        def fit(self, train):
            dates = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
            start = (dates.max().to_period('M') - 2).start_time
            train = train.loc[dates >= start]
            inputs = train[['날짜', '시간', '생산량', '기온']]
            dates = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(dates.max() - dates).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0) * self.day_weights(train)
            weight /= weight.mean()
            category = OneHotEncoder(drop=None, handle_unknown='ignore', sparse_output=False)
            if self.coding == 'weighted_variance':
                category = make_pipeline(category, FoldWeightedStandardScaler(weight))
            self.base = clone(m242_pipeline).set_params(features=self.feature_transformer(), preprocess__category=category, preprocess__remainder__standardscaler=FoldWeightedStandardScaler(weight))
            self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            self.residuals, self.specialist = ([], None)
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                correction.fit(inputs.loc[weekend], residual[weekend])
                self.residuals.append(correction)
            self.training_summary_ = dict(base='V353', coding=self.coding, window='recent3', category='all hour-weekday indicators; no privileged omitted category', category_scaling='weighted StandardScaler' if self.coding == 'weighted_variance' else 'none', numeric_scaling='original V353 weighted StandardScaler', training_rows=len(train), train_start=int(train['날짜'].min()), train_end=int(train['날짜'].max()), weekend_correction='original 0.75 RBF + 0.25 polynomial SVR')
            return self

    # Source: phase2_weekend_scaling_models.py
    """Change only the numeric geometry inside the existing two weekend SVRs."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.preprocessing import RobustScaler
    pass
    pass
    pass

    class Phase2WeekendScalingModel(Phase2WeightedScalingModel):

        def __init__(self, config, seed=42):
            self.member, self.weekend_scaler = (config['base'], config['weekend_scaler'])
            self.seed = seed
            assert self.member in ('V351', 'V353', 'V365', 'V366', 'V367', 'V368')
            assert self.weekend_scaler in ('weighted', 'iqr')
            assert self.member in ('V351', 'V353') or self.weekend_scaler == 'iqr'
            super().__init__({'base': 'V271' if self.member == 'V351' else 'V299'}, seed)

        def fit(self, train):
            if self.member != 'V351':
                dates = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
                train = train.loc[dates >= (dates.max().to_period('M') - 2).start_time]
            if self.member in ('V365', 'V366', 'V367', 'V368'):
                variant = {'V365': 'no_hinge', 'V366': 'asinh100', 'V367': 'state_neutral', 'V368': 'weekpart_neutral'}[self.member]
                model = Phase2GeometryComboModel({'variant': variant}, self.seed).fit(train)
                self.base, self.specialist = (model.base, None)
                self.training_summary_ = dict(model.training_summary_)
            else:
                super().fit(train)
            inputs = train[['날짜', '시간', '생산량', '기온']]
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            if self.weekend_scaler == 'weighted':
                dates = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
                weight = np.exp2(-(dates.max() - dates).dt.days.to_numpy() / 90)
                weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0) * self.day_weights(train)
                weight = weight[weekend]
                weight /= weight.mean()
                scaler = FoldWeightedStandardScaler(weight)
            else:
                scaler = RobustScaler(quantile_range=(25.0, 75.0))
            self.residuals = []
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, preprocess__remainder__standardscaler=clone(scaler), svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                correction.fit(inputs.loc[weekend], residual[weekend])
                self.residuals.append(correction)
            self.training_summary_.update(member=self.member, weekend_scaler=self.weekend_scaler, weekend_svr_loss='unchanged unweighted', replaced_step='weekend SVRs preprocess__remainder__standardscaler')
            return self

    # Source: phase2_joint_geometry_models.py
    """Combine preserved production features and quality-mass normalization."""
    pass
    pass

    class Phase2JointGeometryModel(Phase2GeometryComboModel):

        def __init__(self, config, seed=42):
            features = config['features_variant']
            assert features in ('no_hinge', 'asinh100')
            super().__init__({'variant': features}, seed)
            self.quality_balance = config['quality_balance']
            assert self.quality_balance in ('state', 'weekpart')

        def day_weights(self, train):
            model = Phase2QualitySupportModel({'balance': self.quality_balance, 'window': 'recent3'}, self.seed)
            weights = model.day_weights(train)
            self.weight_audit_ = model.weight_audit_
            return weights

        def fit(self, train):
            super().fit(train)
            self.training_summary_.update(features_variant=self.variant, quality_balance=self.quality_balance, quality_normalization=self.weight_audit_)
            return self

    # Source: phase2_catboost_models.py
    """Two fixed CatBoost losses provide new error patterns for phase-two pooling."""
    from importlib.metadata import version
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.multioutput import MultiOutputRegressor
    pass
    pass

    class Phase2CatBoostModel:

        def __init__(self, config, seed=42):
            self.mode = config.get('mode', 'joint')
            self.seed = seed
            assert self.mode in ('joint', 'mae')

        def fit(self, train):
            from catboost import CatBoostRegressor
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0)
            weight /= weight.mean()
            self.preprocess = clone(m242_pipeline[:-1]).set_params(features=ThermalStateComboFeatures())
            matrix = self.preprocess.fit_transform(inputs)
            parameters = dict(iterations=500, depth=4, learning_rate=0.04, l2_leaf_reg=10.0, random_seed=self.seed, thread_count=2, task_type='CPU', allow_writing_files=False, verbose=False, loss_function='MultiRMSE' if self.mode == 'joint' else 'MAE')
            estimator = CatBoostRegressor(**parameters)
            self.tree = estimator if self.mode == 'joint' else MultiOutputRegressor(estimator, n_jobs=1)
            self.tree.fit(matrix, train[POWER], sample_weight=weight)
            self.training_summary_ = dict(catboost_version=version('catboost'), seed=self.seed, mode=self.mode, parameters=parameters)
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            return self.tree.predict(self.preprocess.transform(inputs))

    # Source: phase2_components.py
    """Frozen phase-one components reused by forward-chain phase-two ensembles."""
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    COMPONENTS = {'V019': (ObservedStateModel, {'kind': 'observed_state', 'region': 'zero'}), 'V028': (DuplicateModel, {'kind': 'duplicate', 'mode': 'curve_sqrt'}), 'V054': (ThermalModel, {'kind': 'thermal', 'mode': 'intensity'}), 'V070': (StateInteractionModel, {'kind': 'state_interaction', 'signal': 'on', 'grouping': 'weekend'}), 'V081': (ThermalStateComboModel, {'kind': 'thermal_state_combo', 'mode': 'thermal_state'}), 'V089': (ProductionEfficiencyModel, {'kind': 'production_efficiency', 'scale': 'global', 'mode': 'add'}), 'V096': (ThermalBalanceModel, {'kind': 'thermal_balance', 'grouping': 'state_temperature', 'base': 'V081'}), 'V102': (NeighborModel, {'kind': 'neighbor', 'mode': 'direct', 'representation': 'engineered'}), 'V154': (PoissonModel, {'kind': 'poisson', 'mode': 'poisson'}), 'V159': (RadialHuberModel, {'kind': 'radial_huber', 'loss': 'radial'}), 'V174': (TreeDirectModel, {'kind': 'tree_direct', 'mode': 'standalone'}), 'V217': (Phase2QualityComboModel, {'kind': 'phase2_quality_combo', 'mode': 'state_temperature_duplicate'}), 'V239': (Phase2QualityComboModel, {'kind': 'phase2_quality_combo', 'mode': 'raw_throughput'}), 'V240': (Phase2QualityComboModel, {'kind': 'phase2_quality_combo', 'mode': 'radial'}), 'V241': (Phase2WindowModel, {'kind': 'phase2_window', 'window': 'recent1'}), 'V242': (Phase2WindowModel, {'kind': 'phase2_window', 'window': 'recent2'}), 'V243': (Phase2WindowModel, {'kind': 'phase2_window', 'window': 'recent3'}), 'V244': (Phase2WindowModel, {'kind': 'phase2_window', 'window': 'unweighted'}), 'V245': (Phase2RepresentationModel, {'kind': 'phase2_representation', 'representation': 'V054', 'weekend_correction': 'none'}), 'V246': (Phase2RepresentationModel, {'kind': 'phase2_representation', 'representation': 'V054', 'weekend_correction': 'rbf'}), 'V247': (Phase2RepresentationModel, {'kind': 'phase2_representation', 'representation': 'V070', 'weekend_correction': 'none'}), 'V248': (Phase2RepresentationModel, {'kind': 'phase2_representation', 'representation': 'V070', 'weekend_correction': 'rbf'}), 'V249': (Phase2MedianModel, {'kind': 'phase2_median_loss', 'base': 'V081'}), 'V250': (Phase2MedianModel, {'kind': 'phase2_median_loss', 'base': 'V217'}), 'V258': (Phase2WindowModel, {'kind': 'phase2_window', 'base': 'V217', 'window': 'recent3'}), 'V259': (Phase2WindowModel, {'kind': 'phase2_window', 'base': 'V217', 'window': 'no_age'}), 'V260': (Phase2StrongRepresentationModel, {'kind': 'phase2_strong_representation', 'representation': 'V081', 'weekend_correction': 'none'}), 'V261': (Phase2StrongRepresentationModel, {'kind': 'phase2_strong_representation', 'representation': 'V081', 'weekend_correction': 'rbf'}), 'V262': (Phase2StrongRepresentationModel, {'kind': 'phase2_strong_representation', 'representation': 'V217', 'weekend_correction': 'none'}), 'V263': (Phase2StrongRepresentationModel, {'kind': 'phase2_strong_representation', 'representation': 'V217', 'weekend_correction': 'rbf'}), 'V265': (Phase2ScalingModel, {'kind': 'phase2_scaling', 'base': 'V081'}), 'V266': (Phase2ScalingModel, {'kind': 'phase2_scaling', 'base': 'V217'}), 'V267': (Phase2WeekpartModel, {'kind': 'phase2_weekpart', 'base': 'V081'}), 'V268': (Phase2WeekpartModel, {'kind': 'phase2_weekpart', 'base': 'V217'}), 'V269': (Phase2QualityComboModel, {'kind': 'phase2_quality_combo', 'mode': 'features_r191'}), 'V270': (Phase2QualityComboModel, {'kind': 'phase2_quality_combo', 'mode': 'features_v054'}), 'V271': (Phase2QualityComboModel, {'kind': 'phase2_quality_combo', 'mode': 'features_v070'}), 'V277': (Phase2WeightedWeekendModel, {'kind': 'phase2_weighted_weekend', 'weight_policy': 'recency'}), 'V278': (Phase2WeightedWeekendModel, {'kind': 'phase2_weighted_weekend', 'weight_policy': 'quality'}), 'V279': (Phase2WeightedWeekendModel, {'kind': 'phase2_weighted_weekend', 'weight_policy': 'combined'}), 'V286': (Phase2BootstrapModel, {'kind': 'phase2_bootstrap', 'block': 'day', 'seed_confirmation_required': True}), 'V287': (Phase2BootstrapModel, {'kind': 'phase2_bootstrap', 'block': 'week', 'seed_confirmation_required': True}), 'V299': (Phase2WindowModel, {'kind': 'phase2_window', 'base': 'V271', 'window': 'recent3'}), 'V300': (Phase2StrongRepresentationModel, {'kind': 'phase2_strong_representation', 'representation': 'V258', 'weekend_correction': 'rbf'}), 'V301': (Phase2StrongRepresentationModel, {'kind': 'phase2_strong_representation', 'representation': 'V271', 'weekend_correction': 'rbf'}), 'V302': (Phase2ScalingModel, {'kind': 'phase2_scaling', 'base': 'V258'}), 'V303': (Phase2ScalingModel, {'kind': 'phase2_scaling', 'base': 'V271'}), 'V304': (Phase2StrongRepresentationModel, {'kind': 'phase2_strong_representation', 'representation': 'V266', 'weekend_correction': 'rbf'}), 'V317': (Phase2ScalingModel, {'kind': 'phase2_scaling', 'base': 'V299'}), 'V318': (Phase2StrongRepresentationModel, {'kind': 'phase2_strong_representation', 'representation': 'V299', 'weekend_correction': 'rbf'}), 'V319': (Phase2QualityComboModel, {'kind': 'phase2_quality_combo', 'mode': 'radial_v070'}), 'V320': (Phase2QualityComboModel, {'kind': 'phase2_quality_combo', 'mode': 'radial_v070_recent3'}), 'V321': (Phase2WindowModel, {'kind': 'phase2_window', 'base': 'V239', 'window': 'recent3'}), 'V322': (Phase2ScalingModel, {'kind': 'phase2_scaling', 'base': 'V239'}), 'V323': (Phase2StrongRepresentationModel, {'kind': 'phase2_strong_representation', 'representation': 'V239', 'weekend_correction': 'rbf'}), 'V324': (Phase2StrongRepresentationModel, {'kind': 'phase2_strong_representation', 'representation': 'V303', 'weekend_correction': 'rbf'}), 'V325': (Phase2LinearSVRModel, {'kind': 'phase2_linear_svr', 'window': 'full', 'scaler': 'standard', 'seed_confirmation_required': True}), 'V326': (Phase2LinearSVRModel, {'kind': 'phase2_linear_svr', 'window': 'full', 'scaler': 'iqr', 'seed_confirmation_required': True}), 'V327': (Phase2LinearSVRModel, {'kind': 'phase2_linear_svr', 'window': 'recent3', 'scaler': 'standard', 'seed_confirmation_required': True}), 'V328': (Phase2LinearSVRModel, {'kind': 'phase2_linear_svr', 'window': 'recent3', 'scaler': 'iqr', 'seed_confirmation_required': True}), 'V330': (Phase2LinearTreeModel, {'kind': 'phase2_linear_tree', 'base': 'V217', 'history': 'full', 'seed_confirmation_required': True}), 'V331': (Phase2LinearTreeModel, {'kind': 'phase2_linear_tree', 'base': 'V217', 'history': 'recent3', 'seed_confirmation_required': True}), 'V332': (Phase2LinearTreeModel, {'kind': 'phase2_linear_tree', 'base': 'V271', 'history': 'full', 'seed_confirmation_required': True}), 'V333': (Phase2LinearTreeModel, {'kind': 'phase2_linear_tree', 'base': 'V271', 'history': 'recent3', 'seed_confirmation_required': True}), 'V334': (Phase2StrongRepresentationModel, {'kind': 'phase2_strong_representation', 'representation': 'V317', 'weekend_correction': 'rbf'}), 'V335': (Phase2ScalingModel, {'kind': 'phase2_scaling', 'base': 'V321'}), 'V336': (Phase2StrongRepresentationModel, {'kind': 'phase2_strong_representation', 'representation': 'V321', 'weekend_correction': 'rbf'}), 'V337': (Phase2QualityComboModel, {'kind': 'phase2_quality_combo', 'mode': 'radial_v070_recent3_iqr'}), 'V338': (Phase2DistinctSupportModel, {'kind': 'phase2_distinct_support', 'grouping': 'state_temperature'}), 'V339': (Phase2DistinctSupportModel, {'kind': 'phase2_distinct_support', 'grouping': 'state_temperature_weekpart'}), 'V340': (Phase2DistinctSupportModel, {'kind': 'phase2_distinct_support', 'grouping': 'state_temperature_daytime'}), 'V341': (Phase2DistinctSupportModel, {'kind': 'phase2_distinct_support', 'grouping': 'temperature_production'}), 'V342': (Phase2ProductionRepresentationModel, {'kind': 'phase2_production_representation', 'representation': 'no_hinge', 'history': 'full'}), 'V343': (Phase2ProductionRepresentationModel, {'kind': 'phase2_production_representation', 'representation': 'no_hinge', 'history': 'recent3'}), 'V344': (Phase2ProductionRepresentationModel, {'kind': 'phase2_production_representation', 'representation': 'asinh100', 'history': 'full'}), 'V345': (Phase2ProductionRepresentationModel, {'kind': 'phase2_production_representation', 'representation': 'asinh100', 'history': 'recent3'}), 'V349': (Phase2QualitySupportModel, {'kind': 'phase2_quality_support', 'balance': 'weekpart', 'window': 'recent3'}), 'V350': (Phase2WeightedScalingModel, {'kind': 'phase2_weighted_scaling', 'base': 'V217'}), 'V351': (Phase2WeightedScalingModel, {'kind': 'phase2_weighted_scaling', 'base': 'V271'}), 'V352': (Phase2WeightedScalingModel, {'kind': 'phase2_weighted_scaling', 'base': 'V258'}), 'V353': (Phase2WeightedScalingModel, {'kind': 'phase2_weighted_scaling', 'base': 'V299'}), 'V354': (Phase2QualitySupportModel, {'kind': 'phase2_quality_support', 'balance': 'state', 'window': 'full'}), 'V355': (Phase2QualitySupportModel, {'kind': 'phase2_quality_support', 'balance': 'state', 'window': 'recent3'}), 'V356': (Phase2QualitySupportModel, {'kind': 'phase2_quality_support', 'balance': 'weekpart', 'window': 'full'}), 'V357': (Phase2ClockPoolModel, {'kind': 'phase2_clock_pool', 'blocks': 'state'}), 'V358': (Phase2ClockPoolModel, {'kind': 'phase2_clock_pool', 'blocks': 'hinge'}), 'V359': (Phase2ClockPoolModel, {'kind': 'phase2_clock_pool', 'blocks': 'state_hinge'}), 'V361': (Phase2StrongRepresentationModel, {'kind': 'phase2_strong_representation', 'representation': 'V351', 'weekend_correction': 'rbf'}), 'V362': (Phase2StrongRepresentationModel, {'kind': 'phase2_strong_representation', 'representation': 'V353', 'weekend_correction': 'rbf'}), 'V363': (Phase2WeightedScalingModel, {'kind': 'phase2_weighted_scaling', 'base': 'V271', 'loss': 'radial'}), 'V364': (Phase2WeightedScalingModel, {'kind': 'phase2_weighted_scaling', 'base': 'V299', 'loss': 'radial'}), 'V365': (Phase2GeometryComboModel, {'kind': 'phase2_geometry_combo', 'variant': 'no_hinge'}), 'V366': (Phase2GeometryComboModel, {'kind': 'phase2_geometry_combo', 'variant': 'asinh100'}), 'V367': (Phase2GeometryComboModel, {'kind': 'phase2_geometry_combo', 'variant': 'state_neutral'}), 'V368': (Phase2GeometryComboModel, {'kind': 'phase2_geometry_combo', 'variant': 'weekpart_neutral'}), 'V369': (Phase2CategoricalModel, {'kind': 'phase2_categorical', 'coding': 'symmetric'}), 'V370': (Phase2CategoricalModel, {'kind': 'phase2_categorical', 'coding': 'weighted_variance'}), 'V371': (Phase2WeekendScalingModel, {'kind': 'phase2_weekend_scaling', 'base': 'V351', 'weekend_scaler': 'weighted'}), 'V372': (Phase2WeekendScalingModel, {'kind': 'phase2_weekend_scaling', 'base': 'V353', 'weekend_scaler': 'weighted'}), 'V373': (Phase2WeekendScalingModel, {'kind': 'phase2_weekend_scaling', 'base': 'V351', 'weekend_scaler': 'iqr'}), 'V374': (Phase2WeekendScalingModel, {'kind': 'phase2_weekend_scaling', 'base': 'V353', 'weekend_scaler': 'iqr'}), 'V376': (Phase2GeometryComboModel, {'kind': 'phase2_geometry_combo', 'variant': 'no_hinge', 'weekend_correction': 'rbf'}), 'V377': (Phase2GeometryComboModel, {'kind': 'phase2_geometry_combo', 'variant': 'weekpart_neutral', 'weekend_correction': 'rbf'}), 'V378': (Phase2GeometryComboModel, {'kind': 'phase2_geometry_combo', 'variant': 'no_hinge', 'loss': 'radial'}), 'V379': (Phase2GeometryComboModel, {'kind': 'phase2_geometry_combo', 'variant': 'weekpart_neutral', 'loss': 'radial'}), 'V380': (Phase2JointGeometryModel, {'kind': 'phase2_joint_geometry', 'features_variant': 'no_hinge', 'quality_balance': 'state'}), 'V381': (Phase2JointGeometryModel, {'kind': 'phase2_joint_geometry', 'features_variant': 'no_hinge', 'quality_balance': 'weekpart'}), 'V382': (Phase2JointGeometryModel, {'kind': 'phase2_joint_geometry', 'features_variant': 'asinh100', 'quality_balance': 'state'}), 'V383': (Phase2JointGeometryModel, {'kind': 'phase2_joint_geometry', 'features_variant': 'asinh100', 'quality_balance': 'weekpart'}), 'V385': (Phase2WeekendScalingModel, {'kind': 'phase2_weekend_scaling', 'base': 'V365', 'weekend_scaler': 'iqr'}), 'V386': (Phase2WeekendScalingModel, {'kind': 'phase2_weekend_scaling', 'base': 'V366', 'weekend_scaler': 'iqr'}), 'V387': (Phase2WeekendScalingModel, {'kind': 'phase2_weekend_scaling', 'base': 'V368', 'weekend_scaler': 'iqr'}), 'V388': (Phase2WeekendScalingModel, {'kind': 'phase2_weekend_scaling', 'base': 'V367', 'weekend_scaler': 'iqr'})}
    V305_CONFIG = {'kind': 'phase2_blend', 'phase': 2, 'reference': 'V217', 'strategy': 'median', 'hypothesis': 'P2_H41', 'members': ['V299', 'V303', 'V258']}

    class M242Component:

        def fit(self, train):
            self.model = fit_m242(train)
            return self

        def predict(self, frame):
            return self.model.predict(frame)

    def make_component(name, seed=42):
        if name == 'M242':
            return M242Component()
        if name == 'N207':
            return N207Model()
        if name == 'R191':
            return R192Model(zero_weight=0.5)
        if name == 'R100':
            return R100Model()
        if name in ('V176', 'V177'):
            pass
            return Phase2CatBoostModel({'mode': 'joint' if name == 'V176' else 'mae'}, seed)
        if name == 'V305':
            pass
            return Phase2BlendModel(dict(V305_CONFIG), seed)
        model, config = COMPONENTS[name]
        return model(dict(config), seed)

    def fit_component(name, train, seed=42):
        return make_component(name, seed).fit(train)

    # Source: phase2_blend_models.py
    """Chronological, rowwise multi-model blending; callers supply past OOF only."""
    import numpy as np
    import pandas as pd
    from scipy import sparse
    from scipy.optimize import linprog, minimize
    from scipy.stats import rankdata
    from sklearn.linear_model import Ridge
    pass
    pass
    INPUTS = ['날짜', '시간', '생산량', '기온', '풍속', '습도', '강수량']

    def _members(config):
        return list(config.get('members', config.get('components', [])))

    def _prior(config, count):
        if 'prior_weights' in config:
            weights = np.asarray(config['prior_weights'], dtype=float)
        elif config.get('prior', 'anchor') == 'uniform':
            weights = np.ones(count)
        else:
            members = _members(config)
            anchor = config.get('anchor', 'V081')
            weights = np.zeros(count)
            weights[members.index(anchor) if anchor in members else 0] = 1.0
        assert weights.shape == (count,) and (weights >= 0).all() and (weights.sum() > 0)
        return weights / weights.sum()

    def _groups(frame, grouping):
        date = pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d')
        on = frame['생산량'].to_numpy() > 0
        weekend = date.dt.dayofweek.to_numpy() >= 5
        hour = frame['시간'].to_numpy()
        band = np.where(hour < 8, 0, np.where(hour <= 18, 1, 2))
        return {'zero': on.astype(int), 'weekend': weekend.astype(int), 'state_weekend': on.astype(int) * 2 + weekend, 'hour_band': band, 'state_hour': on.astype(int) * 3 + band}[grouping]

    def _indicators(frame):
        date = pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d')
        return np.column_stack([np.ones(len(frame)), frame['생산량'].to_numpy() > 0, date.dt.dayofweek.to_numpy() >= 5, frame['시간'].between(8, 18).to_numpy()]).astype(float)

    def _family_predictions(predictions, members, families, representative):
        flattened = [member for family in families.values() for member in family]
        assert len(flattened) == len(members) and sorted(flattened) == sorted(members)
        assert representative in ('mean', 'median')
        reduce = np.mean if representative == 'mean' else np.median
        return np.stack([reduce(predictions[:, [members.index(member) for member in family], :], axis=1) for family in families.values()], axis=1)

    def _conditional_groups(state, predictions, inputs):
        if 'regime' in state:
            return apply_regime(inputs, state['regime'])
        if state['group'] == 'disagreement':
            anchor = predictions[:, state['disagreement_anchor_index'], :]
            disagreement = np.abs(predictions - anchor[:, None, :]).mean(axis=(1, 2))
            return (disagreement > state['disagreement_threshold']).astype(int)
        return _groups(inputs, state['group'])

    def _flat(prediction, target, row_weight):
        matrix = prediction.transpose(0, 2, 1).reshape(-1, prediction.shape[1])
        weights = np.repeat(row_weight, prediction.shape[2])
        return (matrix, target.reshape(-1), weights / weights.sum())

    def _simplex(prediction, target, row_weight, prior, config, strategy):
        matrix, truth, weight = _flat(prediction, target, row_weight)
        count, models = matrix.shape
        penalty = float(config.get('regularization', 0.25 if strategy == 'simplex_mae' else 0.1))
        if strategy == 'simplex_mae':
            identity = sparse.eye(count, format='csr')
            zeros = sparse.csr_matrix((count, models))
            model_eye = sparse.eye(models, format='csr')
            padding = sparse.csr_matrix((models, count))
            constraints = sparse.vstack([sparse.hstack([matrix, -identity, zeros]), sparse.hstack([-matrix, -identity, zeros]), sparse.hstack([model_eye, padding, -model_eye]), sparse.hstack([-model_eye, padding, -model_eye])], format='csr')
            result = linprog(np.r_[np.zeros(models), weight, np.full(models, penalty)], A_ub=constraints, b_ub=np.r_[truth, -truth, prior, -prior], A_eq=sparse.csr_matrix(np.r_[np.ones(models), np.zeros(count + models)][None]), b_eq=[1.0], bounds=(0.0, None), method='highs')
            assert result.success, result.message
            weights = result.x[:models]
            details = dict(objective=float(result.fun), solver='highs', regularization=penalty)
        else:
            scale = max(float(np.sqrt(np.average((matrix @ prior - truth) ** 2, weights=weight))), 1.0)

            def objective(weights):
                error = matrix @ weights - truth
                value = np.sum(weight * error ** 2) / scale ** 2 + penalty * np.sum((weights - prior) ** 2)
                gradient = 2 * matrix.T @ (weight * error) / scale ** 2 + 2 * penalty * (weights - prior)
                return (value, gradient)
            result = minimize(objective, prior, jac=True, method='SLSQP', bounds=[(0.0, 1.0)] * models, constraints={'type': 'eq', 'fun': lambda w: w.sum() - 1.0, 'jac': lambda w: np.ones(models)}, options={'ftol': 1e-10, 'maxiter': 500})
            assert result.success, result.message
            weights = result.x
            details = dict(objective=float(result.fun), solver='SLSQP', regularization=penalty, scale=scale)
        weights = np.maximum(weights, 0.0)
        weights /= weights.sum()
        return (weights, details)

    def fit_meta(predictions, targets, inputs, config):
        """Fit on strictly historical OOF supplied by the caller; no outer truth."""
        predictions, targets = (np.asarray(predictions), np.asarray(targets))
        assert predictions.ndim == 3 and predictions.shape[2] == 4
        assert targets.shape == (len(inputs), 4) and len(predictions) == len(inputs)
        assert len(inputs) and np.isfinite(predictions).all() and np.isfinite(targets).all()
        months = inputs['날짜'].to_numpy() // 100
        assert months.min() >= 202103 and months.max() <= 202108
        history = config.get('history', 'expanding')
        assert history in ('expanding', 'recent3')
        if history == 'recent3':
            selected = np.isin(months, np.unique(months)[-3:])
            predictions, targets = (predictions[selected], targets[selected])
            inputs, months = (inputs.iloc[np.flatnonzero(selected)], months[selected])
        strategy = config.get('strategy', config.get('method', 'mean'))
        members = _members(config)
        assert len(members) == predictions.shape[1]
        prior = _prior(config, len(members))
        _, inverse, counts = np.unique(months, return_inverse=True, return_counts=True)
        row_weight = 1.0 / counts[inverse]
        state = dict(strategy=strategy, members=members, training_rows=len(inputs), training_months=[int(m) for m in np.unique(months)], month_row_counts={str(m): int((months == m).sum()) for m in np.unique(months)}, prior=prior.tolist(), weighting='equal_month', history=history)
        if strategy in ('mean', 'median', 'trimmed'):
            state['weights'] = (np.ones(len(members)) / len(members)).tolist()
        elif strategy == 'fixed':
            weights = np.asarray(config['weights'], dtype=float)
            assert weights.shape == prior.shape and (weights >= 0).all() and (weights.sum() > 0)
            state['weights'] = (weights / weights.sum()).tolist()
        elif strategy in ('performance', 'filtered_mean'):
            mae = np.average(np.abs(predictions - targets[:, None, :]).mean(axis=2), axis=0, weights=row_weight)
            state['past_mae'] = dict(zip(members, mae.tolist()))
            if strategy == 'performance':
                rule = config.get('performance', 'inverse_mae')
                assert rule in ('inverse_mae', 'rank')
                weights = 1.0 / (np.maximum(mae, 1e-08) if rule == 'inverse_mae' else rankdata(mae, method='average'))
                weights /= weights.sum()
                state.update(performance=rule, performance_weights=weights.tolist(), prior_shrink=0.5, weights=(0.5 * prior + 0.5 * weights).tolist())
            else:
                anchor = config.get('anchor', 'V081')
                index = members.index(anchor) if anchor in members else 0
                eligible = mae <= mae[index]
                eligible[index] = True
                representative = config.get('representative', 'mean')
                assert representative in ('mean', 'median')
                state.update(anchor=members[index], representative=representative, eligible_indices=np.flatnonzero(eligible).tolist(), eligible_members=[name for name, keep in zip(members, eligible) if keep], weights=(eligible / eligible.sum()).tolist())
        elif strategy == 'per_target':
            base_strategy = config.get('base_strategy', 'simplex_mae')
            assert base_strategy in ('simplex_mae', 'ridgeconvex')
            state.update(weights=[], optimization=[], base_strategy=base_strategy)
            for target in range(4):
                weights, detail = _simplex(predictions[:, :, target:target + 1], targets[:, target:target + 1], row_weight, prior, config, base_strategy)
                state['weights'].append(weights.tolist())
                state['optimization'].append(detail)
        elif strategy == 'hierarchical':
            families = config['families']
            representative = config.get('representative', 'mean')
            grouped = _family_predictions(predictions, members, families, representative)
            if config.get('prior') == 'uniform' and 'prior_weights' not in config:
                family_prior = np.ones(len(families)) / len(families)
            else:
                family_prior = np.array([sum((prior[members.index(member)] for member in family)) for family in families.values()])
            base_strategy = config.get('base_strategy', 'simplex_mae')
            assert base_strategy in ('simplex_mae', 'ridgeconvex')
            weights, detail = _simplex(grouped, targets, row_weight, family_prior, config, base_strategy)
            state.update(families=families, representative=representative, base_strategy=base_strategy, family_prior=family_prior.tolist(), weights=weights.tolist(), optimization=detail)
        elif strategy in ('simplex_mae', 'ridgeconvex', 'conditional'):
            base_strategy = config.get('base_strategy', 'simplex_mae') if strategy == 'conditional' else strategy
            weights, state['optimization'] = _simplex(predictions, targets, row_weight, prior, config, base_strategy)
            state['weights'] = weights.tolist()
            if strategy == 'conditional':
                grouping = config.get('group', 'state_weekend')
                state.update(group=grouping, groups={}, group_shrink_rows=720, minimum_group_rows=96)
                if grouping in ('temperature', 'production_band'):
                    state['regime'] = fit_regime(inputs, grouping)
                if grouping == 'disagreement':
                    anchor = config.get('anchor', 'V081')
                    index = members.index(anchor) if anchor in members else 0
                    disagreement = np.abs(predictions - predictions[:, index:index + 1, :]).mean(axis=(1, 2))
                    state.update(disagreement_anchor_index=index, disagreement_anchor=members[index], disagreement_quantile=0.75, disagreement_threshold=float(np.quantile(disagreement, 0.75)))
                groups = _conditional_groups(state, predictions, inputs)
                for group in np.unique(groups):
                    if group < 0:
                        continue
                    selected = groups == group
                    rows = int(selected.sum())
                    if rows < state['minimum_group_rows']:
                        continue
                    local, detail = _simplex(predictions[selected], targets[selected], row_weight[selected], weights, config, base_strategy)
                    shrink = rows / (rows + state['group_shrink_rows'])
                    state['groups'][str(int(group))] = dict(rows=rows, shrink=shrink, weights=(weights * (1 - shrink) + local * shrink).tolist(), optimization=detail)
        elif strategy == 'ridge':
            base = np.einsum('nkt,k->nt', predictions, prior)
            difference = (predictions - base[:, None, :]).transpose(0, 2, 1).reshape(-1, len(members))
            scale = np.maximum(difference.std(axis=0), 1.0)
            design = np.column_stack([difference / scale, np.repeat(_indicators(inputs), 4, axis=0)])
            alpha = float(config.get('alpha', 10000.0))
            estimator = Ridge(alpha=alpha, fit_intercept=False)
            estimator.fit(design, (targets - base).reshape(-1), sample_weight=np.repeat(row_weight / row_weight.mean(), 4))
            state.update(scale=scale.tolist(), coefficient=estimator.coef_.tolist(), alpha=alpha)
        else:
            raise ValueError(strategy)
        return state

    def predict_meta(state, predictions, inputs):
        predictions = np.asarray(predictions)
        strategy = state['strategy']
        if strategy == 'median':
            return np.median(predictions, axis=1)
        if strategy == 'trimmed':
            assert predictions.shape[1] >= 3
            return np.sort(predictions, axis=1)[:, 1:-1, :].mean(axis=1)
        if strategy == 'filtered_mean':
            reduce = np.mean if state['representative'] == 'mean' else np.median
            return reduce(predictions[:, state['eligible_indices'], :], axis=1)
        if strategy == 'per_target':
            return np.einsum('nkt,tk->nt', predictions, state['weights'])
        if strategy == 'hierarchical':
            grouped = _family_predictions(predictions, state['members'], state['families'], state['representative'])
            return np.einsum('nkt,k->nt', grouped, state['weights'])
        if strategy == 'ridge':
            base = np.einsum('nkt,k->nt', predictions, state['prior'])
            difference = (predictions - base[:, None, :]).transpose(0, 2, 1).reshape(-1, predictions.shape[1])
            design = np.column_stack([difference / state['scale'], np.repeat(_indicators(inputs), 4, axis=0)])
            return base + (design @ state['coefficient']).reshape(-1, 4)
        prediction = np.einsum('nkt,k->nt', predictions, state['weights'])
        if strategy == 'conditional':
            groups = _conditional_groups(state, predictions, inputs)
            for group, fitted in state['groups'].items():
                selected = groups == int(group)
                prediction[selected] = np.einsum('nkt,k->nt', predictions[selected], fitted['weights'])
        return prediction

    class Phase2Blend:
        """Small cache-harness adapter; all state is JSON serializable."""

        def __init__(self, config, seed=42):
            self.config, self.seed = (config, seed)

        def fit(self, inputs, predictions, targets):
            self.state_ = fit_meta(predictions, targets, inputs, self.config)
            return self

        def predict(self, inputs, predictions):
            return predict_meta(self.state_, predictions, inputs)

        def audit(self):
            return self.state_

    class Phase2BlendModel:
        """Fixed heads need no meta labels; learned heads use forward-chain rows."""

        def __init__(self, config, seed=42):
            self.config, self.seed = (config, seed)

        def fit(self, train):
            pass
            assert train['날짜'].between(20210101, 20210831).all()
            self.members_ = _members(self.config)
            strategy = self.config.get('strategy', self.config.get('method', 'mean'))
            if strategy in ('mean', 'median', 'trimmed', 'fixed'):
                weights = np.asarray(self.config['weights'], dtype=float) if strategy == 'fixed' else np.ones(len(self.members_))
                assert weights.shape == (len(self.members_),) and (weights >= 0).all() and (weights.sum() > 0)
                self.meta_ = Phase2Blend(self.config, self.seed)
                self.meta_.state_ = dict(strategy=strategy, members=self.members_, weights=(weights / weights.sum()).tolist(), training_rows=0, training_months=[], meta_training_required=False)
            else:
                months = train['날짜'].to_numpy() // 100
                frames, predictions, targets = ([], [], [])
                for month in sorted(set(months)):
                    if month < 202103:
                        continue
                    past, valid = (train.loc[months < month], train.loc[months == month])
                    prediction = np.stack([make_component(name, self.seed).fit(past).predict(valid[INPUTS]) for name in self.members_], axis=1)
                    frames.append(valid[INPUTS])
                    predictions.append(prediction)
                    targets.append(valid[POWER].to_numpy())
                self.meta_ = Phase2Blend(self.config, self.seed).fit(pd.concat(frames), np.concatenate(predictions), np.concatenate(targets))
            self.models_ = [make_component(name, self.seed).fit(train) for name in self.members_]
            self.training_summary_ = self.meta_.audit()
            return self

        def predict(self, frame):
            predictions = np.stack([model.predict(frame[INPUTS]) for model in self.models_], axis=1)
            return self.meta_.predict(frame[INPUTS], predictions)

    NAME, CONFIG, SEED = 'V384', {'kind': 'phase2_blend', 'strategy': 'median', 'members': ['V258', 'V271', 'V299', 'V300', 'V301', 'V303', 'V304', 'V317', 'V318', 'V319', 'V320', 'V321', 'V324', 'V334', 'V336', 'V337', 'V341', 'V342', 'V343', 'V344', 'V345', 'V349', 'V350', 'V351', 'V352', 'V353', 'V355', 'V356', 'V358', 'V361', 'V362', 'V363', 'V364', 'V365', 'V366', 'V367', 'V368', 'V369', 'V371', 'V372', 'V373', 'V374', 'V376', 'V377', 'V378', 'V379', 'V380', 'V381', 'V382', 'V383', 'V385', 'V386', 'V387', 'V388'], 'membership_rule': 'P2_AUTO_MAE1215_ALL_MEDIAN', 'membership_fingerprint': '556a7cda542359c8ccedd8123c9fde096df885fc7543ece01f956e01a2ceb833', 'seed_confirmation_required': False, 'phase': 2, 'reference': 'V305', 'hypothesis': 'P2_AUTO_MAE1215_ALL_MEDIAN'}, 42
    INPUTS = ['날짜', '시간', '생산량', '기온', '풍속', '습도', '강수량']
    np.random.seed(SEED)
    filtered_train = train_df.loc[
        train_df['날짜'].between(20210101, 20210831) & train_df['시간'].between(0, 23)
    ].sort_values(['날짜', '시간']).copy()
    assert len(filtered_train) == 5784
    assert not filtered_train.duplicated(['날짜', '시간']).any()
    assert filtered_train[POWER].notna().all().all()

    def fit_exported(frame):
        return Phase2BlendModel(CONFIG, SEED).fit(frame)

    # 1~3월→4월, …, 1~7월→8월. 구성 모델 학습 기간·가중치 규칙 원본 그대로
    scores, predictions = [], []
    with threadpool_limits(limits=2):
        for month in VALIDATION_MONTHS:
            tr = filtered_train.loc[filtered_train['날짜'].between(20210101, month * 100 - 1)]
            va = filtered_train.loc[filtered_train['날짜'].floordiv(100).eq(month)]
            assert not tr.empty and not va.empty
            assert tr['날짜'].max() < va['날짜'].min()
            fold_model = fit_exported(tr)
            prediction = fold_model.predict(va[INPUTS])
            assert prediction.shape == (len(va), 4) and np.isfinite(prediction).all()
            scores.append(dict(month=month, **metrics(va[POWER], prediction)))
            predictions.append(pd.DataFrame(prediction, index=va.index, columns=POWER).assign(month=month))
            print(f"{NAME} {month}: MAE={scores[-1]['MAE']:.6f}")
            del fold_model
        # 최종 모델: 1~8월 데이터 전달, 구성 모델별 학습 범위 적용
        final_model = fit_exported(filtered_train)
    return pd.DataFrame(scores).set_index('month'), pd.concat(predictions).sort_index(), final_model


cv_scores_v384, cv_predictions_v384, model_12 = train_v384(train_df)
cv_results["V384"] = {"scores": cv_scores_v384}
print("V384 월별 확장 교차검증")
print(cv_scores_v384.to_string())
print("월평균 MAE:", cv_scores_v384["MAE"].mean())


V384 202104: MAE=16.688030
V384 202105: MAE=10.578546
V384 202106: MAE=11.855114
V384 202107: MAE=12.292130
V384 202108: MAE=8.339727
V384 월별 확장 교차검증
              MAE       RMSE        R2
month                                 
202104  16.688030  28.272518  0.725269
202105  10.578546  23.945780  0.809619
202106  11.855114  20.199652  0.867923
202107  12.292130  23.241973  0.855100
202108   8.339727  15.049044  0.940670
월평균 MAE: 11.95070926839079


#### V385 (단독 파이프라인: V365 + 주말 IQR 표준화)

- V365 기반 회귀(최근 3개월 학습) + 주말 잔차 RBF·poly SVR(RobustScaler IQR), 월별 확장 윈도우 CV, MAE·RMSE·R²

In [19]:
# V385: 전체 파이프라인 단일 셀 (외부 연구 모듈 import 불필요)
# 원본: scripts/research_v2/cells_phase2/V385.py
# 원본 SHA256: 9692316044c3c965e7a2972e27ffae7f6bc72c15c30061913a2e59bd96fa3acf
# 내부 이름 함수 범위 한정 (앞선 모델 클래스·피처·설정 보호)
# 9월 예측·평가: '9월 테스트' 셀
def train_v385(train_df):
    # %% V385: standalone cell for notebooks/01model.ipynb
    # Requires train_df; test_df is never scored by this cell.

    # Source: notebook_models.py
    from collections import Counter, defaultdict
    import pandas as pd
    from sklearn.linear_model import LinearRegression
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import HuberRegressor
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.pipeline import Pipeline, make_pipeline
    from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
    from sklearn.svm import SVR
    from threadpoolctl import threadpool_limits
    import numpy as np
    import pandas as pd
    SEED = 42
    POWER = ['15분', '30분', '45분', '60분']
    VALIDATION_MONTHS = [202104, 202105, 202106, 202107, 202108]

    def metrics(y, pred):
        y, pred = (np.asarray(y).ravel(), np.asarray(pred).ravel())
        return dict(MAE=mean_absolute_error(y, pred), RMSE=np.sqrt(mean_squared_error(y, pred)), R2=r2_score(y, pred))

    def m242_features(frame):
        """날짜·시간·생산량·기온 4개 열 기반 피처"""
        assert frame['시간'].between(0, 23).all(), 'hour must be 0..23'
        hour = frame['시간']
        weekday = pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d').dt.dayofweek
        log_prod = np.log1p(frame['생산량'])
        on = log_prod.gt(0).where(log_prod.notna()).astype(float)
        hinge = (log_prod - np.log1p(100)).clip(lower=0)
        x = pd.DataFrame({'hour_weekday': hour.astype(str) + '_' + weekday.astype(str), '기온': frame['기온'], 'log_prod': log_prod, 'on': on, 'temp_on': frame['기온'] * on, 'temp_above20': (frame['기온'] - 20).clip(lower=0)}, index=frame.index)
        for day in range(7):
            x[f'prod_day{day}'] = log_prod * weekday.eq(day)
        for h in range(24):
            x[f'prod_hour{h}'] = log_prod * hour.eq(h)
        for h in range(24):
            x[f'prod_hinge_hour{h}'] = hinge * hour.eq(h)
        x['prod_sin'] = log_prod * np.sin(2 * np.pi * hour / 24)
        x['prod_cos'] = log_prod * np.cos(2 * np.pi * hour / 24)
        return x
    m242_pipeline = Pipeline([('features', FunctionTransformer(m242_features)), ('preprocess', ColumnTransformer([('category', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), ['hour_weekday'])], remainder=make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler()))), ('huber', MultiOutputRegressor(HuberRegressor(alpha=1.5, epsilon=1.05, max_iter=2000, tol=1e-07)))])

    def m242_sample_weight(train):
        """최근 가중(반감기 90일), 생산량 0 행 2배 가중"""
        date = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
        weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
        weight = np.where(train['생산량'].eq(0), 2.0, 1.0) * weight / weight.mean()
        return weight / weight.mean()

    def fit_m242(train):
        model = clone(m242_pipeline)
        with threadpool_limits(limits=2):
            return model.fit(train, train[POWER], huber__sample_weight=m242_sample_weight(train))

    def is_weekend(frame):
        return pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d').dt.dayofweek.ge(5)

    def n207_residual_features(frame):
        """주말 잔차 보정용: 시간·요일(범주), 기온·로그 생산량"""
        assert frame['시간'].between(0, 23).all(), 'hour must be 0..23'
        return pd.DataFrame({'시간': frame['시간'], '기온': frame['기온'], 'weekday': pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d').dt.dayofweek, 'log_prod': np.log1p(frame['생산량'])}, index=frame.index)
    n207_residual_pipeline = Pipeline([('features', FunctionTransformer(n207_residual_features)), ('preprocess', ColumnTransformer([('category', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['시간', 'weekday'])], remainder=make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler()))), ('svr', MultiOutputRegressor(SVR(kernel='rbf', C=10, epsilon=1, gamma='scale', tol=0.0001)))])

    class N207Model:
        """M242(전체 행 학습) + 주말 행 잔차 보정기. predict(frame) → (행 수, 4)"""

        def fit(self, train):
            self.base = fit_m242(train)
            residual = train[POWER] - self.base.predict(train)
            weekend = is_weekend(train)
            with threadpool_limits(limits=2):
                self.residual = clone(n207_residual_pipeline).fit(train.loc[weekend], residual.loc[weekend])
            return self

        def predict(self, frame):
            pred = self.base.predict(frame)
            weekend = is_weekend(frame).to_numpy()
            if weekend.any():
                pred[weekend] += self.residual.predict(frame.loc[weekend])
            return pred
    from sklearn.base import BaseEstimator, TransformerMixin

    class RHuberFeatures(TransformerMixin, BaseEstimator):
        """날짜 추세 기준일 fit 시 고정(평가 시 재계산 없음)"""

        def __init__(self, trend=False):
            self.trend = trend

        def fit(self, frame, y=None):
            if self.trend:
                date = pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d')
                self.anchor_ = float(date.max().toordinal())
            return self

        def transform(self, frame):
            assert frame['생산량'].dropna().ge(0).all()
            x = m242_features(frame)
            if self.trend:
                date = pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d')
                ordinal = date.map(lambda value: value.toordinal()).astype(float)
                gate = frame['생산량'].eq(0) & date.dt.dayofweek.lt(5)
                x['bounded_date_trend'] = ((ordinal - self.anchor_) / 60).clip(-1, 0) * gate
            return x

    class RWeekendModel:
        """전체 학습 행 Huber + 주말 학습 행 잔차 SVR. 출력 순서 = POWER"""

        def __init__(self, kernel='rbf', numeric_weight=1.0, zero_weight=2.0, trend=False):
            self.kernel = kernel
            self.numeric_weight = numeric_weight
            self.zero_weight = zero_weight
            self.trend = trend

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days / 90)
            weight = (weight / weight.mean()).to_numpy()
            weight = np.where(inputs['생산량'].eq(0), self.zero_weight, 1.0) * weight
            weight = weight / weight.mean()
            self.base = clone(m242_pipeline).set_params(features=RHuberFeatures(trend=self.trend))
            with threadpool_limits(limits=2):
                self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = date.dt.dayofweek.ge(5).to_numpy()
            assert weekend.any(), '주말 잔차 모델을 학습할 행이 필요합니다.'
            self.residual = Pipeline([('features', FunctionTransformer(n207_residual_features)), ('preprocess', ColumnTransformer([('category', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['시간', 'weekday']), ('numeric', make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler()), ['기온', 'log_prod'])], transformer_weights={'numeric': self.numeric_weight})), ('svr', MultiOutputRegressor(SVR(kernel=self.kernel, degree=2, coef0=1, C=10, epsilon=1, gamma='scale', tol=0.0001)))])
            with threadpool_limits(limits=2):
                self.residual.fit(inputs.loc[weekend], residual[weekend])
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            prediction = self.base.predict(inputs)
            weekend = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d').dt.dayofweek.ge(5).to_numpy()
            if weekend.any():
                prediction[weekend] += self.residual.predict(inputs.loc[weekend])
            return prediction

    class R192Model:
        """R192: 날짜 추세를 넣은 RWeekendModel 두 개(RBF 수치 가중 .5 / poly)를 .75/.25로 조합.
        zero_weight: Huber 생산량 0 행 가중치 (R192=1.0, R191=0.5)."""

        def __init__(self, zero_weight=1.0):
            self.zero_weight = zero_weight

        def fit(self, train):
            self.members = [RWeekendModel('rbf', numeric_weight=0.5, zero_weight=self.zero_weight, trend=True).fit(train), RWeekendModel('poly', zero_weight=self.zero_weight, trend=True).fit(train)]
            return self

        def predict(self, frame):
            left, right = (member.predict(frame) for member in self.members)
            return 0.75 * left + 0.25 * right

    class R100Model:

        def fit(self, train):
            self.members = [RWeekendModel('rbf').fit(train), RWeekendModel('poly').fit(train)]
            return self

        def predict(self, frame):
            left, right = (member.predict(frame) for member in self.members)
            zero = frame['생산량'].eq(0).to_numpy()[:, None]
            return np.where(zero, left, right)

    # Source: duplicate_models.py
    """Weight exact repeated training-day curves without removing any rows."""
    from collections import Counter
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from threadpoolctl import threadpool_limits
    pass

    class DuplicateModel:

        def __init__(self, config, seed=42):
            self.mode = config.get('mode', 'curve_equal')
            assert self.mode in ('curve_equal', 'curve_sqrt')

        def day_weights(self, train):
            curves = {}
            for date, day in train.groupby('날짜'):
                day = day.sort_values('시간')
                if len(day) == 24 and np.array_equal(day['시간'].to_numpy(), np.arange(24)):
                    curves[date] = day[POWER].to_numpy(dtype='<f8').tobytes()
            counts = Counter(curves.values())
            exponent = 1.0 if self.mode == 'curve_equal' else 0.5
            weights = {date: counts[curve] ** (-exponent) for date, curve in curves.items()}
            return train['날짜'].map(weights).fillna(1.0).to_numpy()

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0) * self.day_weights(train)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=RHuberFeatures(trend=True))
            weekend = is_weekend(inputs).to_numpy()
            self.residuals = []
            with threadpool_limits(limits=2):
                self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
                residual = train[POWER].to_numpy() - self.base.predict(inputs)
                for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                    correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                    correction.fit(inputs.loc[weekend], residual[weekend])
                    self.residuals.append(correction)
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            prediction = self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            if weekend.any():
                left, right = (model.predict(inputs.loc[weekend]) for model in self.residuals)
                prediction[weekend] += 0.75 * left + 0.25 * right
            return prediction

    # Source: state_interaction_models.py
    """H23: retain pooled training while adding state-dependent calendar responses."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.svm import SVR
    pass

    class StateInteractionFeatures(RHuberFeatures):

        def __init__(self, signal='on', grouping='weekday'):
            super().__init__(trend=True)
            self.signal, self.grouping = (signal, grouping)

        def transform(self, frame):
            features = super().transform(frame)
            weekday = pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d').dt.dayofweek
            groups = weekday if self.grouping == 'weekday' else weekday.ge(5).astype(int)
            logprod = np.log1p(frame['생산량'])
            signal = logprod if self.signal == 'logprod' else logprod.gt(0).where(logprod.notna()).astype(float)
            interactions = {f'state_hour_{hour}_group_{group}': signal * frame['시간'].eq(hour) * groups.eq(group) for hour in range(24) for group in range(7 if self.grouping == 'weekday' else 2)}
            return pd.concat([features, pd.DataFrame(interactions, index=frame.index)], axis=1)

    class StateInteractionModel:

        def __init__(self, config, seed=42):
            self.signal, self.grouping = (config.get('signal', 'on'), config.get('grouping', 'weekday'))
            assert self.signal in {'on', 'logprod'}
            assert self.grouping in {'weekday', 'weekend'}

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=StateInteractionFeatures(self.signal, self.grouping))
            self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            self.corrections = []
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                model = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr=MultiOutputRegressor(SVR(kernel=kernel, degree=2, coef0=1, C=10, epsilon=1, gamma='scale', tol=0.0001)))
                model.fit(inputs.loc[weekend], residual[weekend])
                self.corrections.append(model)
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            prediction = self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            if weekend.any():
                prediction[weekend] += 0.75 * self.corrections[0].predict(inputs.loc[weekend]) + 0.25 * self.corrections[1].predict(inputs.loc[weekend])
            return prediction

    # Source: thermal_state_combo_models.py
    """H27: original V054 intensity features plus original V070 state features."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.svm import SVR
    pass
    pass

    class ThermalStateComboFeatures(StateInteractionFeatures):

        def __init__(self):
            super().__init__(signal='on', grouping='weekend')

        def transform(self, frame):
            features = super().transform(frame)
            log_prod = np.log1p(frame['생산량'])
            features['temperature_intensity'] = frame['기온'] * log_prod
            features['cooling_intensity'] = (frame['기온'] - 20).clip(lower=0) * log_prod
            return features

    class ThermalStateComboModel(StateInteractionModel):

        def __init__(self, config, seed=42):
            assert config.get('mode', 'thermal_state') == 'thermal_state'
            super().__init__({'signal': 'on', 'grouping': 'weekend'}, seed)

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=ThermalStateComboFeatures())
            self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            self.corrections = []
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                model = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr=MultiOutputRegressor(SVR(kernel=kernel, degree=2, coef0=1, C=10, epsilon=1, gamma='scale', tol=0.0001)))
                model.fit(inputs.loc[weekend], residual[weekend])
                self.corrections.append(model)
            return self

    # Source: production_efficiency_models.py
    """H30: raw throughput response on V081, with train-only normalization."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.svm import SVR
    pass
    pass

    class ProductionEfficiencyFeatures(ThermalStateComboFeatures):

        def __init__(self, scale='global', replacement=False):
            super().__init__()
            self.scale, self.replacement = (scale, replacement)

        def fit(self, frame, y=None):
            super().fit(frame, y)
            positive = frame.loc[frame['생산량'].gt(0)].copy()
            positive['weekday'] = pd.to_datetime(positive['날짜'].astype(str), format='%Y%m%d').dt.dayofweek
            self.global_median = positive['생산량'].median()
            self.hour_medians = positive.groupby('시간')['생산량'].median()
            self.cell_medians = positive.groupby(['시간', 'weekday'])['생산량'].median()
            return self

        def transform(self, frame):
            features = super().transform(frame)
            denominator = np.full(len(frame), self.global_median)
            if self.scale == 'calendar':
                weekday = pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d').dt.dayofweek
                keys = pd.MultiIndex.from_arrays([frame['시간'], weekday], names=['시간', 'weekday'])
                denominator = self.cell_medians.reindex(keys).to_numpy()
                denominator = np.where(np.isnan(denominator), frame['시간'].map(self.hour_medians), denominator)
                denominator = np.where(np.isnan(denominator), self.global_median, denominator)
            if self.replacement:
                remove = [name for name in features if name.startswith('prod_') or name in {'log_prod', 'temperature_intensity', 'cooling_intensity'}]
                features = features.drop(columns=remove)
            throughput = frame['생산량'] / denominator
            features['throughput'] = throughput
            features['throughput_temperature'] = throughput * frame['기온']
            features['throughput_cooling'] = throughput * (frame['기온'] - 20).clip(lower=0)
            return features

    class ProductionEfficiencyModel:

        def __init__(self, config, seed=42):
            self.scale, self.mode = (config.get('scale', 'global'), config.get('mode', 'add'))
            assert self.scale in {'global', 'calendar'}
            assert self.mode in {'add', 'replace_log'}

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=ProductionEfficiencyFeatures(self.scale, self.mode == 'replace_log'))
            self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            self.corrections = []
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                model = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr=MultiOutputRegressor(SVR(kernel=kernel, degree=2, coef0=1, C=10, epsilon=1, gamma='scale', tol=0.0001)))
                self.corrections.append(model.fit(inputs.loc[weekend], residual[weekend]))
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            prediction = self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            if weekend.any():
                prediction[weekend] += 0.75 * self.corrections[0].predict(inputs.loc[weekend]) + 0.25 * self.corrections[1].predict(inputs.loc[weekend])
            return prediction

    # Source: radial_huber_models.py
    """H57: direct four-output Huber with one robust influence per observed hour."""
    import warnings
    import numpy as np
    import pandas as pd
    from scipy.optimize import minimize
    from sklearn.base import BaseEstimator, RegressorMixin, clone
    from sklearn.exceptions import ConvergenceWarning
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.svm import SVR
    pass
    pass
    pass

    def radial_huber_objective(parameters, matrix, target, weight):
        coefficients = parameters[:-5].reshape(matrix.shape[1], 4)
        intercept, scale = (parameters[-5:-1], parameters[-1])
        residual = (matrix @ coefficients + intercept - target) / scale
        radius = np.sqrt(np.mean(residual ** 2, axis=1))
        influence = np.minimum(1.0, 1.05 / np.maximum(radius, np.finfo(float).tiny))
        rho = np.where(radius <= 1.05, radius ** 2, 2 * 1.05 * radius - 1.05 ** 2)
        loss = 4 * (weight @ (scale * (1 + rho))) + 1.5 * np.sum(coefficients ** 2)
        score = weight[:, None] * influence[:, None] * residual
        gradient = np.r_[(2 * matrix.T @ score + 3 * coefficients).ravel(), 2 * score.sum(axis=0), 4 * (weight @ (1 + rho - 2 * radius * np.minimum(radius, 1.05)))]
        return (float(loss), gradient)

    class RadialHuberRegressor(RegressorMixin, BaseEstimator):

        def fit(self, matrix, target, sample_weight=None):
            target = np.asarray(target)
            weight = np.ones(len(target)) if sample_weight is None else np.asarray(sample_weight)
            initial = np.zeros(matrix.shape[1] * 4 + 5)
            initial[-5:-1] = np.median(target, axis=0)
            initial[-1] = max(float(np.median(np.sqrt(np.mean((target - initial[-5:-1]) ** 2, axis=1)))), 1.0)
            bounds = [(None, None)] * (len(initial) - 1) + [(np.finfo(float).eps * 10, None)]
            result = minimize(radial_huber_objective, initial, args=(matrix, target, weight), method='L-BFGS-B', jac=True, bounds=bounds, options={'maxiter': 2000, 'gtol': 1e-07, 'ftol': 1e-10})
            if result.status == 2:
                raise ValueError('Radial Huber optimization failed: ' + str(result.message))
            if not result.success:
                warnings.warn(str(result.message), ConvergenceWarning)
            self.coef_ = result.x[:-5].reshape(matrix.shape[1], 4)
            self.intercept_, self.scale_ = (result.x[-5:-1], result.x[-1])
            self.training_summary_ = dict(success=bool(result.success), iterations=int(result.nit), message=str(result.message), objective=float(result.fun), scale=float(self.scale_))
            return self

        def predict(self, matrix):
            return matrix @ self.coef_ + self.intercept_

    class RadialHuberModel(StateInteractionModel):

        def __init__(self, config, seed=42):
            assert config.get('loss', 'radial') == 'radial'

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=ThermalStateComboFeatures(), huber=RadialHuberRegressor())
            self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            self.training_summary_ = self.base.named_steps['huber'].training_summary_
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            self.corrections = []
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr=MultiOutputRegressor(SVR(kernel=kernel, degree=2, coef0=1, C=10, epsilon=1, gamma='scale', tol=0.0001)))
                correction.fit(inputs.loc[weekend], residual[weekend])
                self.corrections.append(correction)
            return self

    # Source: thermal_models.py
    """H18: fixed 20-degree cooling interactions with operating state/intensity."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.pipeline import Pipeline, make_pipeline
    from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
    from sklearn.svm import SVR
    from threadpoolctl import threadpool_limits
    pass

    class ThermalFeatures(RHuberFeatures):

        def __init__(self, mode='state_hinge'):
            super().__init__(trend=True)
            self.mode = mode

        def transform(self, frame):
            features = super().transform(frame)
            log_prod = np.log1p(frame['생산량'])
            on = log_prod.gt(0).where(log_prod.notna()).astype(float)
            cooling = (frame['기온'] - 20).clip(lower=0)
            if self.mode in ('state_hinge', 'both'):
                features['cooling_on'] = cooling * on
                features['cooling_off'] = cooling * (1 - on)
            if self.mode in ('intensity', 'both'):
                features['temperature_intensity'] = frame['기온'] * log_prod
                features['cooling_intensity'] = cooling * log_prod
            if self.mode == 'daytime_hinge':
                angle = 2 * np.pi * frame['시간'] / 24
                features['cooling_on'] = cooling * on
                features['cooling_on_sin'] = cooling * on * np.sin(angle)
                features['cooling_on_cos'] = cooling * on * np.cos(angle)
            return features

    class ThermalModel:

        def __init__(self, config, seed=42):
            self.mode = config.get('mode', 'state_hinge')
            assert self.mode in ('state_hinge', 'intensity', 'daytime_hinge', 'both')

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days / 90)
            weight = (weight / weight.mean()).to_numpy()
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=ThermalFeatures(self.mode))
            weekend = date.dt.dayofweek.ge(5).to_numpy()
            self.residuals = []
            with threadpool_limits(limits=2):
                self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
                residual = train[POWER].to_numpy() - self.base.predict(inputs)
                for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                    model = Pipeline([('features', FunctionTransformer(n207_residual_features)), ('preprocess', ColumnTransformer([('category', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['시간', 'weekday']), ('numeric', make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler()), ['기온', 'log_prod'])], transformer_weights={'numeric': numeric_weight})), ('svr', MultiOutputRegressor(SVR(kernel=kernel, degree=2, coef0=1, C=10, epsilon=1, gamma='scale', tol=0.0001)))])
                    self.residuals.append(model.fit(inputs.loc[weekend], residual[weekend]))
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            prediction = self.base.predict(inputs)
            weekend = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d').dt.dayofweek.ge(5).to_numpy()
            if weekend.any():
                left, right = (model.predict(inputs.loc[weekend]) for model in self.residuals)
                prediction[weekend] += 0.75 * left + 0.25 * right
            return prediction

    # Source: combo_models.py
    """Combine original thermal, duplicate-weight, and zero-specialist settings."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from threadpoolctl import threadpool_limits
    pass
    pass
    pass

    class ComboModel(DuplicateModel):

        def __init__(self, config, seed=42):
            self.combination = config.get('mode', 'thermal_duplicate')
            assert self.combination in ('thermal_duplicate', 'thermal_zero', 'thermal_duplicate_zero')
            super().__init__({'mode': 'curve_sqrt'}, seed)

        def feature_transformer(self):
            return ThermalFeatures('intensity')

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            duplicate_weight = 1.0 if self.combination == 'thermal_zero' else self.day_weights(train)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0) * duplicate_weight
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=self.feature_transformer())
            weekend = is_weekend(inputs).to_numpy()
            self.residuals = []
            with threadpool_limits(limits=2):
                self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
                residual = train[POWER].to_numpy() - self.base.predict(inputs)
                for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                    correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                    correction.fit(inputs.loc[weekend], residual[weekend])
                    self.residuals.append(correction)
                self.specialist = None
                if self.combination.endswith('_zero'):
                    region = train.loc[train['생산량'].eq(0)]
                    dates = pd.to_datetime(region['날짜'].astype(str), format='%Y%m%d')
                    specialist_weight = np.exp2(-(dates.max() - dates).dt.days.to_numpy() / 90)
                    specialist_weight /= specialist_weight.mean()
                    self.specialist = clone(m242_pipeline).set_params(features=RHuberFeatures(trend=True))
                    self.specialist.fit(region[['날짜', '시간', '생산량', '기온']], region[POWER], huber__sample_weight=specialist_weight)
            return self

        def predict(self, frame):
            prediction = super().predict(frame)
            if self.specialist is not None:
                zero = frame['생산량'].eq(0).to_numpy()
                if zero.any():
                    prediction[zero] = self.specialist.predict(frame.loc[zero, ['날짜', '시간', '생산량', '기온']])
            return prediction

    # Source: thermal_balance_models.py
    """Balance sparse thermal support with one fixed inverse-square-root rule."""
    import numpy as np
    pass
    pass

    class ThermalBalanceModel(ComboModel):

        def __init__(self, config, seed=42):
            super().__init__({'mode': 'thermal_duplicate'}, seed)
            self.grouping = config.get('grouping', 'temperature')
            assert self.grouping in ('temperature', 'state_temperature')

        def feature_transformer(self):
            return ThermalStateComboFeatures()

        def day_weights(self, train):
            group = train['기온'].floordiv(5).astype('Int64').astype(str)
            if self.grouping == 'state_temperature':
                group = train['생산량'].eq(0).astype(str) + '_' + group
            self.thermal_counts_ = group.value_counts().to_dict()
            weight = 1.0 / np.sqrt(group.map(self.thermal_counts_))
            return (weight / weight.mean()).to_numpy()

    # Source: phase2_quality_combo_models.py
    """Preserve V096 thermal support weights and V028 duplicate-curve weights."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.preprocessing import RobustScaler
    from threadpoolctl import threadpool_limits
    pass
    pass
    pass
    pass
    pass
    pass
    pass

    class Phase2QualityComboModel(ThermalBalanceModel):

        def __init__(self, config, seed=42):
            self.quality_mode = config.get('mode', 'state_temperature_duplicate')
            assert self.quality_mode in ('state_temperature_duplicate', 'raw_throughput', 'radial', 'features_r191', 'features_v054', 'features_v070', 'radial_v070', 'radial_v070_recent3', 'radial_v070_recent3_iqr')
            super().__init__({'grouping': 'state_temperature'}, seed)

        def feature_transformer(self):
            if self.quality_mode == 'raw_throughput':
                return ProductionEfficiencyFeatures(scale='global', replacement=False)
            if self.quality_mode == 'features_r191':
                return RHuberFeatures(trend=True)
            if self.quality_mode == 'features_v054':
                return ThermalFeatures(mode='intensity')
            if self.quality_mode in ('features_v070', 'radial_v070', 'radial_v070_recent3', 'radial_v070_recent3_iqr'):
                return StateInteractionFeatures(signal='on', grouping='weekend')
            return super().feature_transformer()

        def day_weights(self, train):
            return ThermalBalanceModel.day_weights(self, train) * DuplicateModel.day_weights(self, train)

        def fit(self, train):
            if self.quality_mode not in ('radial', 'radial_v070', 'radial_v070_recent3', 'radial_v070_recent3_iqr'):
                return super().fit(train)
            if self.quality_mode in ('radial_v070_recent3', 'radial_v070_recent3_iqr'):
                dates = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
                start = (dates.max().to_period('M') - 2).start_time
                train = train.loc[dates >= start]
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0) * self.day_weights(train)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=self.feature_transformer(), huber=RadialHuberRegressor())
            if self.quality_mode == 'radial_v070_recent3_iqr':
                self.base.set_params(preprocess__remainder__standardscaler=RobustScaler(quantile_range=(25.0, 75.0)))
            weekend = is_weekend(inputs).to_numpy()
            self.residuals, self.specialist = ([], None)
            with threadpool_limits(limits=2):
                self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
                residual = train[POWER].to_numpy() - self.base.predict(inputs)
                for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                    correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                    correction.fit(inputs.loc[weekend], residual[weekend])
                    self.residuals.append(correction)
            self.training_summary_ = self.base.named_steps['huber'].training_summary_
            return self

    # Source: phase2_production_representation_models.py
    """Two fixed production-response assumptions on the strong V271 base."""
    import numpy as np
    import pandas as pd
    pass
    pass

    class Phase2ProductionFeatures(StateInteractionFeatures):

        def __init__(self, representation):
            super().__init__(signal='on', grouping='weekend')
            self.representation = representation

        def transform(self, frame):
            features = super().transform(frame)
            hinge_columns = [f'prod_hinge_hour{hour}' for hour in range(24)]
            if self.representation == 'no_hinge':
                return features.drop(columns=hinge_columns)
            production = frame['생산량'].to_numpy(dtype=float)
            original = np.log1p(production)
            transformed = np.arcsinh(production / 100.0)
            ratio = np.divide(transformed, original, out=np.zeros(len(frame)), where=original > 0)
            columns = ['log_prod', 'prod_sin', 'prod_cos']
            columns += [f'prod_day{day}' for day in range(7)]
            columns += [f'prod_hour{hour}' for hour in range(24)]
            features[columns] = features[columns].mul(ratio, axis=0)
            original_hinge = np.maximum(original - np.log1p(100.0), 0.0)
            transformed_hinge = np.maximum(transformed - np.arcsinh(1.0), 0.0)
            hinge_ratio = np.divide(transformed_hinge, original_hinge, out=np.zeros(len(frame)), where=original_hinge > 0)
            features[hinge_columns] = features[hinge_columns].mul(hinge_ratio, axis=0)
            return features

    class Phase2ProductionRepresentationModel(Phase2QualityComboModel):

        def __init__(self, config, seed=42):
            self.representation, self.history = (config['representation'], config['history'])
            assert self.representation in ('no_hinge', 'asinh100')
            assert self.history in ('full', 'recent3')
            super().__init__({'mode': 'features_v070'}, seed)

        def feature_transformer(self):
            return Phase2ProductionFeatures(self.representation)

        def fit(self, train):
            if self.history == 'recent3':
                dates = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
                train = train.loc[dates >= (dates.max().to_period('M') - 2).start_time]
            super().fit(train)
            self.training_summary_ = dict(base='V271', representation=self.representation, history=self.history, rows=len(train), train_start=int(train['날짜'].min()), train_end=int(train['날짜'].max()), production_reference=100.0, other_settings='original V271 quality weights, Huber and weekend SVRs')
            return self

    # Source: phase2_quality_support_models.py
    """Preserve state/calendar mass while retaining within-group quality weights."""
    import numpy as np
    import pandas as pd
    pass

    class Phase2QualitySupportModel(Phase2QualityComboModel):

        def __init__(self, config, seed=42):
            super().__init__({'mode': 'features_v070'}, seed)
            self.balance = config['balance']
            self.window = config.get('window', 'full')
            assert self.balance in ('state', 'weekpart')
            assert self.window in ('full', 'recent3')

        def day_weights(self, train):
            quality = super().day_weights(train)
            dates = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
            original = np.exp2(-(dates.max() - dates).dt.days.to_numpy() / 90)
            original *= np.where(train['생산량'].eq(0), 0.5, 1.0)
            group = train['생산량'].gt(0) if self.balance == 'state' else dates.dt.dayofweek.ge(5)
            original = pd.Series(original, index=train.index)
            weighted = original * quality
            normalizer = weighted.groupby(group).sum() / original.groupby(group).sum()
            adjusted = quality / group.map(normalizer).to_numpy()
            self.weight_audit_ = dict(balance=self.balance, group_normalizers={str(k): float(v) for k, v in normalizer.items()}, original_group_mass={str(k): float(v) for k, v in original.groupby(group).sum().items()}, adjusted_group_mass={str(k): float(v) for k, v in (original * adjusted).groupby(group).sum().items()})
            return adjusted

        def fit(self, train):
            if self.window == 'recent3':
                dates = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
                train = train.loc[dates >= (dates.max().to_period('M') - 2).start_time]
            super().fit(train)
            self.training_summary_ = dict(base='V271', window=self.window, train_start=int(train['날짜'].min()), train_end=int(train['날짜'].max()), rows=len(train), **self.weight_audit_)
            return self

    # Source: phase2_weighted_scaling_models.py
    """Use the Huber training weights for numeric feature means and variances too."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.preprocessing import StandardScaler
    pass
    pass
    pass

    class FoldWeightedStandardScaler(StandardScaler):

        def __init__(self, training_weight):
            self.training_weight = training_weight
            super().__init__()

        def fit(self, matrix, y=None, sample_weight=None):
            assert len(matrix) == len(self.training_weight)
            return super().fit(matrix, y, sample_weight=self.training_weight)

    class Phase2WeightedScalingModel(Phase2QualityComboModel):

        def __init__(self, config, seed=42):
            self.scaling_base = config['base']
            self.loss = config.get('loss', 'independent')
            assert self.scaling_base in ('V217', 'V271', 'V258', 'V299')
            assert self.loss in ('independent', 'radial')
            mode = 'features_v070' if self.scaling_base in ('V271', 'V299') else 'state_temperature_duplicate'
            super().__init__({'mode': mode}, seed)

        def fit(self, train):
            if self.scaling_base in ('V258', 'V299'):
                dates = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
                start = (dates.max().to_period('M') - 2).start_time
                train = train.loc[dates >= start]
            inputs = train[['날짜', '시간', '생산량', '기온']]
            dates = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(dates.max() - dates).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0) * self.day_weights(train)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=self.feature_transformer(), preprocess__remainder__standardscaler=FoldWeightedStandardScaler(weight))
            if self.loss == 'radial':
                self.base.set_params(huber=RadialHuberRegressor())
            self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            self.residuals, self.specialist = ([], None)
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                correction.fit(inputs.loc[weekend], residual[weekend])
                self.residuals.append(correction)
            self.training_summary_ = dict(base=self.scaling_base, numeric_scaler='StandardScaler fitted with exact Huber sample weights', categorical_and_imputation='unchanged and unweighted', weekend_correction='original unweighted 0.75 RBF + 0.25 polynomial SVR', training_rows=len(train), train_start=int(train['날짜'].min()), train_end=int(train['날짜'].max()))
            if self.loss == 'radial':
                self.training_summary_.update(loss='original radial Huber', solver=self.base.named_steps['huber'].training_summary_)
            return self

    # Source: phase2_geometry_combo_models.py
    """Apply V353 weighted feature geometry to four preserved recent-three designs."""
    pass
    pass
    pass
    pass

    class Phase2GeometryComboModel(Phase2WeightedScalingModel):

        def __init__(self, config, seed=42):
            super().__init__({'base': 'V299', 'loss': config.get('loss', 'independent')}, seed)
            self.variant = config['variant']
            self.weekend_correction = config.get('weekend_correction', 'original')
            self.seed = seed
            assert self.variant in ('no_hinge', 'asinh100', 'state_neutral', 'weekpart_neutral')
            assert self.weekend_correction in ('original', 'rbf')

        def feature_transformer(self):
            if self.variant in ('no_hinge', 'asinh100'):
                return Phase2ProductionFeatures(self.variant)
            return super().feature_transformer()

        def day_weights(self, train):
            if self.variant.endswith('_neutral'):
                model = Phase2QualitySupportModel({'balance': self.variant.removesuffix('_neutral'), 'window': 'recent3'}, self.seed)
                weights = model.day_weights(train)
                self.weight_audit_ = model.weight_audit_
                return weights
            return super().day_weights(train)

        def fit(self, train):
            super().fit(train)
            self.training_summary_['structural_combination'] = self.variant
            if self.variant.endswith('_neutral'):
                self.training_summary_['quality_normalization'] = self.weight_audit_
            if self.weekend_correction == 'rbf':
                self.training_summary_['weekend_correction'] = 'original RBF SVR alone'
            return self

        def predict(self, frame):
            if self.weekend_correction == 'original':
                return super().predict(frame)
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            prediction = self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            if weekend.any():
                prediction[weekend] += self.residuals[0].predict(inputs.loc[weekend])
            return prediction

    # Source: phase2_weekend_scaling_models.py
    """Change only the numeric geometry inside the existing two weekend SVRs."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.preprocessing import RobustScaler
    pass
    pass
    pass

    class Phase2WeekendScalingModel(Phase2WeightedScalingModel):

        def __init__(self, config, seed=42):
            self.member, self.weekend_scaler = (config['base'], config['weekend_scaler'])
            self.seed = seed
            assert self.member in ('V351', 'V353', 'V365', 'V366', 'V367', 'V368')
            assert self.weekend_scaler in ('weighted', 'iqr')
            assert self.member in ('V351', 'V353') or self.weekend_scaler == 'iqr'
            super().__init__({'base': 'V271' if self.member == 'V351' else 'V299'}, seed)

        def fit(self, train):
            if self.member != 'V351':
                dates = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
                train = train.loc[dates >= (dates.max().to_period('M') - 2).start_time]
            if self.member in ('V365', 'V366', 'V367', 'V368'):
                variant = {'V365': 'no_hinge', 'V366': 'asinh100', 'V367': 'state_neutral', 'V368': 'weekpart_neutral'}[self.member]
                model = Phase2GeometryComboModel({'variant': variant}, self.seed).fit(train)
                self.base, self.specialist = (model.base, None)
                self.training_summary_ = dict(model.training_summary_)
            else:
                super().fit(train)
            inputs = train[['날짜', '시간', '생산량', '기온']]
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            if self.weekend_scaler == 'weighted':
                dates = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
                weight = np.exp2(-(dates.max() - dates).dt.days.to_numpy() / 90)
                weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0) * self.day_weights(train)
                weight = weight[weekend]
                weight /= weight.mean()
                scaler = FoldWeightedStandardScaler(weight)
            else:
                scaler = RobustScaler(quantile_range=(25.0, 75.0))
            self.residuals = []
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, preprocess__remainder__standardscaler=clone(scaler), svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                correction.fit(inputs.loc[weekend], residual[weekend])
                self.residuals.append(correction)
            self.training_summary_.update(member=self.member, weekend_scaler=self.weekend_scaler, weekend_svr_loss='unchanged unweighted', replaced_step='weekend SVRs preprocess__remainder__standardscaler')
            return self

    NAME, CONFIG, SEED = 'V385', {'kind': 'phase2_weekend_scaling', 'base': 'V365', 'weekend_scaler': 'iqr', 'phase': 2, 'reference': 'V305', 'hypothesis': 'P2_H61'}, 42
    INPUTS = ['날짜', '시간', '생산량', '기온', '풍속', '습도', '강수량']
    np.random.seed(SEED)
    filtered_train = train_df.loc[
        train_df['날짜'].between(20210101, 20210831) & train_df['시간'].between(0, 23)
    ].sort_values(['날짜', '시간']).copy()
    assert len(filtered_train) == 5784
    assert not filtered_train.duplicated(['날짜', '시간']).any()
    assert filtered_train[POWER].notna().all().all()

    def fit_exported(frame):
        return Phase2WeekendScalingModel(CONFIG, SEED).fit(frame)

    # 1~3월→4월, …, 1~7월→8월. 구성 모델 학습 기간·가중치 규칙 원본 그대로
    scores, predictions = [], []
    with threadpool_limits(limits=2):
        for month in VALIDATION_MONTHS:
            tr = filtered_train.loc[filtered_train['날짜'].between(20210101, month * 100 - 1)]
            va = filtered_train.loc[filtered_train['날짜'].floordiv(100).eq(month)]
            assert not tr.empty and not va.empty
            assert tr['날짜'].max() < va['날짜'].min()
            fold_model = fit_exported(tr)
            prediction = fold_model.predict(va[INPUTS])
            assert prediction.shape == (len(va), 4) and np.isfinite(prediction).all()
            scores.append(dict(month=month, **metrics(va[POWER], prediction)))
            predictions.append(pd.DataFrame(prediction, index=va.index, columns=POWER).assign(month=month))
            print(f"{NAME} {month}: MAE={scores[-1]['MAE']:.6f}")
            del fold_model
        # 최종 모델: 1~8월 데이터 전달, 구성 모델별 학습 범위 적용
        final_model = fit_exported(filtered_train)
    return pd.DataFrame(scores).set_index('month'), pd.concat(predictions).sort_index(), final_model


cv_scores_v385, cv_predictions_v385, model_13 = train_v385(train_df)
cv_results["V385"] = {"scores": cv_scores_v385}
print("V385 월별 확장 교차검증")
print(cv_scores_v385.to_string())
print("월평균 MAE:", cv_scores_v385["MAE"].mean())


V385 202104: MAE=16.635907
V385 202105: MAE=10.550708
V385 202106: MAE=11.883873
V385 202107: MAE=12.255603
V385 202108: MAE=8.415030
V385 월별 확장 교차검증
              MAE       RMSE        R2
month                                 
202104  16.635907  28.187361  0.726922
202105  10.550708  24.046071  0.808021
202106  11.883873  20.245074  0.867329
202107  12.255603  23.222138  0.855347
202108   8.415030  15.154295  0.939837
월평균 MAE: 11.94822408292967


#### V217 (기온·가동 상태 균형 + 중복 곡선 가중치)

- Huber 회귀 + 주말 잔차 RBF·poly SVR, 가동 상태×기온 구간 균형 가중, 중복 곡선 가중, 시간 감쇠 가중, 월별 확장 윈도우 CV, MAE·RMSE·R²

In [20]:
# V217: 전체 파이프라인 단일 셀 (외부 연구 모듈 import 불필요)
# 원본: scripts/research_v2/cells_phase2/V217.py
# 원본 SHA256: 5bdd3e48d431518866321173d3298b5411cb79fc1aa9236a59a4515e9fb9ce4d
# 내부 이름 함수 범위 한정 (앞선 모델 클래스·피처·설정 보호)
# 9월 예측·평가: '9월 테스트' 셀
def train_v217(train_df):
    # %% V217: standalone cell for notebooks/01model.ipynb
    # Requires train_df; test_df is never scored by this cell.

    # Source: notebook_models.py
    from collections import Counter, defaultdict
    import pandas as pd
    from sklearn.linear_model import LinearRegression
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import HuberRegressor
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.pipeline import Pipeline, make_pipeline
    from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
    from sklearn.svm import SVR
    from threadpoolctl import threadpool_limits
    import numpy as np
    import pandas as pd
    SEED = 42
    POWER = ['15분', '30분', '45분', '60분']
    VALIDATION_MONTHS = [202104, 202105, 202106, 202107, 202108]

    def metrics(y, pred):
        y, pred = (np.asarray(y).ravel(), np.asarray(pred).ravel())
        return dict(MAE=mean_absolute_error(y, pred), RMSE=np.sqrt(mean_squared_error(y, pred)), R2=r2_score(y, pred))

    def m242_features(frame):
        """날짜·시간·생산량·기온 4개 열 기반 피처"""
        assert frame['시간'].between(0, 23).all(), 'hour must be 0..23'
        hour = frame['시간']
        weekday = pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d').dt.dayofweek
        log_prod = np.log1p(frame['생산량'])
        on = log_prod.gt(0).where(log_prod.notna()).astype(float)
        hinge = (log_prod - np.log1p(100)).clip(lower=0)
        x = pd.DataFrame({'hour_weekday': hour.astype(str) + '_' + weekday.astype(str), '기온': frame['기온'], 'log_prod': log_prod, 'on': on, 'temp_on': frame['기온'] * on, 'temp_above20': (frame['기온'] - 20).clip(lower=0)}, index=frame.index)
        for day in range(7):
            x[f'prod_day{day}'] = log_prod * weekday.eq(day)
        for h in range(24):
            x[f'prod_hour{h}'] = log_prod * hour.eq(h)
        for h in range(24):
            x[f'prod_hinge_hour{h}'] = hinge * hour.eq(h)
        x['prod_sin'] = log_prod * np.sin(2 * np.pi * hour / 24)
        x['prod_cos'] = log_prod * np.cos(2 * np.pi * hour / 24)
        return x
    m242_pipeline = Pipeline([('features', FunctionTransformer(m242_features)), ('preprocess', ColumnTransformer([('category', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), ['hour_weekday'])], remainder=make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler()))), ('huber', MultiOutputRegressor(HuberRegressor(alpha=1.5, epsilon=1.05, max_iter=2000, tol=1e-07)))])

    def m242_sample_weight(train):
        """최근 가중(반감기 90일), 생산량 0 행 2배 가중"""
        date = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
        weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
        weight = np.where(train['생산량'].eq(0), 2.0, 1.0) * weight / weight.mean()
        return weight / weight.mean()

    def fit_m242(train):
        model = clone(m242_pipeline)
        with threadpool_limits(limits=2):
            return model.fit(train, train[POWER], huber__sample_weight=m242_sample_weight(train))

    def is_weekend(frame):
        return pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d').dt.dayofweek.ge(5)

    def n207_residual_features(frame):
        """주말 잔차 보정용: 시간·요일(범주), 기온·로그 생산량"""
        assert frame['시간'].between(0, 23).all(), 'hour must be 0..23'
        return pd.DataFrame({'시간': frame['시간'], '기온': frame['기온'], 'weekday': pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d').dt.dayofweek, 'log_prod': np.log1p(frame['생산량'])}, index=frame.index)
    n207_residual_pipeline = Pipeline([('features', FunctionTransformer(n207_residual_features)), ('preprocess', ColumnTransformer([('category', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['시간', 'weekday'])], remainder=make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler()))), ('svr', MultiOutputRegressor(SVR(kernel='rbf', C=10, epsilon=1, gamma='scale', tol=0.0001)))])

    class N207Model:
        """M242(전체 행 학습) + 주말 행 잔차 보정기. predict(frame) → (행 수, 4)"""

        def fit(self, train):
            self.base = fit_m242(train)
            residual = train[POWER] - self.base.predict(train)
            weekend = is_weekend(train)
            with threadpool_limits(limits=2):
                self.residual = clone(n207_residual_pipeline).fit(train.loc[weekend], residual.loc[weekend])
            return self

        def predict(self, frame):
            pred = self.base.predict(frame)
            weekend = is_weekend(frame).to_numpy()
            if weekend.any():
                pred[weekend] += self.residual.predict(frame.loc[weekend])
            return pred
    from sklearn.base import BaseEstimator, TransformerMixin

    class RHuberFeatures(TransformerMixin, BaseEstimator):
        """날짜 추세 기준일 fit 시 고정(평가 시 재계산 없음)"""

        def __init__(self, trend=False):
            self.trend = trend

        def fit(self, frame, y=None):
            if self.trend:
                date = pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d')
                self.anchor_ = float(date.max().toordinal())
            return self

        def transform(self, frame):
            assert frame['생산량'].dropna().ge(0).all()
            x = m242_features(frame)
            if self.trend:
                date = pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d')
                ordinal = date.map(lambda value: value.toordinal()).astype(float)
                gate = frame['생산량'].eq(0) & date.dt.dayofweek.lt(5)
                x['bounded_date_trend'] = ((ordinal - self.anchor_) / 60).clip(-1, 0) * gate
            return x

    class RWeekendModel:
        """전체 학습 행 Huber + 주말 학습 행 잔차 SVR. 출력 순서 = POWER"""

        def __init__(self, kernel='rbf', numeric_weight=1.0, zero_weight=2.0, trend=False):
            self.kernel = kernel
            self.numeric_weight = numeric_weight
            self.zero_weight = zero_weight
            self.trend = trend

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days / 90)
            weight = (weight / weight.mean()).to_numpy()
            weight = np.where(inputs['생산량'].eq(0), self.zero_weight, 1.0) * weight
            weight = weight / weight.mean()
            self.base = clone(m242_pipeline).set_params(features=RHuberFeatures(trend=self.trend))
            with threadpool_limits(limits=2):
                self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = date.dt.dayofweek.ge(5).to_numpy()
            assert weekend.any(), '주말 잔차 모델을 학습할 행이 필요합니다.'
            self.residual = Pipeline([('features', FunctionTransformer(n207_residual_features)), ('preprocess', ColumnTransformer([('category', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['시간', 'weekday']), ('numeric', make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler()), ['기온', 'log_prod'])], transformer_weights={'numeric': self.numeric_weight})), ('svr', MultiOutputRegressor(SVR(kernel=self.kernel, degree=2, coef0=1, C=10, epsilon=1, gamma='scale', tol=0.0001)))])
            with threadpool_limits(limits=2):
                self.residual.fit(inputs.loc[weekend], residual[weekend])
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            prediction = self.base.predict(inputs)
            weekend = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d').dt.dayofweek.ge(5).to_numpy()
            if weekend.any():
                prediction[weekend] += self.residual.predict(inputs.loc[weekend])
            return prediction

    class R192Model:
        """R192: 날짜 추세를 넣은 RWeekendModel 두 개(RBF 수치 가중 .5 / poly)를 .75/.25로 조합.
        zero_weight: Huber 생산량 0 행 가중치 (R192=1.0, R191=0.5)."""

        def __init__(self, zero_weight=1.0):
            self.zero_weight = zero_weight

        def fit(self, train):
            self.members = [RWeekendModel('rbf', numeric_weight=0.5, zero_weight=self.zero_weight, trend=True).fit(train), RWeekendModel('poly', zero_weight=self.zero_weight, trend=True).fit(train)]
            return self

        def predict(self, frame):
            left, right = (member.predict(frame) for member in self.members)
            return 0.75 * left + 0.25 * right

    class R100Model:

        def fit(self, train):
            self.members = [RWeekendModel('rbf').fit(train), RWeekendModel('poly').fit(train)]
            return self

        def predict(self, frame):
            left, right = (member.predict(frame) for member in self.members)
            zero = frame['생산량'].eq(0).to_numpy()[:, None]
            return np.where(zero, left, right)

    # Source: duplicate_models.py
    """Weight exact repeated training-day curves without removing any rows."""
    from collections import Counter
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from threadpoolctl import threadpool_limits
    pass

    class DuplicateModel:

        def __init__(self, config, seed=42):
            self.mode = config.get('mode', 'curve_equal')
            assert self.mode in ('curve_equal', 'curve_sqrt')

        def day_weights(self, train):
            curves = {}
            for date, day in train.groupby('날짜'):
                day = day.sort_values('시간')
                if len(day) == 24 and np.array_equal(day['시간'].to_numpy(), np.arange(24)):
                    curves[date] = day[POWER].to_numpy(dtype='<f8').tobytes()
            counts = Counter(curves.values())
            exponent = 1.0 if self.mode == 'curve_equal' else 0.5
            weights = {date: counts[curve] ** (-exponent) for date, curve in curves.items()}
            return train['날짜'].map(weights).fillna(1.0).to_numpy()

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0) * self.day_weights(train)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=RHuberFeatures(trend=True))
            weekend = is_weekend(inputs).to_numpy()
            self.residuals = []
            with threadpool_limits(limits=2):
                self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
                residual = train[POWER].to_numpy() - self.base.predict(inputs)
                for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                    correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                    correction.fit(inputs.loc[weekend], residual[weekend])
                    self.residuals.append(correction)
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            prediction = self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            if weekend.any():
                left, right = (model.predict(inputs.loc[weekend]) for model in self.residuals)
                prediction[weekend] += 0.75 * left + 0.25 * right
            return prediction

    # Source: state_interaction_models.py
    """H23: retain pooled training while adding state-dependent calendar responses."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.svm import SVR
    pass

    class StateInteractionFeatures(RHuberFeatures):

        def __init__(self, signal='on', grouping='weekday'):
            super().__init__(trend=True)
            self.signal, self.grouping = (signal, grouping)

        def transform(self, frame):
            features = super().transform(frame)
            weekday = pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d').dt.dayofweek
            groups = weekday if self.grouping == 'weekday' else weekday.ge(5).astype(int)
            logprod = np.log1p(frame['생산량'])
            signal = logprod if self.signal == 'logprod' else logprod.gt(0).where(logprod.notna()).astype(float)
            interactions = {f'state_hour_{hour}_group_{group}': signal * frame['시간'].eq(hour) * groups.eq(group) for hour in range(24) for group in range(7 if self.grouping == 'weekday' else 2)}
            return pd.concat([features, pd.DataFrame(interactions, index=frame.index)], axis=1)

    class StateInteractionModel:

        def __init__(self, config, seed=42):
            self.signal, self.grouping = (config.get('signal', 'on'), config.get('grouping', 'weekday'))
            assert self.signal in {'on', 'logprod'}
            assert self.grouping in {'weekday', 'weekend'}

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=StateInteractionFeatures(self.signal, self.grouping))
            self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            self.corrections = []
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                model = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr=MultiOutputRegressor(SVR(kernel=kernel, degree=2, coef0=1, C=10, epsilon=1, gamma='scale', tol=0.0001)))
                model.fit(inputs.loc[weekend], residual[weekend])
                self.corrections.append(model)
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            prediction = self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            if weekend.any():
                prediction[weekend] += 0.75 * self.corrections[0].predict(inputs.loc[weekend]) + 0.25 * self.corrections[1].predict(inputs.loc[weekend])
            return prediction

    # Source: thermal_state_combo_models.py
    """H27: original V054 intensity features plus original V070 state features."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.svm import SVR
    pass
    pass

    class ThermalStateComboFeatures(StateInteractionFeatures):

        def __init__(self):
            super().__init__(signal='on', grouping='weekend')

        def transform(self, frame):
            features = super().transform(frame)
            log_prod = np.log1p(frame['생산량'])
            features['temperature_intensity'] = frame['기온'] * log_prod
            features['cooling_intensity'] = (frame['기온'] - 20).clip(lower=0) * log_prod
            return features

    class ThermalStateComboModel(StateInteractionModel):

        def __init__(self, config, seed=42):
            assert config.get('mode', 'thermal_state') == 'thermal_state'
            super().__init__({'signal': 'on', 'grouping': 'weekend'}, seed)

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=ThermalStateComboFeatures())
            self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            self.corrections = []
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                model = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr=MultiOutputRegressor(SVR(kernel=kernel, degree=2, coef0=1, C=10, epsilon=1, gamma='scale', tol=0.0001)))
                model.fit(inputs.loc[weekend], residual[weekend])
                self.corrections.append(model)
            return self

    # Source: production_efficiency_models.py
    """H30: raw throughput response on V081, with train-only normalization."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.svm import SVR
    pass
    pass

    class ProductionEfficiencyFeatures(ThermalStateComboFeatures):

        def __init__(self, scale='global', replacement=False):
            super().__init__()
            self.scale, self.replacement = (scale, replacement)

        def fit(self, frame, y=None):
            super().fit(frame, y)
            positive = frame.loc[frame['생산량'].gt(0)].copy()
            positive['weekday'] = pd.to_datetime(positive['날짜'].astype(str), format='%Y%m%d').dt.dayofweek
            self.global_median = positive['생산량'].median()
            self.hour_medians = positive.groupby('시간')['생산량'].median()
            self.cell_medians = positive.groupby(['시간', 'weekday'])['생산량'].median()
            return self

        def transform(self, frame):
            features = super().transform(frame)
            denominator = np.full(len(frame), self.global_median)
            if self.scale == 'calendar':
                weekday = pd.to_datetime(frame['날짜'].astype(str), format='%Y%m%d').dt.dayofweek
                keys = pd.MultiIndex.from_arrays([frame['시간'], weekday], names=['시간', 'weekday'])
                denominator = self.cell_medians.reindex(keys).to_numpy()
                denominator = np.where(np.isnan(denominator), frame['시간'].map(self.hour_medians), denominator)
                denominator = np.where(np.isnan(denominator), self.global_median, denominator)
            if self.replacement:
                remove = [name for name in features if name.startswith('prod_') or name in {'log_prod', 'temperature_intensity', 'cooling_intensity'}]
                features = features.drop(columns=remove)
            throughput = frame['생산량'] / denominator
            features['throughput'] = throughput
            features['throughput_temperature'] = throughput * frame['기온']
            features['throughput_cooling'] = throughput * (frame['기온'] - 20).clip(lower=0)
            return features

    class ProductionEfficiencyModel:

        def __init__(self, config, seed=42):
            self.scale, self.mode = (config.get('scale', 'global'), config.get('mode', 'add'))
            assert self.scale in {'global', 'calendar'}
            assert self.mode in {'add', 'replace_log'}

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=ProductionEfficiencyFeatures(self.scale, self.mode == 'replace_log'))
            self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            self.corrections = []
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                model = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr=MultiOutputRegressor(SVR(kernel=kernel, degree=2, coef0=1, C=10, epsilon=1, gamma='scale', tol=0.0001)))
                self.corrections.append(model.fit(inputs.loc[weekend], residual[weekend]))
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            prediction = self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            if weekend.any():
                prediction[weekend] += 0.75 * self.corrections[0].predict(inputs.loc[weekend]) + 0.25 * self.corrections[1].predict(inputs.loc[weekend])
            return prediction

    # Source: radial_huber_models.py
    """H57: direct four-output Huber with one robust influence per observed hour."""
    import warnings
    import numpy as np
    import pandas as pd
    from scipy.optimize import minimize
    from sklearn.base import BaseEstimator, RegressorMixin, clone
    from sklearn.exceptions import ConvergenceWarning
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.svm import SVR
    pass
    pass
    pass

    def radial_huber_objective(parameters, matrix, target, weight):
        coefficients = parameters[:-5].reshape(matrix.shape[1], 4)
        intercept, scale = (parameters[-5:-1], parameters[-1])
        residual = (matrix @ coefficients + intercept - target) / scale
        radius = np.sqrt(np.mean(residual ** 2, axis=1))
        influence = np.minimum(1.0, 1.05 / np.maximum(radius, np.finfo(float).tiny))
        rho = np.where(radius <= 1.05, radius ** 2, 2 * 1.05 * radius - 1.05 ** 2)
        loss = 4 * (weight @ (scale * (1 + rho))) + 1.5 * np.sum(coefficients ** 2)
        score = weight[:, None] * influence[:, None] * residual
        gradient = np.r_[(2 * matrix.T @ score + 3 * coefficients).ravel(), 2 * score.sum(axis=0), 4 * (weight @ (1 + rho - 2 * radius * np.minimum(radius, 1.05)))]
        return (float(loss), gradient)

    class RadialHuberRegressor(RegressorMixin, BaseEstimator):

        def fit(self, matrix, target, sample_weight=None):
            target = np.asarray(target)
            weight = np.ones(len(target)) if sample_weight is None else np.asarray(sample_weight)
            initial = np.zeros(matrix.shape[1] * 4 + 5)
            initial[-5:-1] = np.median(target, axis=0)
            initial[-1] = max(float(np.median(np.sqrt(np.mean((target - initial[-5:-1]) ** 2, axis=1)))), 1.0)
            bounds = [(None, None)] * (len(initial) - 1) + [(np.finfo(float).eps * 10, None)]
            result = minimize(radial_huber_objective, initial, args=(matrix, target, weight), method='L-BFGS-B', jac=True, bounds=bounds, options={'maxiter': 2000, 'gtol': 1e-07, 'ftol': 1e-10})
            if result.status == 2:
                raise ValueError('Radial Huber optimization failed: ' + str(result.message))
            if not result.success:
                warnings.warn(str(result.message), ConvergenceWarning)
            self.coef_ = result.x[:-5].reshape(matrix.shape[1], 4)
            self.intercept_, self.scale_ = (result.x[-5:-1], result.x[-1])
            self.training_summary_ = dict(success=bool(result.success), iterations=int(result.nit), message=str(result.message), objective=float(result.fun), scale=float(self.scale_))
            return self

        def predict(self, matrix):
            return matrix @ self.coef_ + self.intercept_

    class RadialHuberModel(StateInteractionModel):

        def __init__(self, config, seed=42):
            assert config.get('loss', 'radial') == 'radial'

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=ThermalStateComboFeatures(), huber=RadialHuberRegressor())
            self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
            self.training_summary_ = self.base.named_steps['huber'].training_summary_
            residual = train[POWER].to_numpy() - self.base.predict(inputs)
            weekend = is_weekend(inputs).to_numpy()
            self.corrections = []
            for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr=MultiOutputRegressor(SVR(kernel=kernel, degree=2, coef0=1, C=10, epsilon=1, gamma='scale', tol=0.0001)))
                correction.fit(inputs.loc[weekend], residual[weekend])
                self.corrections.append(correction)
            return self

    # Source: thermal_models.py
    """H18: fixed 20-degree cooling interactions with operating state/intensity."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.pipeline import Pipeline, make_pipeline
    from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
    from sklearn.svm import SVR
    from threadpoolctl import threadpool_limits
    pass

    class ThermalFeatures(RHuberFeatures):

        def __init__(self, mode='state_hinge'):
            super().__init__(trend=True)
            self.mode = mode

        def transform(self, frame):
            features = super().transform(frame)
            log_prod = np.log1p(frame['생산량'])
            on = log_prod.gt(0).where(log_prod.notna()).astype(float)
            cooling = (frame['기온'] - 20).clip(lower=0)
            if self.mode in ('state_hinge', 'both'):
                features['cooling_on'] = cooling * on
                features['cooling_off'] = cooling * (1 - on)
            if self.mode in ('intensity', 'both'):
                features['temperature_intensity'] = frame['기온'] * log_prod
                features['cooling_intensity'] = cooling * log_prod
            if self.mode == 'daytime_hinge':
                angle = 2 * np.pi * frame['시간'] / 24
                features['cooling_on'] = cooling * on
                features['cooling_on_sin'] = cooling * on * np.sin(angle)
                features['cooling_on_cos'] = cooling * on * np.cos(angle)
            return features

    class ThermalModel:

        def __init__(self, config, seed=42):
            self.mode = config.get('mode', 'state_hinge')
            assert self.mode in ('state_hinge', 'intensity', 'daytime_hinge', 'both')

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days / 90)
            weight = (weight / weight.mean()).to_numpy()
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=ThermalFeatures(self.mode))
            weekend = date.dt.dayofweek.ge(5).to_numpy()
            self.residuals = []
            with threadpool_limits(limits=2):
                self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
                residual = train[POWER].to_numpy() - self.base.predict(inputs)
                for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                    model = Pipeline([('features', FunctionTransformer(n207_residual_features)), ('preprocess', ColumnTransformer([('category', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['시간', 'weekday']), ('numeric', make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler()), ['기온', 'log_prod'])], transformer_weights={'numeric': numeric_weight})), ('svr', MultiOutputRegressor(SVR(kernel=kernel, degree=2, coef0=1, C=10, epsilon=1, gamma='scale', tol=0.0001)))])
                    self.residuals.append(model.fit(inputs.loc[weekend], residual[weekend]))
            return self

        def predict(self, frame):
            inputs = frame[['날짜', '시간', '생산량', '기온']]
            prediction = self.base.predict(inputs)
            weekend = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d').dt.dayofweek.ge(5).to_numpy()
            if weekend.any():
                left, right = (model.predict(inputs.loc[weekend]) for model in self.residuals)
                prediction[weekend] += 0.75 * left + 0.25 * right
            return prediction

    # Source: combo_models.py
    """Combine original thermal, duplicate-weight, and zero-specialist settings."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from threadpoolctl import threadpool_limits
    pass
    pass
    pass

    class ComboModel(DuplicateModel):

        def __init__(self, config, seed=42):
            self.combination = config.get('mode', 'thermal_duplicate')
            assert self.combination in ('thermal_duplicate', 'thermal_zero', 'thermal_duplicate_zero')
            super().__init__({'mode': 'curve_sqrt'}, seed)

        def feature_transformer(self):
            return ThermalFeatures('intensity')

        def fit(self, train):
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            duplicate_weight = 1.0 if self.combination == 'thermal_zero' else self.day_weights(train)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0) * duplicate_weight
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=self.feature_transformer())
            weekend = is_weekend(inputs).to_numpy()
            self.residuals = []
            with threadpool_limits(limits=2):
                self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
                residual = train[POWER].to_numpy() - self.base.predict(inputs)
                for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                    correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                    correction.fit(inputs.loc[weekend], residual[weekend])
                    self.residuals.append(correction)
                self.specialist = None
                if self.combination.endswith('_zero'):
                    region = train.loc[train['생산량'].eq(0)]
                    dates = pd.to_datetime(region['날짜'].astype(str), format='%Y%m%d')
                    specialist_weight = np.exp2(-(dates.max() - dates).dt.days.to_numpy() / 90)
                    specialist_weight /= specialist_weight.mean()
                    self.specialist = clone(m242_pipeline).set_params(features=RHuberFeatures(trend=True))
                    self.specialist.fit(region[['날짜', '시간', '생산량', '기온']], region[POWER], huber__sample_weight=specialist_weight)
            return self

        def predict(self, frame):
            prediction = super().predict(frame)
            if self.specialist is not None:
                zero = frame['생산량'].eq(0).to_numpy()
                if zero.any():
                    prediction[zero] = self.specialist.predict(frame.loc[zero, ['날짜', '시간', '생산량', '기온']])
            return prediction

    # Source: thermal_balance_models.py
    """Balance sparse thermal support with one fixed inverse-square-root rule."""
    import numpy as np
    pass
    pass

    class ThermalBalanceModel(ComboModel):

        def __init__(self, config, seed=42):
            super().__init__({'mode': 'thermal_duplicate'}, seed)
            self.grouping = config.get('grouping', 'temperature')
            assert self.grouping in ('temperature', 'state_temperature')

        def feature_transformer(self):
            return ThermalStateComboFeatures()

        def day_weights(self, train):
            group = train['기온'].floordiv(5).astype('Int64').astype(str)
            if self.grouping == 'state_temperature':
                group = train['생산량'].eq(0).astype(str) + '_' + group
            self.thermal_counts_ = group.value_counts().to_dict()
            weight = 1.0 / np.sqrt(group.map(self.thermal_counts_))
            return (weight / weight.mean()).to_numpy()

    # Source: phase2_quality_combo_models.py
    """Preserve V096 thermal support weights and V028 duplicate-curve weights."""
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.preprocessing import RobustScaler
    from threadpoolctl import threadpool_limits
    pass
    pass
    pass
    pass
    pass
    pass
    pass

    class Phase2QualityComboModel(ThermalBalanceModel):

        def __init__(self, config, seed=42):
            self.quality_mode = config.get('mode', 'state_temperature_duplicate')
            assert self.quality_mode in ('state_temperature_duplicate', 'raw_throughput', 'radial', 'features_r191', 'features_v054', 'features_v070', 'radial_v070', 'radial_v070_recent3', 'radial_v070_recent3_iqr')
            super().__init__({'grouping': 'state_temperature'}, seed)

        def feature_transformer(self):
            if self.quality_mode == 'raw_throughput':
                return ProductionEfficiencyFeatures(scale='global', replacement=False)
            if self.quality_mode == 'features_r191':
                return RHuberFeatures(trend=True)
            if self.quality_mode == 'features_v054':
                return ThermalFeatures(mode='intensity')
            if self.quality_mode in ('features_v070', 'radial_v070', 'radial_v070_recent3', 'radial_v070_recent3_iqr'):
                return StateInteractionFeatures(signal='on', grouping='weekend')
            return super().feature_transformer()

        def day_weights(self, train):
            return ThermalBalanceModel.day_weights(self, train) * DuplicateModel.day_weights(self, train)

        def fit(self, train):
            if self.quality_mode not in ('radial', 'radial_v070', 'radial_v070_recent3', 'radial_v070_recent3_iqr'):
                return super().fit(train)
            if self.quality_mode in ('radial_v070_recent3', 'radial_v070_recent3_iqr'):
                dates = pd.to_datetime(train['날짜'].astype(str), format='%Y%m%d')
                start = (dates.max().to_period('M') - 2).start_time
                train = train.loc[dates >= start]
            inputs = train[['날짜', '시간', '생산량', '기온']]
            date = pd.to_datetime(inputs['날짜'].astype(str), format='%Y%m%d')
            weight = np.exp2(-(date.max() - date).dt.days.to_numpy() / 90)
            weight *= np.where(inputs['생산량'].eq(0), 0.5, 1.0) * self.day_weights(train)
            weight /= weight.mean()
            self.base = clone(m242_pipeline).set_params(features=self.feature_transformer(), huber=RadialHuberRegressor())
            if self.quality_mode == 'radial_v070_recent3_iqr':
                self.base.set_params(preprocess__remainder__standardscaler=RobustScaler(quantile_range=(25.0, 75.0)))
            weekend = is_weekend(inputs).to_numpy()
            self.residuals, self.specialist = ([], None)
            with threadpool_limits(limits=2):
                self.base.fit(inputs, train[POWER], huber__sample_weight=weight)
                residual = train[POWER].to_numpy() - self.base.predict(inputs)
                for kernel, numeric_weight in [('rbf', 0.5), ('poly', 1.0)]:
                    correction = clone(n207_residual_pipeline).set_params(preprocess__transformer_weights={'remainder': numeric_weight}, svr__estimator__kernel=kernel, svr__estimator__degree=2, svr__estimator__coef0=1)
                    correction.fit(inputs.loc[weekend], residual[weekend])
                    self.residuals.append(correction)
            self.training_summary_ = self.base.named_steps['huber'].training_summary_
            return self

    NAME, CONFIG, SEED = 'V217', {'kind': 'phase2_quality_combo', 'mode': 'state_temperature_duplicate', 'phase': 2, 'reference': 'V081', 'hypothesis': 'P2_H13'}, 42
    INPUTS = ['날짜', '시간', '생산량', '기온', '풍속', '습도', '강수량']
    np.random.seed(SEED)
    filtered_train = train_df.loc[
        train_df['날짜'].between(20210101, 20210831) & train_df['시간'].between(0, 23)
    ].sort_values(['날짜', '시간']).copy()
    assert len(filtered_train) == 5784
    assert not filtered_train.duplicated(['날짜', '시간']).any()
    assert filtered_train[POWER].notna().all().all()

    def fit_exported(frame):
        return Phase2QualityComboModel(CONFIG, SEED).fit(frame)

    # 1~3월→4월, …, 1~7월→8월. 구성 모델 학습 기간·가중치 규칙 원본 그대로
    scores, predictions = [], []
    with threadpool_limits(limits=2):
        for month in VALIDATION_MONTHS:
            tr = filtered_train.loc[filtered_train['날짜'].between(20210101, month * 100 - 1)]
            va = filtered_train.loc[filtered_train['날짜'].floordiv(100).eq(month)]
            assert not tr.empty and not va.empty
            assert tr['날짜'].max() < va['날짜'].min()
            fold_model = fit_exported(tr)
            prediction = fold_model.predict(va[INPUTS])
            assert prediction.shape == (len(va), 4) and np.isfinite(prediction).all()
            scores.append(dict(month=month, **metrics(va[POWER], prediction)))
            predictions.append(pd.DataFrame(prediction, index=va.index, columns=POWER).assign(month=month))
            print(f"{NAME} {month}: MAE={scores[-1]['MAE']:.6f}")
            del fold_model
        # 최종 모델: 1~8월 데이터 전달, 구성 모델별 학습 범위 적용
        final_model = fit_exported(filtered_train)
    return pd.DataFrame(scores).set_index('month'), pd.concat(predictions).sort_index(), final_model


cv_scores_v217, cv_predictions_v217, model_14 = train_v217(train_df)
cv_results["V217"] = {"scores": cv_scores_v217}
print("V217 월별 확장 교차검증")
print(cv_scores_v217.to_string())
print("월평균 MAE:", cv_scores_v217["MAE"].mean())


V217 202104: MAE=16.907424
V217 202105: MAE=11.277956
V217 202106: MAE=11.987923
V217 202107: MAE=12.320221
V217 202108: MAE=8.332166
V217 월별 확장 교차검증
              MAE       RMSE        R2
month                                 
202104  16.907424  28.750074  0.715910
202105  11.277956  23.769296  0.812415
202106  11.987923  20.235499  0.867454
202107  12.320221  23.121763  0.856595
202108   8.332166  14.941368  0.941516
월평균 MAE: 12.165138062936407


### 9월 테스트

- 9월 홀드아웃 평가, 월별 확장 윈도우 CV 요약, MAE·RMSE·R²

In [24]:
# 1~8월 확장 윈도우 CV 점수
# scores: 인덱스 검증월, 열 MAE·RMSE·R2, 전 후보 동일 5개 월
for result in cv_results.values():
    assert result["scores"].index.tolist() == VALIDATION_MONTHS
    assert result["scores"][["MAE", "RMSE", "R2"]].notna().all().all()

cv_comparison = pd.DataFrame([
    {"model": name, **result["scores"][["MAE", "RMSE", "R2"]].mean().to_dict()}
    for name, result in cv_results.items()
]).set_index("model").sort_values("MAE", kind="stable")
print("모델별 월평균 CV 성능")
print(cv_comparison.to_string())

# 1~8월 전체 학습 최종 모델 등록. model.predict(frame) → (행 수, 4)
test_models = {
    "OLS": model_0,
    "M242": model_1,
    "LightGBM": model_2,
    "N207": model_3,
    "N045": model_4,
    "N002": model_15,
    "N049": model_16,
    "R192": model_5,
    "R100": model_6,
    "R222": model_7,
    "R191": model_8,
    "V081": model_9,
    "V217": model_14,
    "V305": model_11,
    "V384": model_12,
    "V385": model_13,
    **blend_models,
}

# 9월 정답 사용은 이 셀만
assert test_df[POWER].notna().all().all()
test_inputs = test_df.drop(columns=POWER)  # 정답 열 제외
test_predictions, test_rows = {}, []
for name, model in test_models.items():
    prediction = model.predict(test_inputs)
    test_predictions[name] = pd.DataFrame(prediction, index=test_df.index, columns=POWER)
    test_rows.append({"model": name, **metrics(test_df[POWER], prediction)})

test_scores = pd.DataFrame(test_rows).set_index("model").sort_values("MAE", kind="stable")
print("9월 테스트 성능")
print(test_scores.to_string())

모델별 월평균 CV 성능
                    MAE       RMSE        R2
model                                       
V385          11.948224  22.170988  0.839491
V384          11.950709  22.141794  0.839716
V305          11.984814  22.207421  0.838393
V217          12.165138  22.163600  0.838778
V081          12.316831  22.961422  0.822711
R191          12.594951  23.085961  0.820895
R192          12.625282  23.047139  0.821402
R100          12.627745  23.021437  0.821568
R222          12.636347  23.052724  0.821472
N207          12.704130  23.108981  0.820094
Blend(가동×주말)  12.705841  21.796300  0.835264
Blend(전역)     12.894450  22.632012  0.827439
M242          12.898238  23.536869  0.813492
Blend(가동여부)   13.183833  22.878977  0.823281
LightGBM      14.386454  23.234586  0.806202
N045          14.878298  25.945824  0.778659
N049          16.899860  26.461498  0.770172
N002          19.418250  28.786667  0.717357
OLS           37.138949  43.860150  0.413171
9월 테스트 성능
                    MAE       R

In [25]:
# 모델별 월별 확장 CV 결과 모음 (9월 테스트 불필요)
cv_monthly_summary = pd.concat(
    {name: result["scores"][["MAE", "RMSE", "R2"]] for name, result in cv_results.items()},
    names=["model", "month"],
)
cv_mean_summary = cv_monthly_summary.groupby(level="model", sort=False).mean().sort_values("MAE", kind="stable")

print("월별 확장 윈도우 CV (1~3월→4월, …, 1~7월→8월)")
print(cv_monthly_summary.to_string())
print("\n모델별 월평균 CV 성능 (MAE 오름차순)")
print(cv_mean_summary.to_string())


월별 확장 윈도우 검증: 1~3월→4월, …, 1~7월→8월
                           MAE       RMSE        R2
model        month                                 
OLS          202104  35.223609  41.697684  0.402412
             202105  36.795449  43.739571  0.364793
             202106  35.176090  40.919197  0.458009
             202107  37.163334  44.002327  0.468148
             202108  41.336262  48.941970  0.372494
M242         202104  18.838011  33.147559  0.622357
             202105  11.390720  24.806295  0.795690
             202106  12.163612  20.233590  0.867479
             202107  13.236095  24.133745  0.843767
             202108   8.862750  15.363157  0.938168
LightGBM     202104  21.958903  35.537447  0.565939
             202105  14.931942  28.069227  0.738406
             202106  13.698905  20.219663  0.867662
             202107  12.317017  18.076236  0.912353
             202108   9.025505  14.270359  0.946651
N207         202104  18.491166  32.689998  0.632711
             202105  11.149117